In [ ]:
import pandas as pd
import numpy as np

from astropy.coordinates import SkyCoord, angular_separation, SkyOffsetFrame
import astropy.units as u
from astropy.modeling import models, fitting
from astropy.table import Table, vstack
from astropy.io import fits
from astropy.wcs import WCS
from astropy.nddata.utils import Cutout2D
from astropy.visualization import simple_norm
from reproject import reproject_exact, reproject_interp
#from bayestar import SFH
import json
import glob
import matplotlib.pyplot as plt
from matplotlib.ticker import (MultipleLocator, AutoLocator, AutoMinorLocator, LogLocator)
from pydol.photometry.scripts.gloess import gloess
from pydol.photometry.scripts.catalog_filter import box, ellipse
from pydol.photometry.scripts.cmdtools import gen_CMD, gen_CMD_xcut, gen_CMD_ycut, running_avg
from astropy.stats import SigmaClip, sigma_clipped_stats, gaussian_fwhm_to_sigma
from astropy.stats.biweight import biweight_location, biweight_midvariance,biweight_scale
from pydol import photometry
from pathlib import Path
import subprocess
from scipy.interpolate import griddata, interp1d
from astropy.visualization.wcsaxes import SphericalCircle
from astropy.visualization.wcsaxes import add_beam, add_scalebar
import matplotlib.font_manager as fm
import matplotlib as mpl
from numpy import append, array, exp, \
                linspace, loadtxt, log10, pi, meshgrid, \
                savetxt, sqrt, where, ones, percentile,trapz, all
from scipy import special
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import time, glob, os, sys
import multiprocessing as mp
from pathlib import Path
import seaborn as sb
import stan

from scipy.spatial import Voronoi
from regions import EllipseSkyRegion

In [ ]:
10**(r-0.5)*0.02

In [ ]:
script_dir = str(Path(photometry.__file__).parent.joinpath('scripts'))

In [ ]:
mpl.rcParams["font.family"], mpl.rcParams["font.serif"];

In [ ]:
# Minimalistic seaborn style
sb.set_theme(style="white")

mpl.rcParams.update({
    "text.usetex": True,                # If using LaTeX for labels
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "axes.labelsize": 25,
    "font.size": 25,
    "legend.fontsize": 10,
    "xtick.labelsize": 30,
    "ytick.labelsize": 30,
    "axes.titlesize": 30,
    "lines.linewidth": 1.0,
    "lines.markersize": 3,
    "figure.dpi": 300,                  # High-quality output
    "savefig.dpi": 300,
    "axes.grid": False,                 # Avoid grids unless needed
    "legend.frameon": False             # No legend frame
})

plt.rcParams['axes.titlesize']  = plt.rcParams['axes.labelsize'] = 30
plt.rcParams['xtick.labelsize'] = plt.rcParams['ytick.labelsize'] = 30


In [ ]:

code = """

    functions{
        real P(int N1, int N2, vector v, matrix M) {
            vector[N1] Mj;
            vector[N1] ln_Mj;

            Mj= M*v;
            for (j in 1:N1){
                if (Mj[j]<=0.)
                    Mj[j] = 1.;
            }
            ln_Mj = log(Mj);
            return sum(ln_Mj);
        }
    }

    data {
        int<lower=0> Nj; // number of data
        int<lower=0> Ni; // number of isochrones
        matrix[Nj,Ni] Pij; // Probability matrix
        matrix[Nj,Ni] Cij; // Normalization matrix
    }

    parameters {
        simplex[Ni] a;
    }

    model {
        target += dirichlet_lpdf(a | rep_vector(1., Ni));
        target += P(Nj,Ni,a,Pij);
        target += -1.*P(Nj,Ni,a,Cij);
    }

    """

class Base():

    def Normal_MGk(self, fw_dat,fw_err,Iso_sig):    # Error model apparent maginute
        sig2 = fw_err*fw_err + Iso_sig*Iso_sig          # And normal Isochrone 
        
        return  lambda fw_iso : exp( -0.5*(fw_dat-fw_iso)**2 / sig2 ) / sqrt(2.*pi*sig2)

    def Phi_MGk(self,fwj2, sig_fwj2, fwklim, sig_i2):
        b = sig_i2*sig_i2 + sig_fwj2*sig_fwj2
        b1 = sig_i2*sig_i2/b
        b2 = sig_fwj2*sig_fwj2/b
        b3 = sig_i2*sig_i2/sqrt(b)
        return  lambda fw_i2 : special.ndtr( ( fwklim - b1*fwj2 - b2*fw_i2 ) / b3 )

    def prit(self, m, alpha=0.5,m_50=30):
        
        return lambda m_iso: 0.5*(1 - alpha*(m - m_iso - m_50)/np.sqrt(1 + alpha**2*(m-m_iso-m_50)**2))
             
    def IMF_Krp(self, m, ml=0.1, mint=0.5, mu=350.,a1=1.3,a2=2.3):

        h2 = (mu**(1.-a2)-mint**(1.-a2))/(1.-a2)
        h1 = (mint**(1.-a1)-ml**(1.-a1))/(1.-a1)

        c1 = 1./(h1+h2*mint**(a2-a1))
        c2 = c1*mint**(a2-a1)

        c = ones(len(m))
        c[where(m < mint)] = c1
        c[where(m >= mint)] = c2

        a = ones(len(m))
        a[where(m < mint)] = -a1
        a[where(m >= mint)] = -a2
        imf = c*m**a

        return(imf)


    def P_ij_map(self, IDp):
        fw1_lim = self.fw_lims[0]
        fw2_lim = self.fw_lims[1]
        fw3_lim = self.fw_lims[2]

        sig_i = self.sig_fw[0]

        if not os.path.exists(self.out_dir + 'pij_cij_results'):
            os.mkdir(self.out_dir + 'pij_cij_results')

        filename_p = '%s_Pij_Data_LimMag%.2lf_%srows_%siso_IsoModel_sig%s_IMF_%s_Simple.txt' % \
            (IDp,fw2_lim,str(self.N_dat),str(self.NIso),str(sig_i).replace('.','p'), self.IMF)    ## Opening file
        fp = open(os.path.join(self.out_dir + "pij_cij_results",filename_p),'a')
        args = []

         ## Pij is calculated row by row, i.e. fix j-th dat and run each i-th isochrone.

        for j in range(self.N_dat):                       
    #                    0   1     2    3     4         5        6      7       8        9     10
            args.append([j, self.dat, self.NIso, self.Iso, 
                         fw1_lim, fw2_lim, fw3_lim, self.N_dat, filename_p, sig_i, self.IMF])
        with mp.Pool(self.n_cores) as p:         ## Pooling Pij rows using all the abailable CPUs (Parallel computation)
            results = p.map(self.P_ij_row_map, args)
            Pij_out=[]
            for [j,wr] in results:
                fp.write('{}'.format(' '.join(wr))+'\n')
                Pij_out.append(array(wr, dtype=float))
        fp.close()
        
        return([Pij_out, filename_p])

    def P_ij_row_map(self, args):

        j = args[0]
        dat = args[1]
        Niso = args[2]
        Iso = args[3]

        fw1_lim = args[4]
        fw2_lim = args[5]
        fw3_lim = args[6]

        Ndat = args[7]
        filename_p = args[8]
        sig_i = args[9]
        imf   = args[10]
        
        P_fw1 = self.Normal_MGk(self.dat[2][j], self.dat[3][j],sig_i)
        P_fw2 = self.Normal_MGk(self.dat[4][j], self.dat[5][j],sig_i)
        P_fw3 = self.Normal_MGk(self.dat[6][j], self.dat[7][j],sig_i)

        Phi_fw1 = self.Phi_MGk(self.dat[2][j], self.dat[3][j], fw1_lim, sig_i)
        Phi_fw2 = self.Phi_MGk(self.dat[4][j], self.dat[5][j], fw2_lim, sig_i)
        Phi_fw3 = self.Phi_MGk(self.dat[6][j], self.dat[7][j], fw3_lim, sig_i)

        #Phi_fw1 = self.prit(self.dat[2][j],self.m_50s[0] - self.dm - self.A_fw1, self.a_50s[0])
        #Phi_fw2 = self.prit(self.dat[4][j],self.m_50s[1] - self.dm - self.A_fw2, self.a_50s[1])
        #Phi_fw3 = self.prit(self.dat[6][j],self.m_50s[2] - self.dm - self.A_fw3, self.a_50s[2])
                            
        wr=[]
        ages = np.array(self.ages)
        for i in range(self.NIso):                    ## Isochrone loop

            if self.IMF == "Krp":
                imf_p = self.IMF_Krp(self.Iso[i][1])
            elif self.IMF == "Slp":
                print("Not available at the moment")
                #imf_p = self.IMF_Salp(Iso[i][1])
            else:
                # Default
                imf_p = self.IMF_Krp(self.Iso[i][1])

            Ps   = P_fw1(self.Iso[i][2])*P_fw2(self.Iso[i][3])*P_fw3(self.Iso[i][4])
            
            Phis = Phi_fw1(self.Iso[i][2])*Phi_fw2(self.Iso[i][3])*Phi_fw3(self.Iso[i][4])
            
            delta_t  = 10**(ages[i]-0.05)-10**(ages[i] + 0.05)
            delta_tc = (10**(ages-0.05)-10**(ages+0.05)).sum()
            
            Intg = imf_p*Ps*Phis*(delta_t/delta_tc)

            ## Interand
            p = trapz(Intg,self.Iso[i][1])

            wr.append(str(p))

        return ([j,wr])

    def C_ij_map(self, IDc):

        fw1_lim = self.fw_lims[0]
        fw2_lim = self.fw_lims[1]
        fw3_lim = self.fw_lims[2]
        
        sig_i   = self.sig_fw[0]

        filename_c = '%s_Cij_Data_LimMag%.2lf_%srows_%siso_IsoModel_sig%s_IMF_%s_Simple.txt' % \
            (IDc,fw2_lim,str(self.N_dat),str(self.NIso),str(self.sig_fw[0]).replace('.','p'), self.IMF)
        fp = open(os.path.join(self.out_dir + "pij_cij_results",filename_c),'a')   ## output matrix
        args = []

        # Cij is calcutated row by row, i.e. fix j-th dat and run each i-th isochrone.

        for j in range(self.N_dat):    
            args.append([j, self.dat, self.NIso, self.Iso, fw1_lim, 
                         fw2_lim, fw3_lim, self.N_dat, filename_c, sig_i, self.IMF])

        with mp.Pool(self.n_cores) as p:
            results = p.map(self.C_ij_row_map, args)
            Cij_out=[]
            for [j,wr] in results:
                fp.write('{}'.format(' '.join(wr))+'\n')
                Cij_out.append(array(wr, dtype=float))
        fp.close()
        
        return(Cij_out)

    def C_ij_row_map(self,args):

        j    = args[0]
        dat  = args[1]
        Niso = args[2]
        Iso  = args[3]

        fw1_lim = args[4]
        fw2_lim = args[5]
        fw3_lim = args[6]

        Ndat       = args[7]
        filename_c = args[8]
        sig_i      = args[9]
        imf        = args[10]

        phi_fw1 = self.Phi_MGk(dat[2][j], dat[3][j], fw1_lim, sig_i)
        phi_fw2 = self.Phi_MGk(dat[4][j], dat[5][j], fw2_lim, sig_i)
        phi_fw3 = self.Phi_MGk(dat[6][j], dat[7][j], fw3_lim, sig_i)

        #phi_fw1 = self.prit(self.dat[2][j],self.m_50s[0], self.a_50s[0])
        #phi_fw2 = self.prit(self.dat[4][j],self.m_50s[1], self.a_50s[1])
        #phi_fw3 = self.prit(self.dat[6][j],self.m_50s[2], self.a_50s[2])

        wr = []
        for i in range(Niso):

            if imf == "Krp":
                imf_c = self.IMF_Krp(Iso[i][1])
            elif imf=="Slp":
                print("Not available at the moment")
                #imf_c = self.IMF_Salp(Iso[i][1])
            else:
                imf_c = self.IMF_Krp(Iso[i][1])

            intg_c = imf_c*phi_fw1(Iso[i][2])*phi_fw2(Iso[i][3])*phi_fw3(Iso[i][4])
            p_c = trapz(intg_c,Iso[i][1])

            wr.append(str(p_c))

        return ([j,wr])

    def ai_samp(self, ID, Name):


        ### Data for STAN ###
        dats = {'Nj' : self.N_dat,
                'Ni' : self.NIso,
                'Pij': self.P_ij,
                'Cij': self.C_ij  }

        ############ Running pystan ############

        sm = stan.build(code, data=dats, random_seed=1234)
        fit = sm.sample(num_samples=self.N_smp, num_chains=self.N_wlk, num_warmup=200)
        self.fit = fit
        a_sp = fit["a"].T

        ######### Saving the MCMC sample #########

        N_iso = len(a_sp[0])

        a_perc = array([ percentile(ai,[10,50,90]) for ai in a_sp.T])       ##  10th, 50th, 90th percentiles

        sfh=array([self.Z_age_isos[:,0], self.Z_age_isos[:,1], a_perc[:,0], a_perc[:,1], a_perc[:,2] ]).T

        ##
        hd='Z,Log_age,p10,p50,p90'
        filename = ID+"_ai"+Name+"_Niter"+str(len(a_sp))+".txt"
        savetxt(self.out_dir + filename, sfh, header=hd, fmt="%.6f", delimiter=",",comments='')
        
        return filename


In [ ]:
class BaySFH(Base):
    def __init__(self,df=None,N_wlk=34, N_smp=10000, fw1_lim=30.,fw2_lim=30., fw3_lim=30.0, 
                 A_fw1=0, A_fw2=0, A_fw3=0, sig_fw1=0.1,sig_fw2=0.1, sig_fw3=0.1, 
                 dismod=29.67,isofiles='', isodir=None, ph_sup=100,m_inf=0.1,
                 IMF='Krp',parallel=True, out_dir='.', n_cores=1, m_50s = [25,25,25], a_50s=[0.5,0.5,0.5]):

        self.n_cores = n_cores
        self.out_dir = out_dir
        self.dm = dismod
        self.m_50s = m_50s
        self.a_50s = a_50s
        
        self.N_wlk, self.N_smp = N_wlk, N_smp
        
        self.A_fw1, self.A_fw2, self.A_fw3 = A_fw1, A_fw2, A_fw3
        
        self.fw_lims = array([fw1_lim,fw2_lim,fw3_lim])
        
        self.sig_fw = array([sig_fw1,sig_fw2,sig_fw3])
        
        if isodir is None:
            isodir = f'{data_dir}/test_files/Isochrone.test'
        
        ########### Reading Isochrones ###########
        if (isofiles != ''):
            filelist = []
            f = open(isofiles, "r")
            lines = f.readlines()
            for line in lines:
                filelist.append(line.replace('\n',''))
        
            self.filelist = filelist
            f.close()
        else:
            filelist = glob.glob(os.path.join(isodir, "*.isoc"))
            self.filelist = filelist
        
        #               F435W  F555W  F814W
        #   ph   mass   mag1   mag2   mag3  Z  log_age  phase_prob
        iso = array([ loadtxt(k) for k in sorted(filelist) ], dtype=object)
        
        self.ages = [float(i.split('Myr')[0].split('AGE')[1]) for i in self.filelist]
        
        self.Z_age_isos = array([ loadtxt(k)[0][5:7] for k in sorted(self.filelist) ], dtype=object)
        
        
        for l in range(len(iso)):
            iso[l] = iso[l][where( (iso[l].T[1]>=m_inf) & (iso[l].T[0]<=ph_sup))]       ## mass Truncation & stellar phase Truc
            iso[l] = iso[l].T
        
        self.Iso = iso
        ################################### DATA #######################################
        
        #  0      1       2        3         4          5         6          7
        #  RA    DEC     fw1   fw1_error    fw2     fw2_error    fw3     fw3_error
        
        if df is None:
            
            df = pd.read_fwf(f"{data_dir}/test_files/data.test", sep=' ')
            col_dict = {'RA_HST'  : 'RA',
                        'DEC_HST' : 'DEC',
                        'F435W'   : 'fw1',
                        'eF435W'  : 'fw1_error',
                        'F555W'   : 'fw2',
                        'eF555W'  : 'fw2_error',
                        'F814W'   : 'fw3',
                        'eF814W'  : 'fw3_error'}

            df = df.rename(columns=col_dict)
        keys = ['RA','DEC','fw1','fw1_error','fw2','fw2_error','fw3','fw3_error']
        try:
            dt = df[keys]
        except:
            print("Data Frame input keys: ", keys)
            raise Exception("Input data keys don't match!")
        
        step = int(1)
        dat = dt.values.astype(float)
        msg = "from %d... (FW1 <= %.2lf) (FW2 <= %.2lf) (FW3 <= %.2lf)" % (len(dat), fw1_lim, fw2_lim, fw3_lim)
        
        # Completeness Filtering
        dat = dat[where((dat[:,2] <= fw1_lim) & (dat[:,4]  <= fw2_lim) & (dat[:,6]  <= fw3_lim) )]        # Truncate by apparent magnitude
        
        #dat = dat[where((dat[:,4]  < fw2_lim))]  
        w_dat = dat[:,0]
        
        # Adding Extinction
        dat[:,2] -= A_fw1+dismod
        dat[:,4] -= A_fw2+dismod
        dat[:,6] -= A_fw3+dismod
        
        print ("Selecting %d %s" % (len(dat), msg))
        dat = dat[::step]
        dat = dat
        self.dat = dat.T
        
        self.N_dat = len(self.dat[0])
        self.NIso = len(self.Iso)
        
        self.parallel = parallel
        self.IMF = IMF

    def __call__(self):
        ########################### Execution Routines #################################
        
        start = time.time()
        print ("Starting Pij, Cij computation")
        if (self.parallel):
            print ("\tParallel mode...")
        
            ID = str(int(time.time()))
        
            self.Pij_reslt = self.P_ij_map(ID)
            self.P_ij, Pij_name = self.Pij_reslt[0], self.Pij_reslt[1]
            
            Name = Pij_name[Pij_name.find("_Pij")+4:Pij_name.find(".txt")]
        
            self.C_ij = self.C_ij_map(ID)
        
        else:
            print ("\tSequential mode... Not available for the moment")
           # P_ij(dat, N_dat, r_int, iso, N_iso)
        print ("Finished                                ")
        
        
        end = time.time()
        
        elapsed = end - start
        
        print ("Elapsed time: %02d:%02d:%02d" % (int(elapsed / 3600.), int((elapsed % 3600)/ 60.), elapsed % 60))
        
        ###########################################
        filename = self.ai_samp(ID, Name)
        print("Completed!!!")
        return filename


In [ ]:
plt.rcParams['image.cmap'] = 'jet'
plt.rcParams['image.origin'] = 'lower'
plt.rcParams['figure.figsize'] = (7,5)
plt.rcParams['axes.titlesize'] = plt.rcParams['axes.labelsize'] = 35
plt.rcParams['xtick.labelsize'] = plt.rcParams['ytick.labelsize'] = 35

font1 = {'family': 'sans-serif', 'color': 'black', 'weight': 'normal', 'size': '15'}
font2 = {'family': 'sans-serif', 'color': 'black', 'weight': 'normal', 'size': '25'}

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica"]})


In [ ]:
def add_compass(ax, x=0,y=0,length=0.1, color='white', **kwargs):

    # Get the central coordinate
    center_pix = [x, y]
    ra, dec = ax.wcs.pixel_to_world_values([[x,y]])[0]
    center_sky = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame='fk5')

    # Define the North direction (increasing Dec)
    north_sky = SkyCoord(ra=center_sky.ra, dec=center_sky.dec + length * u.deg)
    north_pix = ax.wcs.world_to_pixel(north_sky)
    
    # Define the East direction (increasing RA)
    east_sky = SkyCoord(ra=center_sky.ra + length * u.deg, dec=center_sky.dec)
    east_pix = ax.wcs.world_to_pixel(east_sky)
    
    # Add arrows to the plot using annotate
    ax.annotate('', xy=(north_pix[0], north_pix[1]), xytext=(center_pix[0], center_pix[1]),
                arrowprops=dict(facecolor=color, edgecolor=color, width=2, ls='-'), zorder=500)
    ax.annotate('', xy=(east_pix[0], east_pix[1]), xytext=(center_pix[0], center_pix[1]),
                arrowprops=dict(facecolor=color, edgecolor=color, width=2, ls='-'), zorder=500)
    
    # Add labels
    ax.text(north_pix[0]*1.01, north_pix[1]*1.1, 'N', color=color, fontsize=20, ha='center', va='center', zorder=500)
    ax.text(east_pix[0], east_pix[1]*0.9, 'E', color=color, fontsize=20, ha='center', va='center', zorder=500)


# **Regions**

In [ ]:
regions_dict = {'bubble' : {'ra'   : 24.1858128,  
                       'dec'  : 15.7725802,
                       'F115W': 25.4, 
                       'F150W': 24.63,
                       'F200W': 23.53,
                      
                       'F435W': 27.88, 
                       'F555W': 27.48,
                       'F814W': 26.52},
                
            'ngc628' : {'ra'   : 24.1738983, 
                       'dec'  : 15.7836543,
                       'F115W': 25.42, 
                       'F150W': 24.55,
                       'F200W': 23.56},
            'm83' :   {'ra'   : 204.2536827 ,  
                       'dec'  : -29.8655432,
                       'F115W': 25.42, 
                       'F150W': 24.55,
                       'F200W': 23.56},
            'm51' :  {'ra'   : 202.4696435,
                       'dec'  : 47.1952141,
                       'F115W': 25.42, 
                       'F150W': 24.55,
                       'F200W': 23.56},  
                
            'bkg3'   : {'ra'   : 24.1728133,
                   'dec'  : 15.7669357,
                   'F115W': 25.42, 
                   'F150W': 24.55,
                   'F200W': 23.56,},
                
            'HI_hole' : {'ra': 24.1975627,
                         'dec' : 15.7525293,
                          'a' : 27.888,
                          'b' : 25.599,
                           'ang' : 140}
           }
                             
            
with open('../data/DS9 regions/regions90_ngc628.json') as json_file:
    data = json.load(json_file)
    
regions_dict.update(data)


with open('../data/DS9 regions/bubbles_m74.reg') as f:
    dat = f.readlines()
    
m74_bubbles = {}
p = 0
for n,i in enumerate(dat[3:]):
    ra = float(i.split(',')[0][8:])
    dec = float(i.split(',')[1])
    a = float(i.split(',')[2][:-1])
    b = float(i.split(',')[3][:-1])
    
    ang = float(i.split(',')[4].split(')')[0])
    color = i.split()[2].split('=')[-1]
    
    if b>a:
        ang -= 90
        t = a
        a = b
        b = t
    
    radius = float(i.split(',')[2].split('"')[0])

    m74_bubbles[f'bubble_m74_{p}'] = {'ra' : ra,
                                  'dec': dec,
                                  'a'  : a,
                                  'b'  : b,
                                  'ang':ang,
                                   'color':color}
    p += 1
        
regions_dict.update(m74_bubbles);

df_m74_bubbles = pd.DataFrame.from_dict(m74_bubbles, orient='index')

Av_dict = { 
            'f275w': 2.02499,
            'f336w': 1.67536,
            'f435w': 1.33879,
            'f555w': 1.03065,
            'f814w': 0.59696,
    
            'f090w': 0.583,
            'f115w': 0.419,
            'f150w': 0.287,
            'f200w': 0.195,
    
            'f438w': 1.34148,
            'f606w': 0.90941,
            'f814w': 0.59845,
    
           'F275W': 2.02499,
           'F336W': 1.67536,
           'F438W': 1.34148,
           'F555W': 1.04089,
           'F814W': 0.59845
          };

In [ ]:
def jwst_data(tab, ra_center=0, dec_center=0,r_out=24/3600, ang=245.00492,
              region_type='box', a=10,b=10, skip_col_change=False):

    if region_type == 'box':
        tab_filt = box(tab,'ra','dec',ra_center, dec_center, 0,0,r_out/3600, r_out/3600, ang)

    elif region_type == 'ellipse':
        tab_filt = ellipse(tab, 'ra', 'dec', ra_center, dec_center, ang, 0, 0, a2=a/3600, b2=b/3600)
        
    elif region_type == 'circle':
        tab['r'] = angular_separation(tab['ra']*u.deg,tab['dec']*u.deg,
                                      ra_center*u.deg, dec_center*u.deg).to(u.arcsec).value
        tab_filt = tab[tab['r']<=r_out]
        
    df = tab_filt.to_pandas()
    col_dict = {'ra'                 : 'RA',
                'dec'                : 'DEC',
                'mag_vega_F115W'     : 'fw1',
                'mag_err_F115W'      : 'fw1_error',
                'mag_vega_F150W'     : 'fw2',
                'mag_err_F150W'      : 'fw2_error',
                'mag_vega_F200W'     : 'fw3',
                'mag_err_F200W'      : 'fw3_error'}

    if not skip_col_change:
        df_filt = df.rename(columns = col_dict)
    
    return df_filt

In [ ]:
def hst_data(tab, ra_center=0, dec_center=0,r_out=24/3600, ang=245.00492,region_type='circle'):

    comp_f115w = models.Polynomial1D(2,c0=27.32470673, c1=1.21351676, c2=-43.98416336)
    comp_f150w = models.Polynomial1D(2,c0=24.30894105, c1=17.79650983, c2=-57.09052021)
    comp_f200w = models.Polynomial1D(2,c0=25.84455633, c1=6.86355348, c2=-90.87952389)

    if region_type == 'box':
        tab_filt = box(tab,'ra','dec',ra_center, dec_center,0,0, r_out/3600, r_out/3600, ang)
        
    elif region_type == 'circle':
        tab['r'] = angular_separation(tab['ra']*u.deg,tab['dec']*u.deg,
                                      ra_center*u.deg, dec_center*u.deg).to(u.arcsec).value
        tab_filt = tab[tab['r']<=r_out]
    else:
        print(f'{region_type} NOT available!')
        
    df = tab_filt.to_pandas()
    col_dict = {'ra_1'                 : 'RA',
                'dec_1'                : 'DEC',
                'mag_vega_F435W'       : 'fw1',
                'mag_err_F435W'        : 'fw1_error',
                'mag_vega_F555W'       : 'fw2',
                'mag_err_F555W'        : 'fw2_error',
                'mag_vega_F814W'       : 'fw3',
                'mag_err_F814W'        : 'fw3_error'}
    
    df_filt = df.rename(columns = col_dict)
    fw1_lim = 30#comp_f115w(np.nanmedian(df['crowd_1']))
    fw2_lim = 27#comp_f150w(np.nanmedian(df['crowd_2']))
    fw3_lim = 30#comp_f200w(np.nanmedian(df['crowd']))
    
    return df_filt, [fw1_lim, fw2_lim, fw3_lim], [ra_center, dec_center]

# **Clusters**

In [ ]:
cluster_catalogs = glob.glob("../data/catalogs/ngc628/clusters/*human*")

In [ ]:
tabs = []
for c in cluster_catalogs:
    tabs.append(Table.read(c))
tab_cluster = vstack(tabs)

In [ ]:
ssp_df = pd.read_csv('../data/ssps/bc2003_hr_stelib_m162_kroup_ssp.wfc3_uvis1_color',
                     skiprows=30, delimiter=r'\s{3,}', engine='python',
                     names=['log-age-yr', 'Vmag', 'Kmag', 'V-F225w', 'V-F336w',
                            'V-F438w', 'V-F547m', 'V-F555w', 'V-F606w', 'V-F625w',
                            'V-F656n', 'V-F657n', 'V-F658n', 'V-F814w', 'log Nly',
                            'Mt/Lb', 'Mt/Lv', 'Mt/Lk'])

In [ ]:
tab_cluster_bubbles = []
clust_ages_t = ssp_df['log-age-yr'].values
x = -ssp_df['V-F438w'] + ssp_df['V-F814w'] + (Av_dict['F438W'] - Av_dict['F814W'])*0.19
y = -ssp_df['V-F336w'] + ssp_df['V-F438w'] + (Av_dict['F336W'] - Av_dict['F438W'])*0.19

clust_t25 = []
for i, row in df_m74_bubbles_s00.iterrows():
    tab_temp = ellipse(tab_cluster,'PHANGS_RA','PHANGS_DEC',
                       row['ra'], row['dec'], row['ang'],
                       0,0,
                       row['a']/3600, row['b']/3600)
    
    tab_temp['Bubble ID'] = row['Bubble ID']
    clust_ages = []

    for row in tab_temp:
        x_obs = row['PHANGS_F435W_VEGA'] - row['PHANGS_F814W_VEGA']
        y_obs = row['PHANGS_F336W_VEGA'] - row['PHANGS_F435W_VEGA']
        
        d = np.sqrt((x-x_obs)**2 + (y-y_obs)**2)
        clust_ages.append(clust_ages_t[d==d.min()][-1])
    tab_temp['logAge'] = clust_ages
    if len(tab_temp)>0:
        tab_cluster_bubbles.append(tab_temp)
tab_cluster_bubbles = vstack(tab_cluster_bubbles)

In [ ]:
# Gen Cluster CMD with 336-435 vs 555

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))


df_filt = tab_cluster_bubbles
df_filt = df_filt[(df_filt['PHANGS_F555W_VEGA']>0) & (df_filt['PHANGS_F814W_VEGA']>0) ]
df_filt = df_filt[(df_filt['PHANGS_F275W_VEGA']>0) & (df_filt['PHANGS_F435W_VEGA']>0) ]

df_filt['logAge'][2] = 7.7
df_filt['logAge'][11] = 6.5
df_filt['logAge'][14] = 6.0
df_filt['logAge'][26] = 7.7

x = df_filt['PHANGS_F435W_VEGA'].value - df_filt['PHANGS_F814W_VEGA'].value
y = df_filt['PHANGS_F336W_VEGA'].value - df_filt['PHANGS_F435W_VEGA'].value
c = df_filt['logAge']

p = 0
for i,j,k in zip(x,y,c):
    ax.annotate(f"{p}: {10**k/1e6:0.2f}",(i,j), fontsize=10, zorder=200)
    p+=1
  # 
df_temp = df_filt[df_filt['PHANGS_CLUSTER_CLASS_HUMAN']==1]
x = df_temp['PHANGS_F435W_VEGA'].value - df_temp['PHANGS_F814W_VEGA'].value
y = df_temp['PHANGS_F336W_VEGA'].value - df_temp['PHANGS_F435W_VEGA'].value
c = df_temp['PHANGS_CLUSTER_CLASS_HUMAN']

ax.scatter(x,y, s=200, color='red', zorder=100, edgecolor='black', label= 'PHANGS Cluster Class 1')

df_temp = df_filt[df_filt['PHANGS_CLUSTER_CLASS_HUMAN']==2]
x = df_temp['PHANGS_F435W_VEGA'].value- df_temp['PHANGS_F814W_VEGA'].value
y = df_temp['PHANGS_F336W_VEGA'].value - df_temp['PHANGS_F435W_VEGA'].value
c = df_temp['PHANGS_CLUSTER_CLASS_HUMAN']

ax.scatter(x,y, s=200, color='green', zorder=100, edgecolor='black', label= 'PHANGS Cluster Class 2')


df_temp = df_filt[df_filt['PHANGS_CLUSTER_CLASS_HUMAN']==3]
x = df_temp['PHANGS_F435W_VEGA'].value - df_temp['PHANGS_F814W_VEGA'].value
y = df_temp['PHANGS_F336W_VEGA'].value - df_temp['PHANGS_F435W_VEGA'].value
c = df_temp['PHANGS_CLUSTER_CLASS_HUMAN']

ax.scatter(x,y, s=200, color='blue', zorder=100, edgecolor='black', label= 'PHANGS Cluster Class 3')

# Theoretical model
x = -ssp_df['V-F438w'] + ssp_df['V-F814w'] + (Av_dict['F438W'] - Av_dict['F814W'])*0.19
y = -ssp_df['V-F336w'] + ssp_df['V-F438w'] + (Av_dict['F336W'] - Av_dict['F438W'])*0.19

ax.plot(x,y,'-', lw=2, color='gray', zorder=2)

# Age labels
ages = np.array([6.,6.54,7,7.51, 7.72,8.01,8.41, 9.11, 9.51,10])

ages_name = []

for i in ages:
  if i > 9:
    i-=9
    i = np.round(10**i,0)
    ages_name.append(f'{i:g} Gyr')
  elif i >=6:
    i-=6
    i = np.round(10**i,0)
    ages_name.append(f'{i:g} Myr')

ind = [True if i in ages else False for i in np.round(ssp_df['log-age-yr'],2) ]
ssp_age_cut = ssp_df[ind]


x = -ssp_age_cut['V-F438w'] + ssp_age_cut['V-F814w'] + (Av_dict['F438W'] - Av_dict['F814W'])*0.19
y = -ssp_age_cut['V-F336w'] + ssp_age_cut['V-F438w'] + (Av_dict['F336W'] - Av_dict['F438W'])*0.19

ax.scatter(x,y, s=40, lw=2, color='black', zorder=111, marker='x')

for i,j,k in zip(x,y, ages_name):
  ax.annotate(k,(-1.5,j), fontsize=15)


ax.set_xlabel('F435W - F814W')
ax.set_ylabel('F336W - F435W');
# Reddening Vector

Av = 1
AF1 =  Av_dict['F438W']*Av
AF2 =  Av_dict['F814W']*Av
AF3 =  Av_dict['F336W']*Av
AF4 =  Av_dict['F438W']*Av

dx = AF1 - AF2
dy = AF3 - AF4

Av_x = -0.48
Av_y = 0.5
ax.annotate('', xy=(Av_x, Av_y),
              xycoords='data',
              xytext=(Av_x+dx, Av_y+dy),
              textcoords='data',
              arrowprops=dict(arrowstyle= '<|-',
                              color='black',
                              lw=2,
                              ls='-')
            )

ax.annotate(f'Av = {Av}', xy=(Av_x-0.1, Av_y-0.1), fontsize=20)


# Aesthetics
ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())


ax.tick_params(which='both', length=14,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=3)
#ax.legend(fontsize=15)
ax.set_xlim(-2,2.7)
ax.set_ylim(-1.8,None)
#ax.axhline(-0.47)
plt.show()


In [ ]:
df_cluster = df_filt.to_pandas()

In [ ]:
df_s1_clusters = []
for ID in df_m74_bubbles_s1['Bubble ID']:
    # if ID not in [1]:
    #     continue
    mask_bub = df_m74_bubbles_s1['Bubble ID'] == ID

    if mask_bub.any():

        bub_ra, bub_dec = df_m74_bubbles_s1.loc[mask_bub, ['ra', 'dec']].values.ravel()
        
        ang  = df_m74_bubbles_s1.loc[mask_bub, ['ang']].values.ravel()[0]
        a, b = df_m74_bubbles_s1.loc[mask_bub, ['a','b']].values.ravel()
        
        df_clust = ellipse(df_filt,'PHANGS_RA','PHANGS_DEC',
                           bub_ra, bub_dec, ang,
                           0,0,
                           a/3600, b/3600).to_pandas()
        
        if len(df_clust)>0:
        
            df_clust['r_ratio'] = np.nan
            df_clust['t25']     = df_m74_bubbles_s1.loc[mask_bub, 't25'].values[0]
            df_clust['t75']     = df_m74_bubbles_s1.loc[mask_bub, 't75'].values[0]
            
            q = b/a
    
            stars  = SkyCoord(df_clust['PHANGS_RA'], df_clust['PHANGS_DEC'], unit='deg')
            center = SkyCoord(bub_ra*u.deg, bub_dec*u.deg)
            dra, ddec = stars.spherical_offsets_to(center)  # returns Angles
            x_east  = dra.to(u.arcsec).value
            y_north = ddec.to(u.arcsec).value
        
            # rotate into ellipse frame
            pa_rad = np.radians(ang)
            xprime =  x_east*np.sin(pa_rad) + y_north*np.cos(pa_rad)
            yprime = -x_east*np.cos(pa_rad) + y_north*np.sin(pa_rad)
        
            # equivalent semi-major/minor axes
            a_eq = np.sqrt(xprime**2 + (yprime/q)**2)
            b_eq = q * a_eq
            r = np.sqrt(a_eq*b_eq)
    
            # --- assign r_ratio ---
            r_bub = df_m74_bubbles_s1.loc[mask_bub, 'r'].values[0]/44
            df_clust['r_ratio'] = r[0] /r_bub
            df_s1_clusters.append(Table.from_pandas(df_clust))

df_s1_clusters = vstack(df_s1_clusters)

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))

df_temp = df_s1_clusters[df_s1_clusters['logAge']<=8]
x = df_temp['r_ratio']
y = 10**df_temp['logAge']/10**df_temp['t25']
ax.scatter(x,y,s=200, edgecolor='black', facecolor='grey')

ax.axhline(y=1, linestyle='--', color='red', lw=2)
ax.set_xlabel(r'r/R$_{\mathrm{B}}$')
ax.set_ylabel(r't$_{\mathrm{c}}$/t$_{\mathrm{25}}$')
ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.set_xlim(-0.15,None)

x_w = 0.15
y_w = 0.4
p = plt.Rectangle((0-x_w/2, 1-y_w/2), x_w, y_w, fill=False, lw=3, edgecolor='black', ls='--')
ax.add_patch(p)

ax.tick_params(which='both', length=14,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=3)

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))

df_filt = tab_cluster_bubbles[tab_cluster_bubbles['logAge']<=8]
df_filt = df_filt[(df_filt['PHANGS_F555W_VEGA']>0) & (df_filt['PHANGS_F814W_VEGA']>0) ]
df_filt = df_filt[(df_filt['PHANGS_F275W_VEGA']>0) & (df_filt['PHANGS_F435W_VEGA']>0) ]

df_temp = df_filt
y = df_temp['PHANGS_F555W_VEGA'].value
x = df_temp['PHANGS_F336W_VEGA'].value - df_temp['PHANGS_F435W_VEGA'].value
c = df_temp['logAge']
img = ax.scatter(x,y, c=c, s=200, cmap='jet', zorder=100, edgecolor='black')

cb = plt.colorbar(img, ax=ax, orientation='horizontal', pad=0.13, aspect = 45)
cb.set_label('log(Age [yr])', fontsize=20)

# Theoretical model
M = -2.5*np.log10(1e5)
y = -ssp_df['V-F555w'] + ssp_df['Vmag'] + (Av_dict['F555W'])*0.19 + 29.81 + M
x = -ssp_df['V-F336w'] + ssp_df['V-F438w'] + (Av_dict['F336W'] - Av_dict['F438W'])*0.19

ax.plot(x,y,'-', lw=2, color='gray', zorder=2)

M = -2.5*np.log10(1e4)
y = -ssp_df['V-F555w'] + ssp_df['Vmag'] + (Av_dict['F555W'])*0.19 + 29.81 + M
x = -ssp_df['V-F336w'] + ssp_df['V-F438w'] + (Av_dict['F336W'] - Av_dict['F438W'])*0.19

ax.plot(x,y,'--', lw=2, color='gray', zorder=2)

M = -2.5*np.log10(1e3)
y = -ssp_df['V-F555w'] + ssp_df['Vmag'] + (Av_dict['F555W'])*0.19 + 29.81 + M
x = -ssp_df['V-F336w'] + ssp_df['V-F438w'] + (Av_dict['F336W'] - Av_dict['F438W'])*0.19

ax.plot(x,y, ls = (0, (1, 1)), lw=2, color='gray', zorder=2)
# Age labels
ages = np.array([6.,6.54,7,7.51, 7.72,8.01,8.41, 9.11, 9.51,10])

ages_name = []

for i in ages:
  if i > 9:
    i-=9
    i = np.round(10**i,2)
    ages_name.append(f'{i} Gyr')
  elif i >=6:
    i-=6
    i = np.round(10**i,2)
    ages_name.append(f'{i} Myr')

ind = [True if i in ages else False for i in np.round(ssp_df['log-age-yr'],2) ]
ssp_age_cut = ssp_df[ind]

y = -ssp_age_cut['V-F555w'] + ssp_age_cut['Vmag'] + (Av_dict['F555W'])*0.19 + 29.81 + M
x = -ssp_age_cut['V-F336w'] + ssp_age_cut['V-F438w'] + (Av_dict['F336W'] - Av_dict['F438W'])*0.19

#ax.scatter(x,y, s=40, lw=2, color='black', zorder=111, marker='x')

ax.set_ylabel('F555W')
ax.set_xlabel('F336W - F435W');
# Reddening Vector

Av  = 1
AF1 = Av_dict['F336W']*Av
AF2 = Av_dict['F438W']*Av
AF3 = Av_dict['F555W']*Av

dx = AF1 - AF2
dy = AF3

Av_x = -1
Av_y = 18
ax.annotate('', xy=(Av_x, Av_y),
              xycoords='data',
              xytext=(Av_x+dx, Av_y+dy),
              textcoords='data',
              arrowprops=dict(arrowstyle= '<|-',
                              color='black',
                              lw=2,
                              ls='-')
            )

ax.annotate(f'Av = {Av}', xy=(Av_x-0.1, Av_y-0.1), fontsize=20)


# Aesthetics
ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())


ax.tick_params(which='both', length=14,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=3)
ax.legend(fontsize=15)
ax.set_xlim(None,-0.5)
ax.set_ylim(None,25)
#ax.axhline(-0.47)
ax.invert_yaxis()
plt.show()


In [ ]:
Mc = [r'$10^3~M_{\odot}$']

In [ ]:
Mc

# **CMD**

In [ ]:
df_cmd_jwst = pd.read_csv("../data/isochrones_master/cmd_jwst_corr.csv")

In [ ]:
Av_dict = { 
            'f275w': 2.02499,
            'f336w': 1.67536,
            'f435w': 1.33879,
            'f555w': 1.03065,
            'f814w': 0.59696,
    
            'f090w': 0.583,
            'f115w': 0.419,
            'f150w': 0.287,
            'f200w': 0.195,
    
            'f438w': 1.34148,
            'f606w': 0.90941,
            'f814w': 0.59845
          }

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.coordinates import angular_separation
import astropy.units as u
from scipy.stats import gaussian_kde
from astropy.modeling import models, fitting
import seaborn as sb

from matplotlib.colors import LinearSegmentedColormap
import subprocess
import pandas as pd

In [ ]:
from matplotlib.ticker import ScalarFormatter

In [ ]:
def gen_CMD(
    tab,
    df_iso = None,
    filters={'filt1': 'f115w', 'filt2': 'f150w'},
    positions={'ra_col': 'ra', 'dec_col': 'dec', 'ra_cen': 0, 'dec_cen': 0},
    region={'r_in': 0, 'r_out': 24, 'spatial_filter': 'circle','ang': 245.00492},
    extinction={'Av': 0.19, 'Av_': 3, 'Av_x': 2, 'Av_y': 19},
    distance_modulus=29.7415,
    axis_limits={'xlims': [-0.5, 2.5], 'ylims': [18, 28]},
    isochrone_params={'met': 0.02, 'ages': [7., 8., 9.]},
    plot_settings={'alpha': 1, 's': 0.2, 'lw': 3},
    error_settings={ 'mag_err_lim': 0.2, 'show_err_model': False, 'ref_xpos': -0.5},
    kde_contours={'gen_kde': False, 'gen_contours': False},
    other_settings={'ab_dist': True, 'skip_data': False, 'show_err_model':False},
    fig=None,
    ax=None):
    
    """
    Generate a Color-Magnitude Diagram (CMD) with optional KDE or contour overlays.

    Parameters
    ----------
    tab : DataFrame
        Input data table containing magnitudes, positions, and errors for sources.
        
    df_iso : DataFrame, optional
        Isochrone data for overlay.

    filters : dict, optional
        Filters used in CMD. Keys:
        - 'filt1': Primary filter for color calculation.
        - 'filt2': Secondary filter for color calculation.
        - 'filt3': Filter for magnitude axis. Defaults to 'filt2'.

    positions : dict, optional
        Positional parameters. Keys:
        - 'ra_col': RA column name.
        - 'dec_col': DEC column name.
        - 'ra_cen': Central RA (in degrees).
        - 'dec_cen': Central DEC (in degrees).

    region : dict, optional
        Region parameters for filtering sources. Keys:
        - 'r_in': Inner radius for selection (arcseconds) (used for circular filters).
        - 'r_out': Outer radius for selection (arcseconds) (used for circular filters).
        - 'spatial_filter': Type of spatial filtering ('circle', 'box', 'ellipse').
        - 'ang': Orientation angle (used for box or ellipse filters).
        - 'width_in', 'height_in': Inner box dimensions (arcseconds) (used for box filters).
        - 'width_out', 'height_out': Outer box dimensions (arcseconds) (used for box filters).
        - 'a1', 'b1': Inner semi-major and semi-minor axes (arcseconds) (used for ellipse filters).
        - 'a2', 'b2': Outer semi-major and semi-minor axes (arcseconds) (used for ellipse filters).

    extinction : dict, optional
        Extinction parameters. Keys:
        - 'Av': Extinction value.
        - 'Av_': Annotation extinction value.
        - 'Av_x', 'Av_y': Annotation arrow position.

    distance_modulus : float, optional
        Distance modulus for CMD adjustments. Default is 29.7415.

    axis_limits : dict, optional
        Plot axis limits. Keys:
        - 'xlims': Limits for x-axis (color).
        - 'ylims': Limits for y-axis (magnitude).

    isochrone_params : dict, optional
        Isochrone parameters for plotting. Keys:
        - 'met': Metallicity for isochrones.
        - 'label_min': Minimum label value for filtering.
        - 'label_max': Maximum label value for filtering.
        - 'ages': List of log ages to plot.

    plot_settings : dict, optional
        General plot settings. Keys:
        - 'alpha': Transparency for isochrone lines.
        - 's': Marker size for scatter plots.
        - 'lw': Line width for isochrones.

    error_settings : dict, optional
        Settings for error handling and plotting. Keys:
        - 'mag_err_cols': Columns for magnitude errors. Defaults to filter-based columns.
        - 'mag_err_lim': Maximum allowable magnitude error.
        - 'show_err_model': Show error models during plotting.
        - 'ref_xpos': Reference x-position for error bars.

    kde_contours : dict, optional
        Settings for KDE or contour plots. Keys:
        - 'gen_kde': Generate KDE overlay.
        - 'gen_contours': Generate contour overlay.

    other_settings : dict, optional
        Miscellaneous settings. Keys:
        - 'ab_dist': Include absolute distance modulus adjustments.
        - 'skip_data': Skip scatter plot of source data.

    fig : matplotlib.figure.Figure, optional
        Existing figure object. If None, a new figure is created.

    ax : matplotlib.axes.Axes, optional
        Existing axis object. If None, a new axis is created.


    Returns
    -------
    tuple
        (fig, ax, tab) where:
        - fig: The figure object.
        - ax: The axis object.
        - tab: The filtered input data table after spatial and error-based selection.
    """
    
    # Fill in default values for nested dictionaries
    filters.setdefault('filt1','f115w')
    filters.setdefault('filt2','f200w')
    filters.setdefault('filt3', filters['filt2'])
    
    positions.setdefault('ra_col','ra')
    positions.setdefault('dec_col','dec')
    positions.setdefault('ra_cen',0)
    positions.setdefault('dec_cen',0)
    
    region.setdefault('r_in',0)
    region.setdefault('r_out',10)
    region.setdefault('spatial_filter','circle')
    
    extinction.setdefault('Av',0.19)
    extinction.setdefault('Av_',3)
    extinction.setdefault('Av_x',3)
    extinction.setdefault('Av_y',19)
    
    axis_limits.setdefault('xlims', [-0.5, 2.5])
    axis_limits.setdefault('ylims', [18, 28])
    
    isochrone_params.setdefault('label_min', 0)
    isochrone_params.setdefault('label_max', 10)
    isochrone_params.setdefault('met', [0.02])
    isochrone_params.setdefault('age', [7,8,9])
    
    plot_settings.setdefault('Av.fontsize',20)
    plot_settings.setdefault('legend.fontsize',20)
    plot_settings.setdefault('lw',3)
    plot_settings.setdefault('s',0.2)
    plot_settings.setdefault('alpha',1)
    plot_settings.setdefault('print_met',False)
    plot_settings.setdefault('legend.ncols',1)
    
    
    error_settings.setdefault('mag_err_cols', [
        f'mag_err_{filters["filt1"].upper()}',
        f'mag_err_{filters["filt2"].upper()}',
        f'mag_err_{filters["filt3"].upper()}',])
    
    error_settings.setdefault('mag_err_lim',0.2)
    error_settings.setdefault('ref_xpos',-0.25)
    
    kde_contours.setdefault('gen_kde',False)
    kde_contours.setdefault('gen_contours',False)
    kde_contours.setdefault('kde_bin',100j)
    kde_contours.setdefault('cmap','jet')
    kde_contours.setdefault('bw', 0.05)
    
    other_settings.setdefault('ab_dist',True)
    other_settings.setdefault('skip_data',False)
    other_settings.setdefault('show_err_model',False)

    # Filter table by magnitude errors
    for col in error_settings['mag_err_cols']:
        tab = tab[tab[col] <= error_settings['mag_err_lim']]

    # Compute angular separation or define square field
    tab['r'] = angular_separation(
        tab[positions['ra_col']] * u.deg,
        tab[positions['dec_col']] * u.deg,
        positions['ra_cen'] * u.deg,
        positions['dec_cen'] * u.deg).to(u.arcsec).value

    if region['spatial_filter']=='circle':
        tab = tab[(tab['r'] >= region['r_in'])
                  & (tab['r'] <= region['r_out'])]
        
    elif region['spatial_filter']=='box':   
        tab = box(tab, positions['ra_col'], positions['dec_col'],
                  positions['ra_cen'], positions['dec_cen'],
                  region['width_in'] / 3600, region['height_in'] / 3600,
                  region['width_out'] / 3600, region['height_out'] / 3600,
                  region['ang'])

    elif region['spatial_filter']=='ellipse':
        tab = ellipse(tab, positions['ra_col'], positions['dec_col'],
                  positions['ra_cen'], positions['dec_cen'],
                  region['ang'], 
                  region['a1'] / 3600,region['b1'] / 3600,
                  region['a2'] / 3600,region['b2'] / 3600)
    elif region['spatial_filter']=='polygon':
        tab = polygon(tab, positions['ra_col'], positions['dec_col'], region['points'])

    # Compute magnitudes and colors
    x = tab[f'mag_vega_{filters["filt1"].upper()}'] - tab[f'mag_vega_{filters["filt2"].upper()}']
    y = tab[f'mag_vega_{filters["filt3"].upper()}']

    x = x.value.astype(float)
    y = y.value.astype(float)

    # Initialize figure and axis if not provided
    if fig is None or ax is None:
        fig, ax = plt.subplots(figsize=(12, 10))

    # Extinction corrections
    AF1 = Av_dict[filters['filt1']] * extinction['Av']
    AF2 = Av_dict[filters['filt2']] * extinction['Av']
    AF3 = Av_dict[filters['filt3']] * extinction['Av']

    # Kernel density estimation or scatter plot
    tick_color = 'black'
    if kde_contours['gen_kde'] and not kde_contours['gen_contours']:
        xx, yy = np.mgrid[
            axis_limits['xlims'][0]:axis_limits['xlims'][1]:kde_contours['kde_bin'],
            axis_limits['ylims'][0]:axis_limits['ylims'][1]:kde_contours['kde_bin']]
        
        positions = np.vstack([xx.ravel(), yy.ravel()])
        values = np.vstack([x, y])

        kernel = gaussian_kde(values, bw_method=kde_contours['bw'])
        f = np.reshape(kernel(positions), xx.shape)
        tick_color='white'
        perc_cut = np.percentile(f.ravel(), 84)

        f[f<=perc_cut] = np.nan
        norm = simple_norm(f.T, 'log', min_percent=10)
        ax.imshow(f.T, cmap=kde_contours['cmap'], extent=(*axis_limits['xlims'], *axis_limits['ylims']),
                  interpolation='nearest', aspect='auto', norm=norm, zorder=100, origin='lower')

    elif kde_contours['gen_contours']:
        ax.scatter(x, y, s=plot_settings['s'], color='black', label='data')
        cmap_custom = LinearSegmentedColormap.from_list("custom_grey_to_white", ["grey", "white"], N=256)
        sb.kdeplot(x=x, y=y, levels=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99],
                   ax=ax, fill=True, cmap=cmap_custom)
        
        sb.kdeplot(x=x, y=y, levels=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99],
                   ax=ax, color='black')

    if not other_settings['skip_data']:
        ax.scatter(x, y, s=plot_settings['s'], color='black', label='data', zorder=50)
        
    ax.set_xlim(axis_limits['xlims'][0],axis_limits['xlims'][1])
    ax.set_ylim(axis_limits['ylims'][0],axis_limits['ylims'][1])
    
    ax.tick_params(which='both', length=15,direction="in", 
                   bottom=True, top=True,left=True, width = 3,
                   color=tick_color)
    
    ax.tick_params(which='minor', length=8, width = 3, color=tick_color)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_major_locator(AutoLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    
    # Handle isochrones
        
    age_lin = []
    for i in isochrone_params['ages']:
        if i < 6:
            age_lin.append(f'{np.round(10**i,1)} Myr')
        if i >= 6  and i < 9:
            i -= 6
            age_lin.append(f'{np.round(10**i,1)} Myr')
        elif i >= 9:
            i -= 9
            age_lin.append(f'{np.round(10**i,1)} Gyr')
            
    if df_iso is not None:
        df_iso = df_iso[(df_iso['label']>=isochrone_params['label_min'])
                       & (df_iso['label']<=isochrone_params['label_max'])]
                        
        for i,age in enumerate(isochrone_params['ages']):
            t = df_iso[(np.round(df_iso['logAge'],1) == age)]
            for Z in isochrone_params['met']:
                subset = t[t['Zini'] == Z]
                x_iso = subset[f"{filters['filt1'].upper()}mag"] + AF1 - (
                        subset[f"{filters['filt2'].upper()}mag"] + AF2)
                y_iso = subset[f"{filters['filt3'].upper()}mag"] + AF3 + distance_modulus
                       
                mask = (y_iso.values[1:]- y_iso.values[:-1])<3
                mask = np.array([True] + list(mask))
                mask = np.where(~mask, np.nan, 1)
                
                if len(isochrone_params['met'])>1 or plot_settings['print_met']:
                    label = label=age_lin[i]+ f' {Z}'
                else:
                    label = label=age_lin[i]
                               
                ax.plot(x_iso*mask, y_iso*mask, lw=plot_settings['lw'],
                        label=label,alpha=plot_settings['alpha'], zorder=200)

    # Absolute magnitude
    if other_settings['ab_dist']:
        yticks = ax.get_yticks()
        yticks_n = yticks - distance_modulus - AF3
        
        dy = yticks_n - np.floor(yticks_n)
        ax1 = ax.twinx()  # instantiate a second axes that shares the same x-axis            
        ax1.set_ylabel(r'$M_{' + f"{filters['filt3'].upper()}" + r'}$')  # we already handled the x-label with ax1
        ax1.set_yticks(yticks - dy, np.floor(yticks_n), fontsize=30)
        ax1.yaxis.set_minor_locator(AutoMinorLocator())
        
        ax1.set_xlim(axis_limits['xlims'][0],axis_limits['xlims'][1])
        ax1.set_ylim(axis_limits['ylims'][0],axis_limits['ylims'][1])
        
        ax1.tick_params(which='both', length=15,direction="in",
                        right=True, width = 3, color=tick_color)
        ax1.tick_params(which='minor', length=8, width = 3, color=tick_color)
        
        
        ax1.invert_yaxis()

    # Extinction Vector
    AF1_ = Av_dict[filters['filt1']] * extinction['Av_']
    AF2_ = Av_dict[filters['filt2']] * extinction['Av_']
    AF3_ = Av_dict[filters['filt3']] * extinction['Av_']

    dx = AF1_ - AF2_
    dy = AF3_

    ax.annotate('', xy=(extinction['Av_x'], extinction['Av_y']),
                 xycoords='data',
                 xytext=(extinction['Av_x']+dx, 
                         extinction['Av_y']+dy),
                 textcoords='data',
                 arrowprops=dict(arrowstyle= '<|-',
                                 color=tick_color,
                                 lw=plot_settings['lw'],
                                 ls='-')
               )

    ax.annotate(f"Av = {extinction['Av_']}",
                xy=(extinction['Av_x']-0.1, extinction['Av_y']-0.1)
                ,fontsize=plot_settings['Av.fontsize'],
                color=tick_color)
    
    # Error models
    if not other_settings['skip_data']:
        ref = tab[f"mag_vega_{filters['filt3'].upper()}"]
        ref_new = np.arange(np.ceil(y.min()),np.floor(y.max())+0.5,0.5)

        mag_err1 = tab[error_settings['mag_err_cols'][0]]
        mag_err2 = tab[error_settings['mag_err_cols'][1]]

        if len(error_settings['mag_err_cols'])>2:
            mag_err3 = tab[error_settings['mag_err_cols'][2]]
        else:
            mag_err3 = mag_err2

        col_err = np.sqrt(mag_err1**2 + mag_err2**2)
        init = models.Exponential1D()
        fit = fitting.LevMarLSQFitter()
        model_col = fit(init,ref,col_err)

        init = models.Exponential1D()
        fit = fitting.LevMarLSQFitter()
        model_mag = fit(init,ref,mag_err3)

        x = error_settings['ref_xpos'] + 0*ref_new
        y = ref_new
        yerr = model_mag(ref_new)
        xerr = model_col(ref_new)
        
        if other_settings['show_err_model']:
            plt.show()
            plt.scatter(ref, mag_err3)
            plt.plot(ref_new,yerr,'--r')
            plt.show()
            plt.scatter(ref, col_err)
            plt.plot(ref_new,xerr,'--r')
            plt.show()
        ax.errorbar(x, y, yerr, xerr ,fmt='o', color = 'red', markersize=0.5, capsize=2) 
    
    # Labels, ticks, and legend
    ax.invert_yaxis()
    ax.set_xlabel(f"{filters['filt1'].upper()} - {filters['filt2'].upper()}")
    ax.set_ylabel(filters['filt3'].upper())
    ax.legend(fontsize=plot_settings['legend.fontsize'], ncols = plot_settings['legend.ncols'])

    fig.tight_layout()
    
    return fig, ax, tab

In [ ]:
tab_spatial = Table.from_pandas(pd.read_csv("../data/NGC628_spatial.csv"))

In [ ]:
n_50 = []
n_50_c = []
df_m74_bubbles_s0 = df_m74_bubbles[df_m74_bubbles['class']==1]

tab_temp = Table()

ras  = np.linspace(tab_spatial['RA'].min(), tab_spatial['RA'].max(), 400)
decs = np.linspace(tab_spatial['DEC'].min(), tab_spatial['DEC'].max(), 400)
y,x = np.meshgrid(ras,decs)
tab_temp['RA']  = y.ravel()
tab_temp['DEC'] = x.ravel()

df_temp = box(tab_temp,'RA','DEC',regions_dict['ngc628']['ra'],regions_dict['ngc628']['dec'], 0,0, 6/60, 2/60)

ras  = df_temp['RA']
decs = df_temp['DEC']

r = angular_separation(ras*u.deg, decs*u.deg, 
                        regions_dict['ngc628']['ra']*u.deg, regions_dict['ngc628']['dec']*u.deg).to(u.arcsec).value


r_ind = np.argsort(r)

ras  = ras[r_ind]
decs = decs[r_ind]


for i, row in df_m74_bubbles_s0.iterrows():
    a = row['a']
    b = row['b']
    ra = row['ra']
    dec = row['dec']
    ang = row['ang']
    
    df_filt = ellipse(tab_spatial,'RA','DEC',ra,dec,ang, 0,0,a/3600,b/3600)
    df_filt = df_filt[df_filt['Log_Age']<=7.7]
    n_50.append(len(df_filt))

    ns = []
    for i in range(5):
        j = np.random.randint(0, len(ras))
    
        ang = np.random.uniform(0,360)
        
        df_filt = ellipse(tab_spatial,'RA','DEC',ras[j],decs[j],ang, 0,0,a/3600,b/3600)
        df_filt = df_filt[df_filt['Log_Age']<=7.7]
        ns.append(len(df_filt))
    n_50_c.append(ns)


In [ ]:
n_50_c = np.array(n_50_c)

In [ ]:
n_50_c.shape

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))

x = np.sqrt(df_m74_bubbles_s0['a']*df_m74_bubbles_s0['b'])*44 
y = n_50_c

for i in range(5):
    y = n_50_c[:,i]
    ax.scatter(x,y,s= 10, facecolor='red',edgecolor='black')

y = n_50

ax.scatter(x,y,s= 50, edgecolor='black')

#ax.xaxis.set_major_locator(LogLocator())
#ax.xaxis.set_minor_locator(AutoMinorLocator())

#ax.yaxis.set_major_locator(LogLocator())
#ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=2)

ax.set_xlabel(r'R$_{\rm Bubble}$ [pc]')
#ax.set_xlim(20,1e3)
#ax.set_ylim(0.8,50)
ax.tick_params(axis='x', pad=10)

ax.set_ylabel(r'N$_{\rm SG}$')
ax.set_yscale('log');
ax.set_xscale('log');

In [ ]:
df_m74_bubbles_s0['n50'] = n_50

df_m74_bubbles_s0['gal_r'] = angular_separation(df_m74_bubbles_s0['ra'].values*u.deg, df_m74_bubbles_s0['dec'].values*u.deg,
                                        regions_dict['ngc628']['ra']*u.deg,  regions_dict['ngc628']['dec']*u.deg).to(u.arcsec).value

df_m74_bubbles_s0.to_csv("../data/DS9 regions/m74_bubbles_s0.csv", index=None)

In [ ]:
df_m74_bubbles_s0 = pd.read_csv("../data/DS9 regions/m74_bubbles_s0.csv")

df_m74_bubbles_s1 = df_m74_bubbles_s0[df_m74_bubbles_s0['n50']>=10]
df_m74_bubbles_s2 = df_m74_bubbles_s0[df_m74_bubbles_s0['n50']<10]

df_m74_bubbles_s00 = df_m74_bubbles_s0.sort_values('r', ascending=False)
df_m74_bubbles_s00['Bubble ID'] = np.arange(0,len(df_m74_bubbles_s00),1)
df_m74_bubbles_s00 = df_m74_bubbles_s00.sort_index()

df_m74_bubbles_s1 = df_m74_bubbles_s00[df_m74_bubbles_s00['n50']>=10]
df_m74_bubbles_s2 = df_m74_bubbles_s00[df_m74_bubbles_s00['n50']<10]

p = 0
n_total = []
for i, row in df_m74_bubbles_s1.iterrows():
    df = pd.read_csv(f"../SFH/ngc628/JWST_bubble_obs/{p}_spatial.csv")
    n_total.append(len(df))
    p+=1

df_m74_bubbles_s1['N_total'] = n_total

In [ ]:
df_m74_bubbles_s1.to_csv("../data/DS9 regions/m74_bubbles_s1.csv")

In [ ]:
df_cmd              = df_cmd_jwst[(df_cmd_jwst['logAge']==7.7) & (df_cmd_jwst['Zini']==0.02)]
#df_cmd['F200Wmag'] += 29.81 + Av_dict['f200w']*0.19

In [ ]:
df_cmd = df_cmd[df_cmd['F200Wmag']<=26]

In [ ]:
df_cmd[df_cmd['Mini']==df_cmd['Mini'].min()]['label'].values,  df_cmd['Mini'].min()

In [ ]:
def IMF_Krp( m, ml=0.1, mint=0.5, mu=350.,a1=1.3,a2=2.3):

    h2 = (mu**(1.-a2)-mint**(1.-a2))/(1.-a2)
    h1 = (mint**(1.-a1)-ml**(1.-a1))/(1.-a1)

    c1 = 1./(h1+h2*mint**(a2-a1))
    c2 = c1*mint**(a2-a1)

    c = ones(len(m))
    c[where(m < mint)] = c1
    c[where(m >= mint)] = c2

    a = ones(len(m))
    a[where(m < mint)] = -a1
    a[where(m >= mint)] = -a2
    imf = c*m**a

    return(imf)

In [ ]:
Mx = np.array([42])
Mu = np.array([350])
Ni = np.array([10])
Mlim = np.array([6.8878588676])

alpha1 = 1.3
alpha2 = 2.3
Mc = 0.5
Ml = 0.1

C2 = (Ni*(1-alpha2))/(Mx**(1-alpha2) - Mlim**(1-alpha2))
C1 = C2*Mc**(alpha1 - alpha2)

# Ncorrs
Ncorrs = []
for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni):
    if mlim!=0:
        M = np.linspace(Ml,Mc,10000)  
        Nlim1 = np.trapz(c1*M**(-alpha1), M)

        M = np.linspace(Mc,mlim,10000)  
        Nlim2 = np.trapz(c2*M**(-alpha2), M)

        Ncorr = ni + Nlim1 + Nlim2
    else:
        Ncorr = 0
    Ncorrs.append(Ncorr)
Ncorrs = np.array(Ncorrs)

# int M
for c1, c2, mlim, ni in zip(C1,C2,Mu, Ni):
    if mlim!=0:
        M = np.linspace(Ml,Mc,10000)  
        M1 = np.trapz(M*c1*M**(-alpha1), M)

        M = np.linspace(Mc,mlim,10000)  
        M2 = np.trapz(M*c2*M**(-alpha2), M)

        Mtot = M1 + M2
    else:
        Mtot = 0
    
# Mean Mass
h2 = (Mx**(1.-alpha2)-Mc**(1.-alpha2))/(1.-alpha2)
h2 *= Mc**(alpha2-alpha1)
h1 = (Mc**(1.-alpha1)-Ml**(1.-alpha1))/(1.-alpha1)

C1 = 1/(h1+h2)
C2 = C1*Mc**(alpha2-alpha1)

Mean_M = []
for c1, c2, mu in zip(C1,C2,Mu):
    M = np.linspace(Ml,Mc,10000)  
    Mean_M1 = np.trapz(M*c1*M**(-alpha1), M)

    M = np.linspace(Mc,mu,10000)  
    Mean_M2 = np.trapz(M*c2*M**(-alpha2), M)
    
    Mean_M.append(Mean_M1+Mean_M2)
  
Mean_M = np.array(Mean_M)

Mean_M*Ncorrs, Mtot

In [ ]:
tab_spatial = Table.from_pandas(pd.read_csv("../data/NGC628_spatial.csv"))

In [ ]:
age_median = []

for i, row in df_m74_bubbles_s2.iterrows():
    a   = row['a']
    b   = row['b']
    ra  = row['ra']
    dec = row['dec']
    ang = row['ang']
    
    df_filt = ellipse(tab_spatial,'RA','DEC',ra,dec,ang, 0,0,a/3600,b/3600)
    df_filt = df_filt[df_filt['Log_Age']<=7.7]

    age_median.append(np.nanpercentile(df_filt['Log_Age'],50))
    
age_median = np.array(age_median)

In [ ]:
tab = Table.read('../photometry/ngc628/f115w_f150w_f200w_photometry.fits')

ra_cen = regions_dict['ngc628']['ra']# 
dec_cen = regions_dict['ngc628']['dec']# 

a = 200
b = 200
ang = 0

filters = {'filt1':'f115w',
           'filt2':'f200w',
           'filt3':'f200w'}

positions = {'ra_col': 'ra',
             'dec_col' : 'dec',
             'ra_cen': ra_cen,
             'dec_cen': dec_cen}

region = {'a1':0,
          'b1':0,
          'a2':a,
          'b2':b,
          'ang':ang,
          'spatial_filter': 'ellipse'}

extinction = {'Av'  : 0.19,
              'Av_x': 4,
              'Av_y': 25,
              'Av_' : 3}

axis_limits= {'xlims': [-1, 5.2], 
              'ylims': [17, 28]}

isochrone_params={'met': [0.02],
                  'label_min': 0,
                  'label_max': 10,
                  'ages': [6.7, 7, 7.7, 8,]}

error_settings = {'ref_xpos': -0.5,
                  'mag_err_lim':0.2}

plot_settings = {'s':3, 'legend.ncols':2, 'alpha':1}

base_cmap = plt.cm.cubehelix_r

# Modify it to start at white
colors = base_cmap(np.linspace(0, 1, 256))
colors[0] = [1, 1, 1, 1]  # Set the lowest color to white (RGBA)

# Create new colormap
new_cmap = LinearSegmentedColormap.from_list('white_min_cmap', colors)

kde_contours = {'gen_kde':False, 'kde_bin' : 300j, 'cmap': 'jet', 'bw': 0.1}

fig, ax = plt.subplots(figsize=(12,10))

df_temp = df_cmd_jwst.copy()
df_temp = df_temp[((df_temp['logAge']==6.7) & (df_temp['label']<5)) | (df_temp['logAge']>6.7)]
fig,ax, tab1 = gen_CMD(tab, 
                      df_temp,
                      filters, 
                      positions,
                      region,
                      extinction,
                      29.81,
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      kde_contours=kde_contours,
                      other_settings={'ab_dist': False, 'skip_data': False},
                      fig=fig, ax=ax)

isochrone_params['ages'] = [9.0, 10.0,]
isochrone_params['met']  = [0.002]

fig,ax, tab1 = gen_CMD(tab, 
                      df_cmd_jwst,
                      filters, 
                      positions,
                      region,
                      extinction,
                      29.81,
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      other_settings={'ab_dist': True, 'skip_data': True},
                      fig=fig, ax=ax)
#fig.savefig("NGC628_cmd.png", bbox_inches='tight')



ax.plot(x,y,'--r', lw=3)

In [ ]:
tab = Table.read('../photometry/ngc628/f115w_f150w_f200w_photometry.fits')
tab1 = tab.copy()
tab1['col'] = tab1['mag_vega_F115W'] -  tab1['mag_vega_F200W']
m = -8.5
y0 = 20
x0 = 1.4
y = m*(tab1['col']-x0) + y0
cond1 = (tab1['mag_vega_F200W']<=20.5) & (tab1['col']<=2)
cond2 = (tab1['mag_vega_F200W']<=y) & (tab1['mag_vega_F200W']<=25)
tab_young = tab1[cond1 | cond2]

tab1[cond1].write("../data/catalogs/ngc628/RSGs.fits", overwrite=True)
tab_young.write("../data/catalogs/ngc628/young.fits", overwrite=True)

tab = tab_young

ra_cen = regions_dict['ngc628']['ra']# 
dec_cen = regions_dict['ngc628']['dec']# 

a = 2000
b = 2000
ang = 0

filters = {'filt1':'f115w',
           'filt2':'f200w',
           'filt3':'f200w'}

positions = {'ra_col': 'ra',
             'dec_col' : 'dec',
             'ra_cen': ra_cen,
             'dec_cen': dec_cen}

region = {'a1':0,
          'b1':0,
          'a2':a,
          'b2':b,
          'ang':ang,
          'spatial_filter': 'ellipse'}

extinction = {'Av'  : 0.19,
              'Av_x': 4,
              'Av_y': 25,
              'Av_' : 3}

axis_limits= {'xlims': [-1, 5.2], 
              'ylims': [17, 28]}

isochrone_params={'met': [0.02],
                  'label_min': 0,
                  'label_max': 10,
                  'ages': [6.7, 7, 7.7, 8,]}

error_settings = {'ref_xpos': -0.5,
                  'mag_err_lim':0.2}

plot_settings = {'s':3, 'legend.ncols':2, 'alpha':1}

base_cmap = plt.cm.cubehelix_r

# Modify it to start at white
colors = base_cmap(np.linspace(0, 1, 256))
colors[0] = [1, 1, 1, 1]  # Set the lowest color to white (RGBA)

# Create new colormap
new_cmap = LinearSegmentedColormap.from_list('white_min_cmap', colors)

kde_contours = {'gen_kde':False, 'kde_bin' : 300j, 'cmap': 'jet', 'bw': 0.1}

fig, ax = plt.subplots(figsize=(12,10))

df_temp = df_cmd_jwst.copy()
df_temp = df_temp[((df_temp['logAge']==6.7) & (df_temp['label']<5)) | (df_temp['logAge']>6.7)]
fig,ax, tab1 = gen_CMD(tab, 
                      df_temp,
                      filters, 
                      positions,
                      region,
                      extinction,
                      29.81,
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      kde_contours=kde_contours,
                      other_settings={'ab_dist': False, 'skip_data': False},
                      fig=fig, ax=ax)

isochrone_params['ages'] = [9.0, 10.0,]
isochrone_params['met']  = [0.002]

fig,ax, tab1 = gen_CMD(tab, 
                      df_cmd_jwst,
                      filters, 
                      positions,
                      region,
                      extinction,
                      29.81,
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      other_settings={'ab_dist': True, 'skip_data': True},
                      fig=fig, ax=ax)
#fig.savefig("NGC628_cmd.png", bbox_inches='tight')

In [ ]:
x = tab_young['ra']
y = tab_young['dec']

plt.scatter(x,y,s=0.1)

In [ ]:
tab = Table.read('../photometry/ngc628/f115w_f150w_f200w_photometry.fits')

ind = []
n_star_RSG = []
n_star_young = []
n_star_total = []
for i in range(len(df_m74_bubbles)):
    ra_cen  = df_m74_bubbles['ra'][i] # 
    dec_cen = df_m74_bubbles['dec'][i] # regions_dict['bkg3']['dec']# 
    a       = df_m74_bubbles['a'][i]
    b       = df_m74_bubbles['b'][i]
    ang     = df_m74_bubbles['ang'][i]
    r       = np.sqrt(a*b)*44

    filters = {'filt1':'f115w',
               'filt2':'f200w',
               'filt3':'f200w'}
    
    positions = {'ra_col': 'ra',
                 'dec_col' : 'dec',
                 'ra_cen': ra_cen,
                 'dec_cen': dec_cen}
    
    region = {'a1':0,
              'b1':0,
              'a2':a,
              'b2':b,
              'ang':ang,
              'spatial_filter': 'ellipse'}
    
    extinction = {'Av'  : 0.19,
                  'Av_x': 2,
                  'Av_y': 25,
                  'Av_' : 2}
    
    axis_limits= {'xlims': [-0.5, 3], 
                  'ylims': [18, 27]}
    
    isochrone_params={'met': [0.02],
                      'label_min': 0,
                      'label_max': 10,
                      'ages': [6.7, 7, 8],
                      'mask_frac' :5}
    
    error_settings = {'ref_xpos': -0.2,
                      'mag_err_lim':0.2}
    
    plot_settings = {'s':6, 'legend.ncols':2, 'alpha':0.6}

    fig, ax = plt.subplots(figsize=(12,10))
    df_temp = df_cmd_jwst.copy()
    df_temp = df_temp[((df_temp['logAge']==6.7) & (df_temp['label']<5)) | (df_temp['logAge']>6.7)]
    try:
        fig,ax, tab1 = gen_CMD(tab, 
                              df_temp,
                              filters, 
                              positions,
                              region,
                              extinction,
                              29.81,
                              axis_limits,
                              isochrone_params,
                              plot_settings=plot_settings,
                              error_settings=error_settings,
                              other_settings={'ab_dist': False, 'skip_data': False},
                              fig=fig, ax=ax)

        tab1['col'] = tab1['mag_vega_F115W'] -  tab1['mag_vega_F200W']
        m = -8.5
        y0 = 20
        x0 = 1.4
        y = m*(tab1['col']-x0) + y0
        cond1 = (tab1['mag_vega_F200W']<=20.5) & (tab1['col']<=2)
        cond2 = (tab1['mag_vega_F200W']<=y) & (tab1['mag_vega_F200W']<=25)

        n_star_RSG.append(len(tab1[cond1]))
        n_star_young.append(len(tab1[cond1|cond2]))
        n_star_total.append(len(tab1))
    except:

        n_star_RSG.append(0)
        n_star_young.append(0)
        n_star_total.append(0)
        continue
    isochrone_params['ages'] = [9.0, 10.0,]
    isochrone_params['met'] = [0.002]
    
    fig,ax, tab1 = gen_CMD(tab, 
                          df_cmd_jwst,
                          filters, 
                          positions,
                          region,
                          extinction,
                          29.81,
                          axis_limits,
                          isochrone_params,
                          plot_settings=plot_settings,
                          error_settings=error_settings,
                          other_settings={'ab_dist': False, 'skip_data': True},
                          fig=fig, ax=ax)
    
    tab1['col'] = tab1['mag_vega_F115W'] -  tab1['mag_vega_F200W']
    
    fig.savefig(f'figures/stars/Bubble_{i}.png', bbox_inches='tight')
    plt.close(fig)

df_m74_bubbles['nstar_RSG']   = n_star_RSG
df_m74_bubbles['nstar_young'] = n_star_young
df_m74_bubbles['nstar_total'] = n_star_total

# **Bubble Sample**

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(12,6), sharey=True, width_ratios=[1, 0.25])

x = df_m74_bubbles_s1['gal_r']*44e-3
y = np.sqrt(df_m74_bubbles_s1['a']*df_m74_bubbles_s1['b'])*44
c = df_m74_bubbles_s1['b']/df_m74_bubbles_s0['a']

ax = axs[0]
ax.scatter(x,y, s=50, color='black', edgecolor='black')

x = df_m74_bubbles_s0['gal_r']*44e-3
y = np.sqrt(df_m74_bubbles_s0['a']*df_m74_bubbles_s0['b'])*44
bins  = [0]
bin_x = 1
rs    = []
while bins[-1]<=1.1*x.max() and bin_x<= 1.1*x.max():
    ind = (x>bins[-1]) & (x<=bin_x)
    if len(x[ind])>=5:
        rs.append(np.percentile(y[ind], 50))
        bins.append(bin_x)
    else:
        bin_x+=1
bins    = np.array(bins)        
bin_cen = 0.5*(bins[1:] + bins[:-1])

ax.plot(bin_cen, rs, color='black', linestyle='dashed', lw=3)
ax.set_yscale('log')

#ax.set_ylim(1,1e3)
#ax.set_yticks([1,10,100,1e3])

ax.set_xlabel('Galactocentric distance [kpc]')
ax.set_ylabel(r'R$_{\rm Bubble}$ [pc]')

ax = axs[1]

bins = np.logspace(1, 3, 10)
ax.hist(y, bins=bins, orientation='horizontal', color='black', linewidth=2, histtype='step')
#ax.set_xticks([0,50,100])

for ax in axs:
    ax.xaxis.set_major_locator(AutoLocator())
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    
    ax.yaxis.set_major_locator(LogLocator())
    #ax.yaxis.set_minor_locator(AutoMinorLocator())
    
    ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True, width=3)
    ax.tick_params(which='minor', length=8, width=2)

plt.subplots_adjust(wspace=0)

x = df_m74_bubbles_s2['gal_r']*44e-3
y = np.sqrt(df_m74_bubbles_s2['a']*df_m74_bubbles_s2['b'])*44

ax = axs[0]
ax.scatter(x,y, s=60,  color=None,  facecolor=(0, 0, 0, 0), edgecolor='black' )

ax = axs[1]

y = np.sqrt(df_m74_bubbles_s1['a']*df_m74_bubbles_s1['b'])*44
bins = np.logspace(1, 3, 10)
ax.hist(y, bins=bins, color='black', alpha=0.6,orientation='horizontal', log=False);

In [ ]:
df_m74_bubbles_s1['n50'].min()

In [ ]:
len(df_m74_bubbles_s0)

In [ ]:
df_m74_bubbles_s0[(df_m74_bubbles_s0['nstar_RSG']<=4) & (df_m74_bubbles_s0['n50']>=10)]

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))

x = np.sqrt(df_m74_bubbles_s0['a']*df_m74_bubbles_s0['b'])*44 
area = np.pi*(df_m74_bubbles_s0['a']*df_m74_bubbles_s0['b'])*44*44

y = df_m74_bubbles_s0['n50']

ax.scatter(x,y,s= 50, edgecolor='black')

ax.xaxis.set_major_locator(LogLocator())
#ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(LogLocator())
#ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=2)

ax.set_xlabel(r'R$_{\rm Bubble}$ [pc]')
ax.set_xlim(20,1e3)
ax.set_ylim(0.8,50)

ax.set_ylabel(r'N$_{\rm SG}$')
ax.set_yscale('log');
ax.set_xscale('log');

In [ ]:
df_m74_bubbles_s1['t25'] = t25
df_m74_bubbles_s1['t50'] = t50
df_m74_bubbles_s1['t75'] = t75

In [ ]:
for i, row in df_m74_bubbles_s1.sort_values('r',ascending=False)[:10].reset_index().iterrows():
    dist = np.round(row['gal_r']*44e-3,1)
    
    print(f"""{i} & {row['ra']} & {row['dec']} &  {np.round(row['a'],2)} & {np.round(row['b'],2)} & {np.round(row['ang'],2)} & {dist} \\\\""")

In [ ]:
y = np.sqrt(df_m74_bubbles_s0['a']*df_m74_bubbles_s0['b'])*44

In [ ]:
df_m74_bubbles_s1

In [ ]:
hdul = fits.open('../data/ngc628/jw01783-o908_t016_miri_f770w_i2d.fits')
f770w = hdul[1].data
f770w_wcs = WCS(hdul[1].header)

tab_RSG = Table.read("../data/catalogs/ngc628/RSGs.fits")

In [ ]:
annot_coords = { 14 : [0,-1/3600],
                 15 : [0,-1/3600],
                 17 : [-0.25/3600,-0.5/3600],
                 18 : [0,-0.5/3600],
                16 : [         0,-1/3600],
                21 : [         0, -0.5/3600],
                47 : [   -4/3600,  0],
                12 : [         0, -0.5/3600],
                23 : [   -1/3600,   0],
                31 : [ -2/3600  , 0  ],
                34 : [ -2/3600  , 0.5/3600   ],
                52 : [ -2.5/3600, 0],
                27 : [ -3.5/3600, 0.5/3600 ],
                69 : [0.75/3600,-1/3600],
                73 : [    1/3600,-2.5/3600],
                82 : [0.8/3600,-2.5/3600],
                91 : [-1.5/3600,0],
                96 : [-2.5/3600,0]}

In [ ]:
fig = plt.figure(figsize=(30,20))
ax = fig.add_subplot(projection=f770w_wcs)
norm = simple_norm(f770w, 'log', vmin=-0.1, vmax=11, log_a=10)

ax.imshow(f770w, cmap='Greys', norm=norm, alpha=0.5)

tab_MS = tab_spatial[tab_spatial['Log_Age']<=7.7]
#x,y = tab_RSG['ra'].value.astype(np.float64), tab_RSG['dec'].value.astype(np.float64)
x,y = tab_MS['RA'].value.astype(np.float64), tab_MS['DEC'].value.astype(np.float64)
#img = ax.scatter(x,y,s=20, color='red', transform=ax.get_transform('icrs'), alpha=0.8)

ax.set_xlim(200,3610)
ax.set_ylim(-350,1410)

ax.set_xlabel(r'${\rm Dec \,(J2000)}$', fontsize=30)
ax.set_ylabel(r'${\rm RA \,(J2000)}$', fontsize=30)

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

for key, data in df_m74_bubbles_s00.iterrows():

    radius = np.sqrt(data['a']*data['b'])
    coord = SkyCoord(ra = data['ra']* u.deg, dec = data['dec']* u.deg)
    ellip = EllipseSkyRegion(coord, 2*data['a']*u.arcsec, 2*data['b']*u.arcsec, data['ang']*u.deg).to_pixel(f770w_wcs)

    if data['color']=='magenta' or data['color']=='red':
        r = ellip.as_artist(color='cyan', lw=3)
        ax.add_patch(r)
    else:
        r = ellip.as_artist(color='yellow', lw=3)
    annot_ra  = data['ra']
    annot_dec = data['dec'] - 1/3600
    if data['Bubble ID'] in annot_coords.keys():
        annot_ra  += annot_coords[data['Bubble ID']][0]
        annot_dec += annot_coords[data['Bubble ID']][1]
    if data['Bubble ID']>21:
        annot_ra  += data['a']/3600
        annot_dec += 1/3600 + data['b']/3600
        
    ax.annotate(str(data['Bubble ID']), (annot_ra, annot_dec), fontsize=15, color='red', zorder=401,
                xycoords =ax.get_transform('icrs'))
    
add_compass(ax,3250,-300, length=0.004, color='black')

for i in range(90):
    region = regions_dict[f'reg_{i}']
    ra  = region['ra']
    dec = region['dec']
    r = plt.Rectangle((ra-12/3600,dec-12/3600), 24/3600,24/3600, rotation_point='center', angle=360-245,
                     zorder=300, fill=False, edgecolor='grey', lw=1, transform=ax.get_transform('icrs'))
    ax.add_patch(r)

    #if i not in [0,1,2,5,11,17,23,29,35,54,60,66,72,78,84]:
    ax.annotate(str(i), (ra, dec), fontsize=15, color='blue', zorder=401,
                xycoords =ax.get_transform('icrs'))
        
#fig.savefig("f770w_bubbles_grids.eps", bbox_inches='tight');
fig.savefig("f770w_bubbles_grids.png", bbox_inches='tight');

In [ ]:
region

In [ ]:
df_m74_bubbles_s1[df_m74_bubbles_s1['Bubble ID']==0]

In [ ]:
dm = 10*10**(29.81/5)
scale = dm*np.tan(np.radians(1/3600))

In [ ]:
scale

In [ ]:
np.sqrt(18.53*18.53*0.98)*scale

In [ ]:
dats = []
for i, row in df_m74_bubbles_s0.sort_values('r',ascending=False).reset_index().iterrows():
    dist = np.round(row['gal_r']*44e-3,2)
    coord = SkyCoord(ra = row['ra'], dec=row['dec'], unit='deg').to_string('hmsdms')

    ra, dec = coord.split()
    ra  = ra.split('.')[0]+ '.' + ra.split('.')[1][:2] +'s'
    dec = dec.split('.')[0]+ '.' + dec.split('.')[1][:2] +'s'

    e =row['b']/row['a']
    r = np.sqrt(row['a']*row['b'])*scale
    pa = row['ang'] + 90

    if pa > 360:
        pa-=360
         
    if pa<-90:
        pa +=180
    if pa>90:
        pa -=180

    dats.append([i, ra, dec, row['a'], e, pa, r, dist])
    print(f"""{i} & {ra} & {dec} &  {row['a']:0.2f} & {e:0.2f} & {pa:0.0f} & {r:0.0f} & {dist} \\\\""")

In [ ]:
pd.DataFrame(dats, columns = ['Bubble ID', 'ra', 'dec', 'a', 'e', 'PA', 'R_B', 'd_GC']).to_csv(
    "../data/DS9 regions/m74_bubbles_s0.dat",
    sep=" ",
    index=False
)

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))

x = np.sqrt(df_m74_bubbles_s2['a']*df_m74_bubbles_s2['b'])*44
bins = np.arange(x.min(), 500, 30)

ax.hist(x, bins=bins, edgecolor='black', linewidth=2, alpha=0.7)

x = np.sqrt(df_m74_bubbles_s1['a']*df_m74_bubbles_s1['b'])*44
ax.hist(x, bins=bins, edgecolor='black', linewidth=2, alpha=0.7)

ax.set_xlabel(r'${\rm Circularized\, Bubble\, Radius\, [pc]}$')
ax.set_ylabel(r'${\rm No.\, of\, Bubbles}$')

# Set log scales
ax.set_yscale('log')
ax.set_xscale('log')

# Force specific major ticks
ax.set_xticks([10, 100, 500])
ax.get_xaxis().set_major_formatter(ScalarFormatter())

# Minor ticks (optional)
ax.minorticks_on()  # disables automatic minor ticks for clarity

# Styling
ax.tick_params(which='both', width=3, direction="in", top=True, right=True,
               bottom=True, left=True)
ax.tick_params(which='major', length=15, direction="in")
ax.tick_params(which='minor', length=7, color='black', direction="in")
#ax.set_xlim(0,500)
plt.show()

# **Completeness**

In [ ]:
f115w_m50 = np.load("../SFH/ngc628/F115W_m50.npy")
f150w_m50 = np.load("../SFH/ngc628/F150W_m50.npy")
f200w_m50 = np.load("../SFH/ngc628/F200W_m50.npy")

f115w_a   = np.load("../SFH/ngc628/F115W_a.npy")
f150w_a   = np.load("../SFH/ngc628/F150W_a.npy")
f200w_a   = np.load("../SFH/ngc628/F200W_a.npy")

In [ ]:
x, y = np.mgrid[0:15,0:6]/1.0
x -= 7
y -= 2.5
x *= 24
y *= 24

r = np.sqrt(x**2 + y**2)

In [ ]:
x

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))

init = models.Polynomial1D(2)
fit = fitting.LevMarLSQFitter()

ax.scatter(r.ravel(), f115w_m50.ravel(), color='blue')

model = fit(init, r.ravel(),f115w_m50.ravel())
ax.plot(r.ravel(), model(r.ravel()), color='blue')

ax.scatter(r.ravel(), f150w_m50.ravel(), color='green')

model = fit(init, r.ravel(),f150w_m50.ravel())
ax.plot(r.ravel(), model(r.ravel()), color='green')

ax.scatter(r.ravel(), f200w_m50.ravel(), color='red')

model = fit(init, r.ravel(),f200w_m50.ravel())

ax.plot(r.ravel(), model(r.ravel()), color='red')

ax.set_ylabel(r'$m_{50}$')
ax.set_xlabel(r'R (arcsec)')

ax.invert_yaxis()

ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=3)

In [ ]:
wcs        = WCS(fits.open("../data/ngc628/F200W_i2d.fits")[1].header)

In [ ]:
df_m74_bubble_sub = pd.read_csv('../data/DS9 regions/m74_bubbles_s1.csv', index_col=0)

In [ ]:
df_m74_bubble_sub

In [ ]:
text = """# Region file format: DS9 version 4.1
global color=green dashlist=8 3 width=1 font="helvetica 10 normal roman" select=1 highlite=1 dash=0 fixed=0 edit=1 move=1 delete=1 include=1 source=1
icrs
"""

with open("../data/DS9 regions/bubble_m74_s2.reg", "w") as f:
    f.write(text)
    for i, row in df_m74_bubbles_s2.sort_values('r',ascending=False).reset_index().iterrows():
        f.write(
            f"""ellipse({row['ra']}, {row['dec']}, {row['a']}", {row['b']}", {row['ang']:.2f}) """
            f"# color=magenta width=3 text={{{i+30}}}\n"
        )


In [ ]:
coords = []
for i in range(90):
    coords.append([regions_dict[f'reg_{i}']['ra'], regions_dict[f'reg_{i}']['dec']])
    
pos = np.array(wcs.world_to_pixel_values(coords))
origin = np.array(wcs.world_to_pixel_values([[24.1738983, 15.7836543]]))

bubble_coords = np.array([df_m74_bubble_sub['ra'], df_m74_bubble_sub['dec']]).T
bubble_pos = np.array(wcs.world_to_pixel_values(bubble_coords))

bubble_coords = np.array([df_m74_bubbles['ra'], df_m74_bubbles['dec']]).T
bubble_pos_main = np.array(wcs.world_to_pixel_values(bubble_coords))

In [ ]:
# bubble_pos[-5] = np.array([7644.47334758, 4260.31925533])

In [ ]:
plt.scatter(pos[:,0], pos[:,1])
plt.scatter(bubble_pos[:,0], bubble_pos[:,1])

In [ ]:
f200w_m50_bubbles = griddata(pos, f200w_m50.ravel(), bubble_pos, method="cubic")
f150w_m50_bubbles = griddata(pos, f150w_m50.ravel(), bubble_pos, method="cubic")
f115w_m50_bubbles = griddata(pos, f115w_m50.ravel(), bubble_pos, method="cubic")

f200w_a_bubbles = griddata(pos, f200w_a.ravel(), bubble_pos, method="cubic")
f150w_a_bubbles = griddata(pos, f150w_a.ravel(), bubble_pos, method="cubic")
f115w_a_bubbles = griddata(pos, f115w_a.ravel(), bubble_pos, method="cubic")

In [ ]:
f115w_a_bubbles

In [ ]:
r = angular_separation(df_m74_bubble_sub['ra'].values*u.deg, df_m74_bubble_sub['dec'].values*u.deg, 
                        regions_dict['ngc628']['ra']*u.deg, regions_dict['ngc628']['dec']*u.deg).to(u.arcsec).value

In [ ]:
len(df_m74_bubble_sub)

In [ ]:
plt.scatter(r, f200w_m50_bubbles)
plt.scatter(r, f150w_m50_bubbles)
plt.scatter(r, f115w_m50_bubbles)

In [ ]:
r = angular_separation(np.array(coords)[:,0]*u.deg, np.array(coords)[:,1]*u.deg, 
                        regions_dict['ngc628']['ra']*u.deg, regions_dict['ngc628']['dec']*u.deg).to(u.arcsec).value

In [ ]:
plt.scatter(r, f200w_m50.ravel())
plt.scatter(r, f150w_m50.ravel())
plt.scatter(r, f115w_m50.ravel())

In [ ]:
plt.scatter(r, f200w_a.ravel())
plt.scatter(r, f150w_a.ravel())
plt.scatter(r, f115w_a.ravel())

In [ ]:
trgb_f200w_mag = np.array([[23.476, 23.477, 23.493, 23.467, 23.513, 23.471],
                           [23.467, 23.605, 23.510, 23.516, 23.453, 23.491],
                           [23.468, 23.666, 23.686, 23.461, 23.439, 23.499],
                           [23.478, 23.443, 23.515, 23.391, 23.466, 23.443],
                           [23.481, 23.502, 23.473, 23.440, 23.452, 23.449],
                           [23.421, 23.460, 23.446, 23.356, 23.308, 23.396],
                           [23.424, 23.408, 23.357, 23.250, 23.353, 23.316],
                           [23.554, 23.411, np.nan, np.nan, 23.281, 23.358],
                           [23.421, 23.335, 23.298, 23.314, 23.442, 23.450],
                           [23.350, 23.455, 23.438, 23.361, 23.449, 23.381],
                           [23.559, 23.445, 23.419, 23.377, 23.363, 23.460],
                           [23.428, 23.475, 23.475, 23.427, 23.477, 23.488],
                           [23.495, 23.425, 23.550, 23.442, 23.443, 23.473],
                           [23.4460, 23.459, 23.526, 23.472, 23.54, 23.425],
                           [23.526, 23.589, 23.541, 23.486, 23.486, 23.540]])

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))
img = ax.imshow(trgb_f200w_mag.T, cmap='jet', extent = (-7.5*24*44e-3,7.5*24*44e-3, -3*24*44e-3,3*24*44e-3))

ax.set_xlabel(r'$\rm x~(kpc)$')
ax.set_ylabel(r'$\rm y~(kpc)$')

cb = plt.colorbar(img, ax=ax, orientation='horizontal', shrink=0.87, pad=0.2)
cb.set_label(r'$m_{\rm TRGB, F200W}$')

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))
img = ax.imshow(f150w_m50.T, cmap='jet', extent = (-7.5*24*44e-3,7.5*24*44e-3, -3*24*44e-3,3*24*44e-3))

ax.set_xlabel(r'$\rm x~(kpc)$')
ax.set_ylabel(r'$\rm y~(kpc)$')

cb = plt.colorbar(img, ax=ax, orientation='horizontal', shrink=0.87, pad=0.2)
cb.set_label(r'$m_{50,\rm F150W}$')

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))
img = ax.imshow(f200w_m50.T, cmap='jet', extent = (-7.5*24*44e-3,7.5*24*44e-3, -3*24*44e-3,3*24*44e-3))

ax.set_xlabel(r'$\rm x~(kpc)$')
ax.set_ylabel(r'$\rm y~(kpc)$')

cb = plt.colorbar(img, ax=ax, orientation='horizontal', shrink=0.87, pad=0.2)
cb.set_label(r'$m_{50,\rm F200W}$')

# **HI**

In [ ]:
hdul = fits.open('../data/NGC628/NGC_628_RO_MOM0_THINGS.fits')
hi_data = hdul[0].data[0,0] 
hi_wcs = WCS(hdul[0].header,naxis=2)

In [ ]:
256+512

In [ ]:
fig = plt.figure(figsize=(30,20))
ax = fig.add_subplot(projection=hi_wcs)
norm = simple_norm(hi_data, 'log', log_a=10)

ax.imshow(hi_data, cmap='gray', norm=norm)

x,y = tab_RSG['ra'].value.astype(np.float64), tab_RSG['dec'].value.astype(np.float64)
#img = ax.scatter(x,y,s=20, color='red', transform=ax.get_transform('icrs'))

ax.set_xlim(256,768)
ax.set_ylim(255,768)

ax.set_xlabel(r'${\rm Dec \,(J2000)}$', fontsize=30)
ax.set_ylabel(r'${\rm RA \,(J2000)}$',  fontsize=30)

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

for key, data in df_m74_bubble_sub.iterrows():
    radius = np.sqrt(data['a']*data['b'])
    coord = SkyCoord(ra = data['ra']* u.deg, dec = data['dec']* u.deg)
    ellip = EllipseSkyRegion(coord, 2*data['a']*u.arcsec, 2*data['b']*u.arcsec, data['ang']*u.deg).to_pixel(hi_wcs)
    
    r = ellip.as_artist(color='cyan', lw=3)
    
    #ax.annotate(str(key), (data['ra'], data['dec']- 1/3600), fontsize=15, color='cyan', zorder=301,
     #           xycoords =ax.get_transform('icrs'))
    ax.add_patch(r)

In [ ]:
x,y = np.where(~np.isnan(hi_data))
hi_values = hi_data[x,y]

ra_hi, dec_hi = hi_wcs.array_index_to_world_values(x,y)
hi_tab = Table([ra_hi,dec_hi, x,y, hi_values.ravel()],names=['ra','dec','x','y','hi_flux'])

In [ ]:
ellipse

In [ ]:
bubble_gas_density = []
for i, data in df_m74_bubble_sub.iterrows():
    ra_cen = data['ra']
    dec_cen = data['dec']
    a = data['a']
    b = data['b']
    ang = data['ang']
    area = np.pi*a*b*4*(44e-3)**2
    
    df_filt = ellipse(hi_tab, 'ra','dec', ra_cen, dec_cen, ang, a/3600,b/3600, 2*a/3600,2*b/3600)
    flux = np.sum(df_filt['hi_flux'])
    bubble_gas_density.append(flux/area)

# **MUSE**

In [ ]:
hdul = fits.open('../data/ngc628/NGC0628_MAPS_native.fits')  # Open the FITS file for reading

muse_wcs = WCS(hdul[6].header)

ha = hdul[30].data
ha_err = hdul[31].data

hb = hdul[6].data
hb_err = hdul[7].data

nii = hdul[36].data
nii_err = hdul[37].data

oiii = hdul[18].data
oiii_err = hdul[19].data

sii16 = hdul[42].data
sii16_err = hdul[43].data

sii30 = hdul[48].data
sii30_err = hdul[49].data
sii = sii16 +  sii30

sii_err = np.sqrt(sii16_err**2 + sii30_err**2)

st_vel = hdul[2].data
st_sigma = hdul[4].data

ha_vel = hdul[32].data
ha_vel_err = hdul[33].data

ha_sigma = hdul[34].data

hdul.close()

In [ ]:
ha_EW = fits.open("../data/ngc628/NGC628_ha_EW.fits")[0].data

In [ ]:
ha_EW.shape

In [ ]:
np.nanmin(ha_err), np.nanmean(ha_err), np.nanmax(ha_err)

In [ ]:
ha_mask1 = ha<=3*ha_err 
ha_mask2 = np.isnan(ha)
ha_mask3 = np.isnan(ha_err)
ha_mask  = ~(ha_mask1 | ha_mask2 | ha_mask3)


hb_mask1 = hb<=3*hb_err 
hb_mask2 = np.isnan(hb)
hb_mask3 = np.isnan(hb_err)
hb_mask  = ~(hb_mask1 | hb_mask2 | hb_mask3)

nii_mask1 = nii<=3*nii_err 
nii_mask2 = np.isnan(nii)
nii_mask3 = np.isnan(nii_err)
nii_mask  = ~(nii_mask1 | nii_mask2 | nii_mask3)

oiii_mask1 = oiii<=3*oiii_err 
oiii_mask2 = np.isnan(oiii)
oiii_mask3 = np.isnan(oiii_err)
oiii_mask  = ~(oiii_mask1 | oiii_mask2 | oiii_mask3)

sii_mask1 = sii<=3*sii_err 
sii_mask2 = np.isnan(sii)
sii_mask3 = np.isnan(sii_err)
sii_mask  = ~(sii_mask1 | sii_mask2 | sii_mask3)

In [ ]:
n2 = nii/ha

n2_mask = ~(~nii_mask | ~ha_mask)

n2_mask = np.where(n2_mask, 1,np.nan)

oh = 8.9 + 0.57*np.log10(n2)*n2_mask

oh = np.where(np.isinf(abs(oh)),np.nan,oh)

In [ ]:
fig = plt.figure(figsize=(13,10))

ax = fig.add_subplot(projection=muse_wcs)
norm = simple_norm(n2_mask, 'log',vmax=1)
img = ax.imshow(n2_mask, cmap='jet', interpolation='nearest')

cb = plt.colorbar(img, ax=ax)

In [ ]:
fig = plt.figure(figsize=(13,10))

ax = fig.add_subplot( projection=muse_wcs)
norm = simple_norm(oh*n2_mask, 'sqrt',vmin=8.5, vmax = 8.9)
img = ax.imshow(oh*n2_mask, norm=norm, cmap='jet', interpolation='nearest')

cb = plt.colorbar(img, ax=ax)
cb.set_label(r'12 + log(OH)')

In [ ]:
x,y = np.where(~np.isnan(oh*n2_mask))
oh_gas_values = oh[x,y]

z_gas_values = 10**(oh_gas_values.ravel()-8.69)*0.017
ra_muse, dec_muse = muse_wcs.array_index_to_world_values(x,y)
met_muse_tab = Table([ra_muse,dec_muse, x,y, z_gas_values],names=['ra','dec','x','y','Z_gas'])

In [ ]:
x_ = np.log10(nii/ha)
y_ = np.log10(oiii/hb)
fig, ax  = plt.subplots(figsize=(10,10))
ax.scatter(x_, y_, s=0.001, color='blue')

x = np.linspace(-2,0)
y = 0.61/(x-0.05) + 1.3

mask1 = y_ > 0.61/(x_-0.05) + 1.3
mask2 = y_ > 0.61/(x_-0.47) + 1.19
mask3 = x_ > 0

ha_EW_mask = (ha_EW<50) | np.isnan( ha_EW)
mask = ~(mask1 | mask2 | mask3 | ~nii_mask | ~ha_mask | ~oiii_mask | ~hb_mask | ha_EW_mask)

sf_mask1 = np.where(mask,1, np.nan )
x_ *= sf_mask1
y_ *= sf_mask1

ax.scatter(x_, y_, s=0.01, color='cyan')

ax.plot(x,y,'--k')

x = np.linspace(-2,0.25)
y = 0.61/(x-0.47) + 1.19

ax.plot(x,y,'-r')

ax.set_xlim(-2, 1)
ax.set_ylim(-1.2,1.5)

ax.set_xlabel(r'$LOG([NII]/H\alpha)$')
ax.set_ylabel(r'$LOG([OIII]/H\beta)$')

ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=3)

In [ ]:
x_ = np.log10(oiii/hb)
y_ = np.log10(sii/ha)
fig, ax  = plt.subplots(figsize=(10,10))

ax.scatter(x_, y_, s=0.001, color='blue')

mask1 = y_ > 0.72/(x_-0.32) + 1.3
mask2 = x_ > 0


mask = ~(mask1 | mask2 | ~oiii_mask | ~hb_mask | ~sii_mask | ~ha_mask | ha_EW_mask)

sf_mask2 = np.where(mask,1, np.nan )
x_ *= sf_mask2
y_ *= sf_mask2

ax.scatter(x_, y_, s=0.001, color='cyan')
           
x = np.linspace(-2,0.2)
y = 0.72/(x-0.32) + 1.3
ax.plot(x,y,'--k')

x = np.linspace(-0.315,0.4)
y = 1.89*x + 0.76

ax.plot(x,y,'-r')

ax.set_xlim(-1, 1)
ax.set_ylim(-1.2,1.5)

ax.set_xlabel(r'$LOG([SII]/H\alpha)$')
ax.set_ylabel(r'$LOG([OIII]/H\beta)$')

ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=3)

In [ ]:
sf_mask = sf_mask1* sf_mask2

In [ ]:
eb_v = 0.934*np.log10((ha/hb)/2.86)*ha_mask*hb_mask
kha  = 2.468
eb_v = np.where(np.isinf(eb_v) | (eb_v<0), np.nan,eb_v)
Av   = 3.1*eb_v

Fha = ha*1e-20
Fha_corr = Fha*10**(kha*eb_v)

Dl = (9.25*u.Mpc).to(u.cm).value

Lha = (Fha_corr*4*np.pi*Dl*Dl)

SFR =  10**(-41.27)*Lha # Kennicutt et.al. (2012)

Av_nan_mask = np.where(ha_mask*hb_mask, 1, np.nan)

In [ ]:
fig = plt.figure(figsize=(13,10))

ax = fig.add_subplot( projection=muse_wcs)
norm = simple_norm(SFR*sf_mask, 'log', vmin=1e-7, vmax=1e-4)
img = ax.imshow(SFR*sf_mask, norm=norm, cmap='jet', interpolation='nearest')
cb = plt.colorbar(img, ax=ax)
cb.set_label(r'SFR [$M_{\odot}.yr^{-1}$]')

for key, data in df_m74_bubbles_s1.iterrows():
    radius = np.sqrt(data['a']*data['b'])
    coord = SkyCoord(ra = data['ra']* u.deg, dec = data['dec']* u.deg)
    ellip = EllipseSkyRegion(coord, 2*data['a']*u.arcsec, 2*data['b']*u.arcsec, data['ang']*u.deg).to_pixel(muse_wcs)
    
    r = ellip.as_artist(color='black', lw=2)
    
    #ax.annotate(str(key), (data['ra'], data['dec']- 1/3600), fontsize=15, color='cyan', zorder=301,
     #           xycoords =ax.get_transform('icrs'))
    ax.add_patch(r)

In [ ]:
fig = plt.figure(figsize=(13,10))

ax = fig.add_subplot( projection=muse_wcs)
norm = simple_norm(Av*Av_nan_mask, 'sqrt')
img = ax.imshow(Av*Av_nan_mask, norm=norm, cmap='jet', interpolation='nearest')
cb = plt.colorbar(img, ax=ax)
cb.set_label(r'Av')

for key, data in df_m74_bubbles_s1.iterrows():
    radius = np.sqrt(data['a']*data['b'])
    coord = SkyCoord(ra = data['ra']* u.deg, dec = data['dec']* u.deg)
    ellip = EllipseSkyRegion(coord, 2*data['a']*u.arcsec, 2*data['b']*u.arcsec, data['ang']*u.deg).to_pixel(muse_wcs)
    
    r = ellip.as_artist(color='black', lw=2)
    
    #ax.annotate(str(key), (data['ra'], data['dec']- 1/3600), fontsize=15, color='cyan', zorder=301,
     #           xycoords =ax.get_transform('icrs'))
    ax.add_patch(r)

In [ ]:
plt.hist(Av.ravel(), log=True)

In [ ]:
x,y = np.where(~np.isnan(Av*Av_nan_mask))
Av_values = Av[x,y]

ra_muse, dec_muse = muse_wcs.array_index_to_world_values(x,y)
av_muse_tab = Table([ra_muse,dec_muse, x,y, Av_values],names=['ra','dec','x','y','Av'])

In [ ]:
av_muse_tab.write("../data/ngc_628_Av.fits", overwrite=True)

In [ ]:
x,y = np.where((~np.isnan(SFR*sf_mask)) & (SFR*sf_mask>0 ))
gas_SFR_values = SFR[x,y]

ra_muse, dec_muse = muse_wcs.array_index_to_world_values(x,y)
gas_SFR_tab = Table([ra_muse,dec_muse, x,y, gas_SFR_values],names=['ra','dec','x','y','SFR'])

In [ ]:
k  = 0
nx = 15
ny = 6

Av_map_muse  = np.zeros((nx,ny))*np.nan
met_map_muse = np.zeros((nx,ny))*np.nan
SFR_map_muse = np.zeros((nx,ny))*np.nan

for i in range(15):
    for j in range(6):

        ra_cen, dec_cen = regions_dict[f'reg_{k}']['ra'], regions_dict[f'reg_{k}']['dec']
        
        tab_filt1 = box(av_muse_tab, 'ra','dec',ra_cen, dec_cen, 0,0,24/3600,24/3600,245)
        Av_map_muse[i, j] = np.median(tab_filt1['Av'])
        
        tab_filt2 = box(met_muse_tab, 'ra','dec',ra_cen, dec_cen, 0,0,24/3600,24/3600,245)
        met_map_muse[i, j] = np.median(tab_filt2['Z_gas'])

        tab_filt3 = box(gas_SFR_tab, 'ra','dec',ra_cen, dec_cen, 0,0,24/3600,24/3600,245)
        SFR_map_muse[i, j] = np.nansum(tab_filt3['SFR'])
        
        k+=1

In [ ]:
met_map_muse.shape

In [ ]:
np.save("../data/muse_metallicity.npy",met_map_muse)

In [ ]:
met_map_muse = np.where(np.isnan(met_map_muse),0.014, met_map_muse)

In [ ]:
Av_map_trgb = np.load('../data/Av_map_TRGB.npy')
Av_map_trgb = np.where(np.isnan(Av_map_trgb), 0, Av_map_trgb)

met_map_trgb = np.load('../data/met_map_TRGB.npy')
met_map_trgb = np.where(np.isnan(met_map_trgb),0.003, met_map_trgb)

In [ ]:
Av_map_trgb_bubble       = griddata(pos, Av_map_trgb.ravel(), bubble_pos, method="cubic")
met_map_muse_bubble      = griddata(pos, met_map_muse.ravel(), bubble_pos, method="cubic")
met_map_trgb_bubble      = griddata(pos, met_map_trgb.ravel(), bubble_pos, method="cubic")

met_map_muse_bubble_main = griddata(pos, met_map_muse.ravel(), bubble_pos_main, method="cubic")
met_map_trgb_bubble_main = griddata(pos, met_map_trgb.ravel(), bubble_pos_main, method="cubic")

In [ ]:
SFR_bubble_muse = []
SFR_bubble_muse_shell = []

for i, row in df_m74_bubble_sub.iterrows():
    df_filt = ellipse(gas_SFR_tab,'ra','dec', row['ra'], row['dec'], row['ang'],0,0, row['a']/3600, row['b']/3600)
    SFR_bubble_muse.append(np.nansum(df_filt['SFR']))

    df_filt = ellipse(gas_SFR_tab,'ra','dec', row['ra'], row['dec'], row['ang'],row['a']/3600,row['b']/3600, 1.2*row['a']/3600, 1.2*row['b']/3600)
    SFR_bubble_muse_shell.append(np.nansum(df_filt['SFR']))
    
SFR_bubble_muse = np.array(SFR_bubble_muse)
SFR_bubble_muse_shell = np.array(SFR_bubble_muse_shell)

In [ ]:
SFR_bubble_s2_muse = []
SFR_bubble_muse_s2_shell = []

for i, row in df_m74_bubbles_s2.iterrows():
    df_filt = ellipse(gas_SFR_tab,'ra','dec', row['ra'], row['dec'], row['ang'],0,0, row['a']/3600, row['b']/3600)
    SFR_bubble_s2_muse.append(np.nansum(df_filt['SFR']))

    df_filt = ellipse(gas_SFR_tab,'ra','dec', row['ra'], row['dec'], row['ang'],row['a']/3600,row['b']/3600, 1.2*row['a']/3600, 1.2*row['b']/3600)
    SFR_bubble_muse_s2_shell.append(np.nansum(df_filt['SFR']))
    
SFR_bubble_s2_muse = np.array(SFR_bubble_s2_muse)
SFR_bubble_muse_s2_shell = np.array(SFR_bubble_muse_s2_shell)

In [ ]:
len(SFR_bubble_s2_muse)

In [ ]:
r = angular_separation(df_m74_bubble_sub['ra'].values*u.deg, df_m74_bubble_sub['dec'].values*u.deg, 
                        regions_dict['ngc628']['ra']*u.deg, regions_dict['ngc628']['dec']*u.deg).to(u.arcsec).value

In [ ]:
x, y = np.mgrid[0:15,0:6]/1.0
x -= 7
y -= 2.5
x *= 24
y *= 24

r = np.sqrt(x**2 + y**2).ravel()*44e-3

In [ ]:
fig, axs = plt.subplots(3,1,figsize = (12,15), sharex=True)

ax= axs[0]
ax.scatter(r,met_map_muse.ravel(), s=80 , color='grey', edgecolor='black')

bins  = np.arange(0, r.max()*1.2,1)
med_z = []
for i in range(len(bins)-1):
    ind = (r>bins[i]) & (r<=bins[1+i])
    med_z.append(np.nanmedian(met_map_muse.ravel()[ind]))

bin_cen = 0.5*(bins[1:] + bins[:-1])

ax.plot(bin_cen, med_z, '--', lw=3, color='black')
    
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")

ax.set_ylabel(r'${\rm Z}$ (Gas)')


ax= axs[1]
ax.scatter(r, met_map_trgb.ravel(), s=80, color='grey', edgecolor='black')

med_z = []
for i in range(len(bins)-1):
    ind = (r>bins[i]) & (r<=bins[1+i])
    med_z.append(np.nanmedian(met_map_trgb.ravel()[ind]))

bin_cen = 0.5*(bins[1:] + bins[:-1])

ax.plot(bin_cen, med_z, '--', lw=3, color='black')

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")

ax.set_ylabel(r'${\rm Z}$ (TRGB)')


ax= axs[2]
ax.scatter(r, Av_map_trgb.ravel(), s=80, color='grey', edgecolor='black')

med_z = []
for i in range(len(bins)-1):
    ind = (r>bins[i]) & (r<=bins[1+i])
    med_z.append(np.nanmedian(Av_map_trgb.ravel()[ind]))

bin_cen = 0.5*(bins[1:] + bins[:-1])

ax.plot(bin_cen, med_z, '--', lw=3, color='black')

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")

ax.set_xlabel(r'${\rm Galactocentric\, distance}$ [kpc]')
ax.set_ylabel(r'${\rm A_V}$ (TRGB)')
plt.subplots_adjust(hspace=0)

In [ ]:
fig, ax = plt.subplots(figsize = (12,6),)
x = np.sqrt(df_m74_bubbles_s1['a']*df_m74_bubbles_s1['b'])*44
area = np.pi*df_m74_bubbles_s1['a']*df_m74_bubbles_s1['b']*(44e-3)**2

y = SFR_bubble_muse #SFRD

#ax.scatter(x,y, s=30 , color='red', edgecolor='black')

area = np.pi*df_m74_bubbles_s2['a']*df_m74_bubbles_s2['b']*(44e-3)**2
x = np.sqrt(df_m74_bubbles_s2['a']*df_m74_bubbles_s2['b'])*44

y = SFR_bubble_s2_muse
ax.scatter(x,y, s=30 , color='red', edgecolor='black')

ax.set_xlabel(r'$R_{\mathrm B}$ [pc]')
ax.set_ylabel(r'$\Sigma_{\rm SFR}~[M_{\odot}.yr^{-1}]$')
ax.set_yscale('log')

# **Isochrones**

### **PARSEC+COLIBRI**

In [ ]:
def IMF_Krp( m, ml=0.1, mint=0.5, mu=350.,a1=1.3,a2=2.3):

    h2 = (mu**(1.-a2)-mint**(1.-a2))/(1.-a2)
    h1 = (mint**(1.-a1)-ml**(1.-a1))/(1.-a1)

    c1 = 1./(h1+h2*mint**(a2-a1))
    c2 = c1*mint**(a2-a1)

    c = ones(len(m))
    c[where(m < mint)] = c1
    c[where(m >= mint)] = c2

    a = ones(len(m))
    a[where(m < mint)] = -a1
    a[where(m >= mint)] = -a2
    imf = c*m**a

    return(imf)

In [ ]:
df_cmd_jwst = pd.read_csv("../data/isochrones_master/cmd_jwst.csv")

In [ ]:
df_temp = df_cmd_jwst[((df_cmd_jwst['logAge']>=6.7) & (df_cmd_jwst['logAge']<=8.9) & (df_cmd_jwst['label']<12) & (df_cmd_jwst['Zini']>=0.014) & (df_cmd_jwst['Zini']<=0.02)) |
                       ((df_cmd_jwst['logAge']>8.9) & (df_cmd_jwst['Zini']>=0.001) & (df_cmd_jwst['Zini']<=0.003))         ]

In [ ]:
# Main Sequence
df_temp.loc[(df_temp['Mini'] > 5)  & (df_temp['label']==0)  & (df_temp['logAge']>=6.)  & (df_temp['logAge']<=7.6) , 'label'] = 11
df_temp.loc[(df_temp['Mini'] > 14) & (df_temp['label']==1) & (df_temp['logAge']>=6.)  & (df_temp['logAge']<=7.6)  , 'label'] = 2
df_temp.loc[(df_temp['label']==11), 'label'] = 1 


In [ ]:
# Post Main Sequence 

df_temp.loc[(df_temp['logg'] < 0.8) & (df_temp['label']==2) & (df_temp['logAge']==7.1)  & (df_temp['Zini']==0.02)  , 'label'] = 3
df_temp.loc[(df_temp['Mini'] > 14.892500144697533) & (df_temp['label']==3) & (df_temp['logAge']==7.1)  & (df_temp['Zini']==0.02)  , 'label'] = 4
df_temp.loc[(df_temp['Mini'] > 15.226776359169617) & (df_temp['label']==4) & (df_temp['logAge']==7.1)  & (df_temp['Zini']==0.02)  , 'label'] = 5
df_temp.loc[(df_temp['Mini'] < 15.27) & (df_temp['Mini'] > 15.2) & (df_temp['label']==2) & (df_temp['logAge']==7.1)  & (df_temp['Zini']==0.02)  , 'label'] = 5
df_temp.loc[(df_temp['Mini'] > 15.27) & (df_temp['label']==2)  & (df_temp['logAge']==7.1)  & (df_temp['Zini']==0.02)  , 'label']  = 6
df_temp.loc[(df_temp['Mini'] > 15.27) & (df_temp['label']==5)  & (df_temp['logAge']==7.1)  & (df_temp['Zini']==0.02)  , 'label']  = 6
df_temp.loc[(df_temp['Mini'] > 15.507) & (df_temp['label']==6) & (df_temp['logAge']==7.1)  & (df_temp['Zini']==0.02)  , 'label'] = 7

df_temp.loc[(df_temp['logg'] < 1.) & (df_temp['label']==2) & (df_temp['logAge']==7.1)  & (df_temp['Zini']==0.017)  , 'label'] = 3
df_temp.loc[(df_temp['Mini'] > 15.03) & (df_temp['label']==3) & (df_temp['logAge']==7.1)  & (df_temp['Zini']==0.017)  , 'label'] = 4
df_temp.loc[(df_temp['Mini'] < 15.48) & (df_temp['Mini'] > 15.3)  & (df_temp['logAge']==7.1)  & (df_temp['Zini']==0.017)  , 'label'] = 5
df_temp.loc[(df_temp['Mini'] > 15.48) & (df_temp['label']==2) & (df_temp['logAge']==7.1)  & (df_temp['Zini']==0.017)  , 'label'] = 6

df_temp.loc[(df_temp['Mini'] > 15.5) & (df_temp['Mini'] < 15.8) & (df_temp['label']==4) & (df_temp['logAge']==7.1)  & (df_temp['Zini']==0.017)  , 'label'] = 6
df_temp.loc[(df_temp['Mini'] > 15.8) & (df_temp['logAge']==7.1)  & (df_temp['Zini']==0.017)  , 'label'] = 7

df_temp.loc[(df_temp['logg'] < 1.)    & (df_temp['label']==2)     & (df_temp['logAge']==7.1)         & (df_temp['Zini']==0.014), 'label'] = 3
df_temp.loc[(df_temp['Mini'] > 15.21) & (df_temp['label']==3)     & (df_temp['logAge']==7.1)      & (df_temp['Zini']==0.014), 'label'] = 4
df_temp.loc[(df_temp['Mini'] > 15.48) & (df_temp['Mini'] < 15.65) & (df_temp['logAge']==7.1)  & (df_temp['Zini']==0.014), 'label'] = 5
df_temp.loc[(df_temp['Mini'] > 15.65) & (df_temp['Mini'] < 16.02) & (df_temp['logAge']==7.1)      & (df_temp['Zini']==0.014), 'label'] = 6

df_temp.loc[(df_temp['Mini'] > 16.02) & (df_temp['logAge']==7.1)  & (df_temp['Zini']==0.014), 'label'] = 7

In [ ]:
df_temp.loc[(df_temp['logg'] < 0.2)    & (df_temp['label']==2)     & (df_temp['logAge']==7.)  & (df_temp['Zini']==0.02), 'label'] = 3
df_temp.loc[(df_temp['Mini'] > 17.701) & (df_temp['label']==3)     & (df_temp['logAge']==7.)  & (df_temp['Zini']==0.02), 'label'] = 4
df_temp.loc[(df_temp['Mini'] > 18.0782) & (df_temp['Mini'] < 18.11) & (df_temp['logAge']==7.)  & (df_temp['Zini']==0.02), 'label'] = 5
df_temp.loc[(df_temp['Mini'] > 18.11) & (df_temp['Mini'] < 18.7) & (df_temp['logAge']==7.)  & (df_temp['Zini']==0.02), 'label'] = 6
df_temp.loc[(df_temp['Mini'] > 18.7) & (df_temp['logAge']==7.)  & (df_temp['Zini']==0.02), 'label'] = 7

df_temp.loc[(df_temp['logg'] < 0.2)    & (df_temp['label']==2)     & (df_temp['logAge']==7.)  & (df_temp['Zini']==0.017), 'label'] = 3
df_temp.loc[(df_temp['Mini'] > 17.92) & (df_temp['label']==3)     & (df_temp['logAge']==7.)  & (df_temp['Zini']==0.017), 'label'] = 4
df_temp.loc[(df_temp['Mini'] > 18.2) & (df_temp['Mini'] < 19) & (df_temp['logAge']==7.)  & (df_temp['Zini']==0.017), 'label'] = 6
df_temp.loc[(df_temp['Mini'] > 19) & (df_temp['logAge']==7.)  & (df_temp['Zini']==0.017), 'label'] = 7

df_temp.loc[(df_temp['logg'] < 0.3)    & (df_temp['label']==2)     & (df_temp['logAge']==7.)  & (df_temp['Zini']==0.014), 'label'] = 3
df_temp.loc[(df_temp['Mini'] > 18.18) & (df_temp['Mini'] < 18.4) & (df_temp['label'] == 3) & (df_temp['logAge']==7.)  & (df_temp['Zini']==0.014), 'label'] = 4
df_temp.loc[(df_temp['Mini'] > 18.4) & (df_temp['Mini'] < 18.5) & (df_temp['logAge']==7.)  & (df_temp['Zini']==0.014), 'label'] = 5
df_temp.loc[(df_temp['Mini'] > 18.5) & (df_temp['Mini'] < 19.35) & (df_temp['logAge']==7.)  & (df_temp['Zini']==0.014), 'label'] = 6
df_temp.loc[(df_temp['Mini'] > 19.309) & (df_temp['logAge']==7.)  & (df_temp['Zini']==0.014), 'label'] = 7

In [ ]:
df_temp.loc[(df_temp['logg'] < 0.1)    & (df_temp['label']==2)   & (df_temp['logAge']==6.9)  & (df_temp['Zini']==0.02), 'label'] = 3
df_temp.loc[(df_temp['Mini'] > 21.7) & (df_temp['Mini'] < 23.2) & (df_temp['logAge']==6.9)  & (df_temp['Zini']==0.02) & (df_temp['label'] == 3) , 'label'] = 4
df_temp.loc[(df_temp['Mini'] > 23.2)                           & (df_temp['logAge']==6.9)  & (df_temp['Zini']==0.02), 'label'] = 7

df_temp.loc[(df_temp['logg'] < 0.05)    & (df_temp['label']==2)  & (df_temp['logAge']==6.9)  & (df_temp['Zini']==0.017), 'label'] = 3
df_temp.loc[(df_temp['Mini'] > 22) & (df_temp['Mini'] < 23.4) & (df_temp['logAge']==6.9)  & (df_temp['Zini']==0.017) & (df_temp['label'] == 3) , 'label'] = 4
df_temp.loc[(df_temp['Mini'] > 23.4)                            & (df_temp['logAge']==6.9)  & (df_temp['Zini']==0.017), 'label'] = 7

df_temp.loc[(df_temp['logg'] < 0.05)    & (df_temp['label']==2)  & (df_temp['logAge']==6.9)  & (df_temp['Zini']==0.014), 'label'] = 3
df_temp.loc[(df_temp['Mini'] > 22.38) & (df_temp['Mini'] < 23.8) & (df_temp['logAge']==6.9)  & (df_temp['Zini']==0.014) & (df_temp['label'] == 3) , 'label'] = 4
df_temp.loc[(df_temp['Mini'] > 23.8)                            & (df_temp['logAge']==6.9)  & (df_temp['Zini']==0.014), 'label'] = 7

In [ ]:
df_temp.loc[(df_temp['logg'] < 0.5)    & (df_temp['label']==2)   & (df_temp['logAge']==6.8)  & (df_temp['Zini']==0.02), 'label'] = 3
df_temp.loc[(df_temp['Mini'] > 27.51) & (df_temp['Mini'] < 29.75) & (df_temp['logAge']==6.8)  & (df_temp['Zini']==0.02) & (df_temp['label'] == 3) , 'label'] = 4
df_temp.loc[(df_temp['Mini'] > 29.75)                           & (df_temp['logAge']==6.8)  & (df_temp['Zini']==0.02), 'label'] = 7

df_temp.loc[(df_temp['logg'] < -0.15)    & (df_temp['label']==2)  & (df_temp['logAge']==6.8)  & (df_temp['Zini']==0.017), 'label'] = 3
df_temp.loc[(df_temp['Mini'] > 27.9) & (df_temp['Mini'] < 29.75) & (df_temp['logAge']==6.8)  & (df_temp['Zini']==0.017) & (df_temp['label'] == 3) , 'label'] = 4
df_temp.loc[(df_temp['Mini'] > 29.75)                            & (df_temp['logAge']==6.8)  & (df_temp['Zini']==0.017), 'label'] = 7

df_temp.loc[(df_temp['logg'] < -0.2)    & (df_temp['label']==2)  & (df_temp['logAge']==6.8)  & (df_temp['Zini']==0.014), 'label'] = 3
df_temp.loc[(df_temp['Mini'] > 28.35) & (df_temp['Mini'] < 30.7) & (df_temp['logAge']==6.8)  & (df_temp['Zini']==0.014) & (df_temp['label'] == 3) , 'label'] = 4
df_temp.loc[(df_temp['Mini'] > 30.7)                            & (df_temp['logAge']==6.8)  & (df_temp['Zini']==0.014), 'label'] = 7

In [ ]:
df_temp.loc[(df_temp['logg'] < 0.)    & (df_temp['label']==2)   & (df_temp['logAge']==6.7)  & (df_temp['Zini']==0.02), 'label'] = 3
df_temp.loc[(df_temp['Mini'] > 36.4) & (df_temp['Mini'] < 36.75) & (df_temp['logAge']==6.7)  & (df_temp['Zini']==0.02) , 'label'] = 4
df_temp.loc[(df_temp['Mini'] > 36.73) & (df_temp['Mini'] < 38.88) & (df_temp['logAge']==6.7)  & (df_temp['Zini']==0.02), 'label'] = 5
df_temp.loc[(df_temp['Mini'] > 38.86) & (df_temp['logg'] >= 4.5) & (df_temp['logAge']==6.7)  & (df_temp['Zini']==0.02) , 'label'] = 6
df_temp.loc[(df_temp['Mini'] > 39.9)                           & (df_temp['logAge']==6.7)  & (df_temp['Zini']==0.02), 'label'] = 7

df_temp.loc[(df_temp['logg'] < 0.)    & (df_temp['label']==2)   & (df_temp['logAge']==6.7)  & (df_temp['Zini']==0.017), 'label'] = 3
df_temp.loc[(df_temp['Mini'] > 37) & (df_temp['Mini'] < 38) & (df_temp['logAge']==6.7)  & (df_temp['Zini']==0.017) , 'label'] = 4
df_temp.loc[(df_temp['Mini'] > 38) & (df_temp['Mini'] < 39) & (df_temp['logAge']==6.7)  & (df_temp['Zini']==0.017), 'label'] = 5
df_temp.loc[(df_temp['Mini'] > 39) & (df_temp['logg'] >= 4) & (df_temp['logAge']==6.7)  & (df_temp['Zini']==0.017) , 'label'] = 6
df_temp.loc[(df_temp['Mini'] > 40)                           & (df_temp['logAge']==6.7)  & (df_temp['Zini']==0.017), 'label'] = 7

df_temp.loc[(df_temp['logg'] < 0.)    & (df_temp['label']==2)   & (df_temp['logAge']==6.7)  & (df_temp['Zini']==0.014), 'label'] = 3
df_temp.loc[(df_temp['Mini'] > 38.75) & (df_temp['Mini'] < 40) & (df_temp['logAge']==6.7)  & (df_temp['Zini']==0.014) , 'label'] = 4
df_temp.loc[(df_temp['Mini'] > 40) & (df_temp['Mini'] < 41) & (df_temp['logAge']==6.7)  & (df_temp['Zini']==0.014), 'label'] = 5
df_temp.loc[(df_temp['Mini'] > 40.5) & (df_temp['logg'] > 2.5) & (df_temp['logAge']==6.7)  & (df_temp['Zini']==0.014) , 'label'] = 6
df_temp.loc[(df_temp['Mini'] > 41.58)                           & (df_temp['logAge']==6.7)  & (df_temp['Zini']==0.014), 'label'] = 7


In [ ]:
df_cmd_jwst_n = df_temp

In [ ]:
df_cmd_jwst_n.to_csv("../data/isochrones_master/cmd_jwst_corr.csv", index=None)

In [ ]:
df_ = df_temp[(df_temp['label']>=2)  & (df_temp['label']<9) & (df_temp['logAge']==6.7) & (df_temp['logAge']==6.7) & (df_temp['Zini']==0.014) ]

In [ ]:
np.array(df_['int_IMF'])[-1]

In [ ]:
x = df_['logTe']
y = df_['logL']

c = df_['label']
fig, ax = plt.subplots()

img= ax.scatter(x,y,s=1, c=c, cmap='tab10', vmin=0, vmax=10)

cb = plt.colorbar(img, ax=ax)
cb.set_ticks(np.arange(0.5,9.6,1))

cb.set_ticklabels(['PMS','MS','SGB','RGB','CHEB1','CHEB2','CHEB3', 'EAGB','TP-AGB','Post-AGB'],
                 fontsize=10)
"""
img = ax.scatter(x,y,s=1, c=c, cmap='jet', vmin=2.6)
plt.colorbar(img, ax=ax)
"""

ax.invert_xaxis()
ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=3)

ax.set_xlabel('logTe')
ax.set_ylabel('logL')
plt.show()


In [ ]:
x = df_['Mini']
y = df_['logg']

c = df_['label']
fig, ax = plt.subplots()

img= ax.scatter(x,y,s=1, c=c, cmap='tab10', vmin=0, vmax=10)

cb = plt.colorbar(img, ax=ax)
cb.set_ticks(np.arange(0.5,9.6,1))

cb.set_ticklabels(['PMS','MS','SGB','RGB','CHEB1','CHEB2','CHEB3', 'EAGB','TP-AGB','Post-AGB'],
                 fontsize=10)
"""
img = ax.scatter(x,y,s=1, c=c, cmap='jet', vmin=2.6)
plt.colorbar(img, ax=ax)
"""

ax.invert_xaxis()
ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=3)

ax.set_xlabel('Mini')
ax.set_ylabel('logg')
plt.show()


In [ ]:
x = df_['F115Wmag'] - df_['F200Wmag']
y = df_['F200Wmag']

c = df_['label']
fig, ax = plt.subplots()

img= ax.scatter(x,y,s=1, c=c, cmap='tab10', vmin=0, vmax=10)
cb = plt.colorbar(img, ax=ax)
cb.set_ticks(np.arange(0.5,9.6,1))

cb.set_ticklabels(['PMS','MS','SGB','RGB','CHEB1','CHEB2','CHEB3', 'EAGB','TP-AGB','Post-AGB'],
                 fontsize=10)

ax.invert_yaxis()
ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=3)

ax.set_xlabel('F115W-F200W')
ax.set_ylabel('F200W')
plt.show()


In [ ]:
tabs_interp = []
for met_ in [0.014, 0.017, 0.02, 0.001, 0.002, 0.003]:
    if not os.path.exists(f'../data/isochrones/JWST_{met_}'):
        os.mkdir(f'../data/isochrones/JWST_{met_}/')
    else:
        os.system(f'rm ../data/isochrones/JWST_{met_}/*')
    
    interp = True
    tabs = pd.DataFrame()
    l = 1
    
    ages = [6.7, 6.8    ,  6.9    ,  7.     ,
                            7.1    ,  7.2    ,  7.3  ,  7.4    ,  7.5    ,  7.6    ,
                            7.7    ,  7.8    ,  7.9  ,  8.,   8.1    ,  8.2    ,
                            8.3    ,  8.4    ,  8.5, 8.6, 8.7, 8.8, 8.9, 9,
                            9.1, 9.2, 9.3,  9.4, 9.5, 9.6, 9.7, 9.8, 9.9, 10.]
    #ages = [7.0]
    for i, age in enumerate(ages):  
    
        if len(str(i+1))==1:
            i = '0' + str(i+1)
        else:
            i =  str(i+1)
    
        for met in [met_]:
            temp = df_cmd_jwst_n[np.round(df_cmd_jwst_n['logAge'],1)==age].copy()
            if age == 6.7:
                temp = temp[temp['label']<5]
            temp = temp[(temp['Zini']==met) & (temp['label']<9) ]
            temp = temp.drop_duplicates(['Mini','logAge','label'])  
            
            if len(temp)==0:
                continue
            if interp:
                # Interpolation using F115W-F200W vs F200W
                temps = []
    
                #for n, lb in enumerate(np.unique(temp['label'])):
                if ( age==7.0 and met == 0.017) | ( age==7.1 and met == 0.014):
                    temp_lb = temp[temp['label']<=4].copy()
    
                    x = temp_lb['F115Wmag'].values - temp_lb['F200Wmag'].values 
                    y = temp_lb['F200Wmag'].values
        
                    diff = np.array([0] + list(np.sqrt((x[1:]-x[:-1])**2 + (y[1:]-y[:-1])**2)))
                    dist = np.cumsum(diff)
        
                    temp_lb['dist'] = dist
                    temp_lb = temp_lb.sort_values('dist')
        
                    x = temp_lb['dist']
                    x_new = np.arange(x.min(), x.max(),0.01)

                    #dn = IMF_Krp(0.5*(temp_lb['Mini'].values[1:] + temp_lb['Mini'].values[:-1])) * np.diff(temp_lb['Mini'].values)
                    dn = np.diff(temp_lb['int_IMF'])
                    dn = np.array([dn[0]] + list(dn))
                    temp_lb['phase_prob'] = dn
        
                    label = np.interp(x_new, x, temp_lb['label'])
                    Mini  = np.interp(x_new, x, temp_lb['Mini'])
                    f115w = np.interp(x_new, x, temp_lb['F115Wmag'])
                    f150w = np.interp(x_new, x, temp_lb['F150Wmag'])
                    f200w = np.interp(x_new, x, temp_lb['F200Wmag'])
                    phase_prob = np.interp(x_new, x, temp_lb['phase_prob'])
                    
                    temp_interp = pd.DataFrame(zip(label, Mini, f115w, f150w, f200w, phase_prob), columns = ['label','Mini','F115Wmag', 'F150Wmag', 'F200Wmag', 'phase_prob'])
                    temp_interp['Zini'] = temp['Zini'].max()
                    temp_interp['logAge'] = temp['logAge'].max()
                    temps.append(temp_interp)
                    
                if (age==7.0 and met == 0.017) | ( age==7.1 and met == 0.014):
                    temp_lb = temp[temp['label']>4].copy()
                else:
                    temp_lb = temp[temp['label']<9].copy()
                x = temp_lb['F115Wmag'].values - temp_lb['F200Wmag'].values 
                y = temp_lb['F200Wmag'].values
    
                diff = np.array([0] + list(np.sqrt((x[1:]-x[:-1])**2 + (y[1:]-y[:-1])**2)))
                dist = np.cumsum(diff)
    
                temp_lb['dist'] = dist
                temp_lb = temp_lb.sort_values('dist')
    
                x = temp_lb['dist']
                x_new = np.arange(x.min(), x.max(),0.01)

                #dn = IMF_Krp(0.5*(temp_lb['Mini'].values[1:] + temp_lb['Mini'].values[:-1])) * np.diff(temp_lb['Mini'].values)
                dn = np.diff(temp_lb['int_IMF'])
                dn = np.array([dn[0]] + list(dn))
                temp_lb['phase_prob'] = dn
    
                label = np.interp(x_new, x, temp_lb['label'])
                Mini  = np.interp(x_new, x, temp_lb['Mini'])
                f115w = np.interp(x_new, x, temp_lb['F115Wmag'])
                f150w = np.interp(x_new, x, temp_lb['F150Wmag'])
                f200w = np.interp(x_new, x, temp_lb['F200Wmag'])
                phase_prob = np.interp(x_new, x, temp_lb['phase_prob'])
                    
                temp_interp = pd.DataFrame(zip(label, Mini, f115w, f150w, f200w, phase_prob), columns = ['label','Mini','F115Wmag', 'F150Wmag', 'F200Wmag', 'phase_prob'])
                temp_interp['Zini'] = temp['Zini'].max()
                temp_interp['logAge'] = temp['logAge'].max()
                
                temps.append(temp_interp)
                
            temp = pd.concat(temps)         
            temp = temp[['label', 'Mini', 'F115Wmag', 'F150Wmag', 'F200Wmag', 'Zini', 'logAge','phase_prob']]
           
            header = list(temp.keys())
            header[0] = '#' + header[0]
                
            if (met>0.01 and age<9) or (met<0.004 and age>=9):
                
                if l<10:
                    temp.to_csv(f'../data/isochrones/JWST_{met}/0{l}_PARSEC1.2S_Z{0.02}_logAGE{age}Myr_JWST_JHK.isoc',sep=' ',
                                   index=None, header = header)
                else:
                    temp.to_csv(f'../data/isochrones/JWST_{met}/{l}_PARSEC1.2S_Z{0.02}_logAGE{age}Myr_JWST_JHK.isoc',sep=' ',
                                   index=None, header = header)
                l += 1
            if len(temp)>0:
                tabs = pd.concat([tabs, temp])
    tabs_interp.append(tabs)  

tabs_interp = pd.concat(tabs_interp)

In [ ]:
t = tabs_interp[(tabs_interp['logAge']==7.9) & (tabs_interp['Zini']==0.014)  & (tabs_interp['label']>=1)  ]
x = t['F115Wmag']-t['F200Wmag']
y = t['F200Wmag'] + 29.81

c = t['label']
fig, ax = plt.subplots()
img= ax.scatter(x,y,s=1, c=c, cmap='tab10', vmin=0, vmax=10)

cb = plt.colorbar(img, ax=ax)
cb.set_ticks(np.arange(0.5,9.6,1))

cb.set_ticklabels(['PMS','MS','SGB','RGB','CHEB1','CHEB2','CHEB3', 'EAGB','TP-AGB','Post-AGB'],
                 fontsize=10)
ax.invert_yaxis()
ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=3)

ax.set_xlabel('logTe')
ax.set_ylabel('logL')
plt.show()


In [ ]:
x = t['Mini']
y = t['F115Wmag']
c = t['label']

fig, ax = plt.subplots()
img= ax.scatter(x,y,s=1, c=c, cmap='tab10', vmin=0, vmax=10)

cb = plt.colorbar(img, ax=ax)
cb.set_ticks(np.arange(0.5,9.6,1))

cb.set_ticklabels(['PMS','MS','SGB','RGB','CHEB1','CHEB2','CHEB3', 'EAGB','TP-AGB','Post-AGB'],
                 fontsize=10)
ax.invert_yaxis()
ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=3)

ax.set_xlabel('Mini')
ax.set_ylabel('F115Wmag')

plt.show()

In [ ]:
fig, ax = plt.subplots()
for i in [6.7,6.8,6.9, 7.0,7.1,7.2,7.4,7.5,7.6,7.7,7.8, 7.9, 8.0]:
    for j in [0.14, 0.017, 0.02]:

        t = tabs_interp[(np.round(tabs_interp['logAge'],1)==np.round(i,1)) & (tabs_interp['Zini']==j)  & (tabs_interp['label']>=1)]
        t = t[t['Mini']>=8]
        x = t['F115Wmag']-t['F200Wmag'] + (Av_dict['f115w'] - Av_dict['f200w'])*0.19
        y = t['F200Wmag'] + 29.81 +  Av_dict['f200w']*0.19
        c = np.log10(t['phase_prob']/t['phase_prob'].sum())
    
        img= ax.scatter(x,y,s=1, c=c, cmap='jet',)

cb = plt.colorbar(img, ax=ax)
cb.set_label('log(dN/N)')

ax.invert_yaxis()
ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=3)

ax.set_xlabel('F115W-F200W')
ax.set_ylabel('F200W')
plt.show()


In [ ]:
t = df_cmd_jwst_n[(df_cmd_jwst_n['logAge']==7.6) & (df_cmd_jwst_n['Zini']==0.014)  & (df_cmd_jwst_n['label']>=2)  ]
x = t['F115Wmag']-t['F200Wmag']
y = t['F200Wmag']

c = t['label']
fig, ax = plt.subplots()
img= ax.scatter(x,y,s=1, c=c, cmap='tab10', vmin=0, vmax=10)

cb = plt.colorbar(img, ax=ax)
cb.set_ticks(np.arange(0.5,9.6,1))

cb.set_ticklabels(['PMS','MS','SGB','RGB','CHEB1','CHEB2','CHEB3', 'EAGB','TP-AGB','Post-AGB'],
                 fontsize=10)
ax.invert_yaxis()
ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=3)

ax.set_xlabel('F115W-F200W')
ax.set_ylabel('F200W')
plt.show()


### **BaSTI**

In [ ]:
fs = glob.glob('../data/isochrones_master/basti_jairo/*/*/*')

In [ ]:
dfs = []
for f in fs:
    with open(f) as fi:
        dat = fi.readlines()
        
    Z_str = dat[4][dat[4].index('Z'):dat[4].index('Y')]
    Z = float(Z_str.split()[-1])
    age = np.log10(float(dat[4].split()[-1])*1e6)
    cols = dat[6].split()[1:]

    data = [[float(i) for i in d.split()] for d in dat[8:]]
    data = np.array(data)
    df = pd.DataFrame(data, columns=cols)
    df['Zini'] = Z
    df['logAge'] = age
    dfs.append(df)
df_cmd_jwst_basti = pd.concat(dfs)
df_cmd_jwst_basti.to_csv('../data/isochrones_master/cmd_jwst_basti_v2.csv', index=None)

In [ ]:
df_cmd_jwst_basti

In [ ]:
np.unique(df_cmd_jwst_basti['logAge'].values)

In [ ]:
import nest_asyncio
nest_asyncio.apply()
del nest_asyncio

# **Data**

## **Simulated observation**

In [ ]:
@models.custom_model
def pritchet(m, alpha=0.5, m_50=22):
    return 0.5*(1 - alpha*(m - m_50)/np.sqrt(1 + alpha**2*(m-m_50)**2))

@models.custom_model
def pritchet_inv(p, alpha=0.5, m_50=22):
    return m_50 + (1/alpha)*(1-2*p)/np.sqrt(1-(1-2*p)**2)


In [ ]:
df_cmd_jwst  = pd.read_csv("../data/isochrones_master/cmd_jwst_corr.csv")

In [ ]:
fs = glob.glob("../data/sim/JWST_1e5/*0.02")
fs = [i for i in  fs if float(i.split('_')[3])<9]
dfs = []
for fi in fs:
    with open(fi) as f:
        dat = f.readlines()
    
    data = []
    
    for i,d in enumerate(dat[13:]):
        if 'Z' not in d and '#' not in d:
            data.append([float(i) for i in d.split()])
            
    df_cmd = pd.DataFrame(data,columns=dat[13][2:].split())
    dfs.append(df_cmd)
df_cmd = pd.concat(dfs)
df_cmd['logAge'] = np.round(np.log10(df_cmd['age']),1)
df_cmd = df_cmd.drop_duplicates(['Mini','logAge','label'])
df_cmd.to_csv("../data/sim/JWST_1e5/sim_merged_Z0.02.csv", index=None)

In [ ]:
df_sim_001 = pd.read_csv("../data/sim/JWST_1e5/sim_merged_Z0.001.csv")
df_sim_002 = pd.read_csv("../data/sim/JWST_1e5/sim_merged_Z0.002.csv")
df_sim_003 = pd.read_csv("../data/sim/JWST_1e5/sim_merged_Z0.003.csv")

df_sim_014 = pd.read_csv("../data/sim/JWST_1e5/sim_merged_Z0.014.csv")
df_sim_017 = pd.read_csv("../data/sim/JWST_1e5/sim_merged_Z0.017.csv")
df_sim_020 = pd.read_csv("../data/sim/JWST_1e5/sim_merged_Z0.02.csv")

In [ ]:
np.unique(df_sim_001['Z']), np.unique(df_sim_002['Z']),np.unique(df_sim_003['Z'])

In [ ]:
np.unique(df_sim_014['Z']), np.unique(df_sim_017['Z']),np.unique(df_sim_020['Z'])

In [ ]:
tab = Table.read('../photometry/ngc628/f115w_f150w_f200w_photometry.fits')
df_cmd_jwst = pd.read_csv("../data/isochrones_master/cmd_jwst.csv")
df = tab[(tab['mag_err_F115W']<=0.2) & (tab['mag_err_F150W']<=0.2) & (tab['mag_err_F200W']<=0.2)] 

x = df['mag_vega_F115W']
y = df['mag_err_F115W']

init = models.Exponential1D()
fit = fitting.LevMarLSQFitter()
model_f115w = fit(init,x,y)

x = df['mag_vega_F150W']
y = df['mag_err_F150W']

model_f150w = fit(init,x,y)

x = df['mag_vega_F200W']
y = df['mag_err_F200W']

model_f200w = fit(init,x,y)

In [ ]:
ages    = np.arange(6.7,10.01,0.1)
ages_l  = ages - 0.5
ages_r  = ages + 0.5
delta_t = 10**ages_r - 10**ages_l

In [ ]:
SFR_in = 1e5/delta_t

In [ ]:
0.1/SFR_in

In [ ]:
df_sim1 = [df_sim_014, df_sim_017, df_sim_020]
df_sim2 = [df_sim_001, df_sim_002, df_sim_003]

mults   = {}

for s1 in range(3):
    for s2 in range(3):
    
        df_filt = pd.concat([df_sim1[s1], df_sim2[s2]]).copy()
        Zs = np.unique(df_filt['Z'])
        
        df_filt = df_filt.rename(columns = {'F115Wmag' : 'mag_vega_F115W',
                                            'F150Wmag' : 'mag_vega_F150W',
                                            'F200Wmag' : 'mag_vega_F200W'
                                           })
        
        df_filt['ra']        = np.float64(0)
        df_filt['dec']       = np.float64(0)
        
        tab_sim = Table.from_pandas(df_filt)
        Av = 0.19
        AF115  =  0.419*Av
        AF150  =  0.287*Av
        AF200  =  0.195*Av
        dismod =  29.81
        
        tab_sim['mag_vega_F115W'] += dismod + AF115
        tab_sim['mag_vega_F150W'] += dismod + AF150
        tab_sim['mag_vega_F200W'] += dismod + AF200
        
        tab_sim = tab_sim[tab_sim['mag_vega_F115W']<=f115w_m50.ravel()[0]]
        tab_sim = tab_sim[tab_sim['mag_vega_F150W']<=f150w_m50.ravel()[0]]
        tab_sim = tab_sim[tab_sim['mag_vega_F200W']<=f200w_m50.ravel()[0]]

        mult = np.ceil(400/tab_sim.to_pandas().groupby('logAge').count()['Z'].values).astype(int)

        mults[f'{Zs[1]}_{Zs[0]}'] = mult

In [ ]:
df_filt = pd.concat([df_sim_014,df_sim_002]).copy()

df_filt = df_filt.rename(columns = {'F115Wmag' : 'mag_vega_F115W',
                                    'F150Wmag' : 'mag_vega_F150W',
                                    'F200Wmag' : 'mag_vega_F200W'
                                   })

df_filt['ra']        = np.float64(0)
df_filt['dec']       = np.float64(0)

tab_sim = Table.from_pandas(df_filt)

Av     = 0.19
AF115  = 0.419*Av
AF150  = 0.287*Av
AF200  = 0.195*Av
dismod = 29.81

tab_sim['mag_vega_F115W'] += dismod + AF115
tab_sim['mag_vega_F150W'] += dismod + AF150
tab_sim['mag_vega_F200W'] += dismod + AF200

tab_sim = tab_sim[tab_sim['mag_vega_F115W']<29]
tab_sim = tab_sim[tab_sim['mag_vega_F150W']<29]
tab_sim = tab_sim[tab_sim['mag_vega_F200W']<29]

tab_sfr = []
ages = np.unique(df_filt['logAge'])


print(len(ages), len(mult))
for i,age in enumerate(ages):
    tab_t = tab_sim[tab_sim['logAge']==age]
    for j in range(mults['0.014_0.002'][i]):
        tab_sfr.append(tab_t)
        
tab_sfr = vstack(tab_sfr)

tab_sfr['mag_err_F115W']  = model_f115w(tab_sfr['mag_vega_F115W'])
tab_sfr['mag_vega_F115W'] = np.random.normal(loc=tab_sfr['mag_vega_F115W'], scale = tab_sfr['mag_err_F115W'])

tab_sfr['mag_err_F150W']  = model_f150w(tab_sfr['mag_vega_F150W'])
tab_sfr['mag_vega_F150W'] = np.random.normal(loc=tab_sfr['mag_vega_F150W'], scale = tab_sfr['mag_err_F150W'])

tab_sfr['mag_err_F200W']  = model_f200w(tab_sfr['mag_vega_F200W'])
tab_sfr['mag_vega_F200W'] = np.random.normal(loc=tab_sfr['mag_vega_F200W'], scale = tab_sfr['mag_err_F200W'])

In [ ]:
fig,axs = plt.subplots(1,2, figsize=(15,5))

x = tab_sim['mag_vega_F115W'] - tab_sim['mag_vega_F200W']
y = tab_sim['mag_vega_F200W']
c = tab_sim['label']
ax = axs[0]
ax.scatter(x,y,c=c, s=1, cmap='tab10', vmin=0, vmax=10)
ax.invert_yaxis()

ax.set_xlabel('F115W-F200W')
ax.set_ylabel('F200W')

x = tab_sfr['mag_vega_F115W'] - tab_sfr['mag_vega_F200W']
y = tab_sfr['mag_vega_F200W']
c = tab_sfr['label']
ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True,width=3)
ax.tick_params(which='minor', length=8,width=3)

ax = axs[1]
ax.scatter(x,y,c=c, s=1, cmap='tab10', vmin=0, vmax=10)

ax.invert_yaxis()

ax.set_xlabel(r'F115W-F200W')
ax.set_ylabel(r'F200W')

ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True,width=3)
ax.tick_params(which='minor', length=8,width=3)

In [ ]:
bins      = np.arange(0,1.01,0.01)
tab_cutss = []

mets_gas  = np.array([0.014, 0.017, 0.02])
mets_trgb = np.array([0.001, 0.002, 0.003])

gas_map  = {0.014: df_sim_014, 0.017: df_sim_017, 0.02: df_sim_020}
trgb_map = {0.001: df_sim_001, 0.002: df_sim_002, 0.003: df_sim_003}

for jj in range(10):
    tab_cuts = []
    for k in range(90):
        met_gas, met_trgb = met_map_muse.ravel()[k], met_map_trgb.ravel()[k]
    
                
        print(met_gas, met_trgb)
        
        # Handle gas metallicity
        if np.isnan(met_gas):
            met_gas = 0.014
        else:
            met_gas = mets_gas[np.argmin(abs(mets_gas - met_gas))]
        
        # Handle TRGB metallicity
        if np.isnan(met_trgb):
            met_trgb = 0.002
        else:
            met_trgb = mets_trgb[np.argmin(abs(mets_trgb - met_trgb))]
        
        # Map directly
        df_sim1 = gas_map[met_gas]
        df_sim2 = trgb_map[met_trgb]
                
        df_filt = pd.concat([df_sim1, df_sim2]).copy()

        df_filt = df_filt.rename(columns = {'F115Wmag' : 'mag_vega_F115W',
                                            'F150Wmag' : 'mag_vega_F150W',
                                            'F200Wmag' : 'mag_vega_F200W'
                                           })
        
        df_filt['ra']        = np.float64(0)
        df_filt['dec']       = np.float64(0)
        
        tab_sim = Table.from_pandas(df_filt)
        Av     = 0.19
        AF115  = 0.419*Av
        AF150  = 0.287*Av
        AF200  = 0.195*Av
        dismod = 29.81
        
        tab_sim['mag_vega_F115W'] += dismod + AF115
        tab_sim['mag_vega_F150W'] += dismod + AF150
        tab_sim['mag_vega_F200W'] += dismod + AF200
        
        tab_sim = tab_sim[tab_sim['mag_vega_F115W']<29]
        tab_sim = tab_sim[tab_sim['mag_vega_F150W']<29]
        tab_sim = tab_sim[tab_sim['mag_vega_F200W']<29]
        
        tab_sfr = []
        ages = np.unique(df_filt['logAge'])
        
    
        for i,age in enumerate(ages):
            tab_t = tab_sim[tab_sim['logAge']==age]
            for j in range(mults[f'{met_gas}_{met_trgb}'][i]):
                tab_sfr.append(tab_t)
                
        tab_sfr = vstack(tab_sfr)
        
        tab_sfr['mag_err_F115W']  = model_f115w(tab_sfr['mag_vega_F115W'])
        tab_sfr['mag_vega_F115W'] = np.random.normal(loc=tab_sfr['mag_vega_F115W'], scale = tab_sfr['mag_err_F115W'])
        
        tab_sfr['mag_err_F150W']  = model_f150w(tab_sfr['mag_vega_F150W'])
        tab_sfr['mag_vega_F150W'] = np.random.normal(loc=tab_sfr['mag_vega_F150W'], scale = tab_sfr['mag_err_F150W'])
        
        tab_sfr['mag_err_F200W']  = model_f200w(tab_sfr['mag_vega_F200W'])
        tab_sfr['mag_vega_F200W'] = np.random.normal(loc=tab_sfr['mag_vega_F200W'], scale = tab_sfr['mag_err_F200W'])
        
        tab_reg = tab_sfr.copy()
        tab_reg['comp_frac'] = pritchet(f200w_a.ravel()[k], f200w_m50.ravel()[k])(tab_reg['mag_vega_F200W'])*  \
                               pritchet(f150w_a.ravel()[k], f150w_m50.ravel()[k])(tab_reg['mag_vega_F150W'])*  \
                               pritchet(f115w_a.ravel()[k], f115w_m50.ravel()[k])(tab_reg['mag_vega_F115W'])

        tab_reg['ra'] = regions_dict[f'reg_{int(k)}']['ra']
        tab_reg['dec'] = regions_dict[f'reg_{int(k)}']['dec']

        tabs = []
        for i in range(len(bins)-1):
            df_cut = tab_reg[(tab_reg['comp_frac']>bins[i]) & (tab_reg['comp_frac']<=bins[i+1])].to_pandas()
            df = df_cut.sample(int(np.round(len(df_cut)*bins[i+1],0)))

            tabs.append(Table.from_pandas(df))

        tab_cut = vstack(tabs) 
        tab_cuts.append(tab_cut)
    tab_cuts = vstack(tab_cuts)
    tab_cutss.append(tab_cuts)
    
    print(jj)
    
    tab_cuts.write(f"../SFH/ngc628/sim_grid/comp_sim_reg_{jj}.fits", overwrite=True)
    

In [ ]:
df_sim1[0]

In [ ]:
bins = np.arange(0,1.01,0.01)

mets_gas  = np.array([0.014, 0.017, 0.02])
mets_trgb = np.array([0.001, 0.002, 0.003])

gas_map  = {0.014: df_sim_014, 0.017: df_sim_017, 0.02: df_sim_020}
trgb_map = {0.001: df_sim_001, 0.002: df_sim_002, 0.003: df_sim_003}

for jj in range(0,10):
    tab_cuts = []
    for k in range(len(df_m74_bubbles_s1)):

        met_gas, met_trgb = met_map_muse_bubble.ravel()[k], met_map_trgb_bubble.ravel()[k]
        
        #print(met_gas, met_trgb)
        
        # Handle gas metallicity
        if np.isnan(met_gas):
            met_gas = 0.014
        else:
            met_gas = mets_gas[np.argmin(abs(mets_gas - met_gas))]
        
        # Handle TRGB metallicity
        if np.isnan(met_trgb):
            met_trgb = 0.002
        else:
            met_trgb = mets_trgb[np.argmin(abs(mets_trgb - met_trgb))]
        
        # Map directly
        df_sim1 = gas_map[met_gas]
        df_sim2 = trgb_map[met_trgb]
        
        df_filt = pd.concat([df_sim1, df_sim2]).copy()
        df_filt = df_filt.rename(columns = {'F115Wmag' : 'mag_vega_F115W',
                                    'F150Wmag' : 'mag_vega_F150W',
                                    'F200Wmag' : 'mag_vega_F200W'
                                   })

        df_filt['ra']        = np.float64(0)
        df_filt['dec']       = np.float64(0)
        
        tab_sim = Table.from_pandas(df_filt)
        Av     =  0.19
        AF115  =  0.419*Av
        AF150  =  0.287*Av
        AF200  =  0.195*Av
        dismod =  29.81
        
        tab_sim['mag_vega_F115W'] += dismod + AF115
        tab_sim['mag_vega_F150W'] += dismod + AF150
        tab_sim['mag_vega_F200W'] += dismod + AF200
        
        tab_sim = tab_sim[tab_sim['mag_vega_F115W']<29]
        tab_sim = tab_sim[tab_sim['mag_vega_F150W']<29]
        tab_sim = tab_sim[tab_sim['mag_vega_F200W']<29]
        
        tab_sfr = []
        ages = np.unique(df_filt['logAge'])

        for i,age in enumerate(ages):
            tab_t = tab_sim[tab_sim['logAge']==age]
            for j in range(mults[f'{met_gas}_{met_trgb}'][i]):
                tab_sfr.append(tab_t)
                
        tab_sfr = vstack(tab_sfr)
        
        tab_sfr['mag_err_F115W']  = model_f115w(tab_sfr['mag_vega_F115W'])
        tab_sfr['mag_vega_F115W'] = np.random.normal(loc=tab_sfr['mag_vega_F115W'], scale = tab_sfr['mag_err_F115W'])
        
        tab_sfr['mag_err_F150W']  = model_f150w(tab_sfr['mag_vega_F150W'])
        tab_sfr['mag_vega_F150W'] = np.random.normal(loc=tab_sfr['mag_vega_F150W'], scale = tab_sfr['mag_err_F150W'])
        
        tab_sfr['mag_err_F200W']  = model_f200w(tab_sfr['mag_vega_F200W'])
        tab_sfr['mag_vega_F200W'] = np.random.normal(loc=tab_sfr['mag_vega_F200W'], scale = tab_sfr['mag_err_F200W'])
        tab_reg = tab_sfr.copy()
        tab_reg['comp_frac'] = pritchet(f200w_a_bubbles.ravel()[k], f200w_m50_bubbles.ravel()[k])(tab_reg['mag_vega_F200W'])*  \
                               pritchet(f150w_a_bubbles.ravel()[k], f150w_m50_bubbles.ravel()[k])(tab_reg['mag_vega_F150W'])*  \
                               pritchet(f115w_a_bubbles.ravel()[k], f115w_m50_bubbles.ravel()[k])(tab_reg['mag_vega_F115W'])

        tab_reg['ra']  = df_m74_bubbles_s1['ra'][k]
        tab_reg['dec'] = df_m74_bubbles_s1['dec'][k]

        tabs = []
        for i in range(len(bins)-1):
            df_cut = tab_reg[(tab_reg['comp_frac']>bins[i]) & (tab_reg['comp_frac']<=bins[i+1])].to_pandas()
            df = df_cut.sample(int(np.round(len(df_cut)*bins[i+1],0)))

            tabs.append(Table.from_pandas(df))

        tab_cut = vstack(tabs) 
        tab_cuts.append(tab_cut)
        break
        
    tab_cuts = vstack(tab_cuts)
    break
    tab_cuts.write(f"../SFH/ngc628/sim_bubble/comp_sim_{jj}.fits", overwrite=True)

In [ ]:
mults[f'{met_gas}_{met_trgb}']

In [ ]:
df = df_m74_bubbles_s1.reset_index()
df = df[df['Bubble ID']==0]

In [ ]:
tab = Table.read(f"../SFH/ngc628/sim_bubble/comp_sim_0.fits")

fig, ax = plt.subplots(figsize=(12,10))
ra_cen = df['ra'].values[0]
dec_cen = df['dec'].values[0]

filters = {'filt1':'f115w',
           'filt2':'f200w',
           'filt3':'f200w'}

positions = {'ra_col' : 'ra',
             'dec_col': 'dec',
             'ra_cen' : ra_cen,
             'dec_cen': dec_cen}

region = {'r_in':0,
          'r_out': 2,
          'spatial_filter': 'circle'}

extinction = {'Av'  : 0.19,
              'Av_x': 1.5,
              'Av_y': 25,
              'Av_' : 1}

axis_limits= {'xlims': [-0.6, 2], 
              'ylims': [17, 28]}

isochrone_params={'met': [0.014],
                  'label_min': 0,
                  'label_max': 10,
                  'ages': [6.7, 7, 7.7, 8,]}

error_settings = {'ref_xpos': -0.4,
                  'mag_err_lim':0.2}

plot_settings = {'s':5, 'legend.ncols':2, 'alpha':0.7, 'lw':3}

kde_contours = {'gen_kde' :False}
df_temp = df_cmd_jwst.copy()
df_temp = df_temp[((df_temp['logAge']==6.7) & (df_temp['label']<6)) | (df_temp['logAge']>6.7)]
fig,ax, tab1 = gen_CMD(tab, 
                      df_temp,
                      filters, 
                      positions,
                      region,
                      extinction,
                      29.81,
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      other_settings={'ab_dist': False, 'skip_data': False},
                      kde_contours=kde_contours,
                      fig=fig, ax=ax)

isochrone_params['ages'] = [9.0, 9.4, 10.]
isochrone_params['met'] = [0.002]

fig,ax, tab1 = gen_CMD(tab, 
                      df_cmd_jwst,
                      filters, 
                      positions,
                      region,
                      extinction,
                      29.81,
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      other_settings={'ab_dist': True, 'skip_data': True},
                      fig=fig, ax=ax)

#fig.savefig('figures/m51_cmd.png', bbox_inches='tight')

In [ ]:
len(tab1)

In [ ]:
x = tab1['ra']
y = tab1['dec']

plt.scatter(x,y)

In [ ]:
df_sim1 = [df_sim_014, df_sim_017, df_sim_020]
df_sim2 = [df_sim_001, df_sim_002, df_sim_003]

SFRs_dict = {}
for s1 in range(3):
    for s2 in range(3):
    
        tab_test = pd.concat([df_sim1[s1],df_sim2[s2]]).copy()
        Zs = np.unique(tab_test['Z'])
        Ni = tab_test.groupby('logAge').count()['age'].values
        ages = np.unique(tab_test['logAge'])
        
        ai = 1
        Ni*=ai
        
        alpha1 = 1.3
        alpha2 = 2.3
        Mc = 0.5
        Ml = 0.1
        
        Mx = np.array([tab_test[tab_test['logAge']==i]['Mini'].max() for i in ages])
        
        h2 = (Mx**(1.-alpha2)-Mc**(1.-alpha2))/(1.-alpha2)
        h2 *= Mc**(alpha2-alpha1)
        h1 = (Mc**(1.-alpha1)-Ml**(1.-alpha1))/(1.-alpha1)
        
        C1 = (Ni*0+1)/(h1+h2)
        C2 = C1*Mc**(alpha2-alpha1)
        
        Ncorrs_ = np.array(Ni)*mults[f'{Zs[1]}_{Zs[0]}']
        
        Mean_M = []
        
        for c1, c2, mu in zip(C1,C2,Mx):
            M = np.linspace(Ml,Mc, 1000)  
            Mean_M1 = np.trapz(M*c1*M**(-alpha1), M)
        
            M = np.linspace(Mc,mu, 1000)  
            Mean_M2 = np.trapz(M*c2*M**(-alpha2), M)
        
            Mean_M.append(Mean_M1+Mean_M2)
        
        Mean_M = np.array(Mean_M)
        
        ages_r = ages+0.05
        ages_l = ages-0.05
        
        #ages_l[0] = 6
        
        delta_t = 10**ages_r- 10**ages_l
        
        age_cen = 0.5*(ages_r + ages_l)
        
        SFRs = (Mean_M*Ncorrs_)/delta_t
        
        fig, ax = plt.subplots(figsize=(10,7))
        
        x_ = age_cen
        y_ = SFRs
        SFRs_dict[f'{Zs[1]}_{Zs[0]}'] = SFRs
        
        SFR_ref = SFRs
        ax.step(x_,y_,where='mid')
        
        ax.set_xlabel('log Age (yr)')
        ax.set_ylabel('SFR')
        
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which='both', width=2,direction="in", top = True,right = True,
                       bottom = True, left = True)
        ax.tick_params(which='major', length=7,direction="in")
        ax.tick_params(which='minor', length=4, color='black',direction="in")
        ax.set_yscale('log')
        plt.close(fig)
        np.save(f"../SFH/ncorrs_{Zs[1]}_{Zs[0]}.npy", Ncorrs_)

In [ ]:
df_cmd_jwst = pd.read_csv("../data/isochrones_master/cmd_jwst_corr.csv")

In [ ]:
tab_cuts = []
for i in range(10):
    tab_cuts.append(Table.read(f"../SFH/ngc628/sim_bubble/comp_sim_{i}.fits"))

In [ ]:
Av = 0.19
AF115  =  0.419*Av
AF150  =  0.287*Av
AF200  =  0.195*Av
dismod =  29.81

mets_gas = np.array([0.014, 0.017, 0.02])
mets_trgb = np.array([0.001, 0.002, 0.003])

gas_map = {0.014: df_sim_014, 0.017: df_sim_017, 0.02: df_sim_020}
trgb_map = {0.001: df_sim_001, 0.002: df_sim_002, 0.003: df_sim_003}

k = 1
fig, axs = plt.subplots(2,1,figsize=(15,10), sharex=True, height_ratios=[1,0.3])
obs_Ncorrs = []
ax = axs[0]
for tab_cut in tab_cuts:
    
    ra_cen = df_m74_bubble_sub['ra'][k]
    dec_cen =  df_m74_bubble_sub['dec'][k]
    a =  df_m74_bubble_sub['a'][k]
    b =  df_m74_bubble_sub['b'][k]
    ang = df_m74_bubble_sub['ang'][k]
    tab_test = jwst_data(tab_cut, ra_cen, dec_cen, 0, ang,  region_type='ellipse',a=a,b=b)

    met_gas, met_trgb = met_map_muse_bubble.ravel()[k], met_map_trgb_bubble.ravel()[k]

    
    # Handle gas metallicity
    if np.isnan(met_gas):
        met_gas = 0.014
    else:
        met_gas = mets_gas[np.argmin(abs(mets_gas - met_gas))]
    
    # Handle TRGB metallicity
    if np.isnan(met_trgb):
        met_trgb = 0.002
    else:
        met_trgb = mets_trgb[np.argmin(abs(mets_trgb - met_trgb))]

    fw1_lim = f115w_m50_bubbles.ravel()[k].ravel()[0]
    fw2_lim = f150w_m50_bubbles.ravel()[k].ravel()[0]
    fw3_lim = f200w_m50_bubbles.ravel()[k].ravel()[0]
    

    
    tab_test = tab_test[tab_test['fw1']<=fw1_lim]
    tab_test = tab_test[tab_test['fw2']<=fw2_lim]
    tab_test = tab_test[tab_test['fw3']<=fw3_lim]
    
    Ni       = tab_test.groupby('logAge').count()['age'].values
    ages     = np.unique(tab_test['logAge'])
    
    ai = 1
    Ni*=ai
    
    alpha1 = 1.3
    alpha2 = 2.3
    Mc = 0.5
    Ml = 0.1

    
    df_cmd = df_cmd_jwst[((df_cmd_jwst['logAge']>=6.7) & (df_cmd_jwst['logAge']<9)    & (df_cmd_jwst['Zini']==met_gas)) | 
                         ((df_cmd_jwst['logAge']>=9)   & (df_cmd_jwst['Zini']==met_trgb) & (df_cmd_jwst['logAge']<10.1))].copy()
    
    df_cmd['F115Wmag'] += dismod + AF115
    df_cmd['F150Wmag'] += dismod + AF150
    df_cmd['F200Wmag'] += dismod + AF200
    
    df_cmd = df_cmd[(df_cmd['F115Wmag']<=fw1_lim) & 
                    (df_cmd['F150Wmag']<=fw2_lim) & 
                    (df_cmd['F200Wmag']<=fw3_lim)]
    
    Mx = np.array([df_cmd[np.round(df_cmd['logAge'],1)==i]['Mini'].max() for i in ages])
    
    Mlim = np.array([df_cmd[np.round(df_cmd['logAge'],1)==i]['Mini'].min() for i in ages])
    
    C2 = (Ni*(1-alpha2))/(Mx**(1-alpha2) - Mlim**(1-alpha2))
    C1 = C2*Mc**(alpha1 - alpha2)
    
    Ncorrs = []
    for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni):
        M = np.linspace(Ml,Mc)  
        Nlim1 = np.trapz(c1*M**(-alpha1), M)
    
        M = np.linspace(Mc,mlim)  
        Nlim2 = np.trapz(c2*M**(-alpha2), M)
    
        Ncorr = ni + Nlim1 + Nlim2
        Ncorrs.append(Ncorr)
    
    Ncorrs = np.array(Ncorrs)
    
    h2 = (Mx[0]**(1.-alpha2)-Mc**(1.-alpha2))/(1.-alpha2)
    h2 *= Mc**(alpha2-alpha1)
    h1 = (Mc**(1.-alpha1)-Ml**(1.-alpha1))/(1.-alpha1)
    
    C1 = (Ni*0+1)/(h1+h2)
    C2 = C1*Mc**(alpha2-alpha1)
    
    Mean_M = []
    
    for c1, c2, mu in zip(C1,C2,Mx):
        M = np.linspace(Ml,Mc)  
        Mean_M1 = np.trapz(M*c1*M**(-alpha1), M)
    
        M = np.linspace(Mc,mu)  
        Mean_M2 = np.trapz(M*c2*M**(-alpha2), M)
    
        Mean_M.append(Mean_M1+Mean_M2)
    
    Mean_M = np.array(Mean_M)
    
    ages_r = ages + 0.05
    ages_l = ages - 0.05
    
    #ages_l[0] = 6
    
    delta_t = 10**ages_r- 10**ages_l
    
    age_cen = 0.5*(ages_r + ages_l)
    
    SFRs    = (Mean_M*Ncorrs)/delta_t
    obs_Ncorrs.append(Ncorrs)
    x = age_cen
    y = SFRs

    ax.step(x,Ncorrs,where='mid', label='Completeness limited', color='red')
ax.step(x,Ncorrs_,where='mid', label = 'Simulated', color='black')

ax.set_ylabel(r'${\rm N_i + N_{lim,i}}$')

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")
ax.set_yscale('log')
    #ax.legend(fontsize=30)
ax = axs[1]

obs_Ncorrs = np.array(obs_Ncorrs)
Ncorrs_val = np.percentile(Ncorrs_/obs_Ncorrs, [16,50,84], axis=0)

ax.step(x, Ncorrs_val[1], where='mid', label = 'Simulated', color='black')
ax.errorbar(x, Ncorrs_val[1], yerr = [Ncorrs_val[1] - Ncorrs_val[0], Ncorrs_val[2] - Ncorrs_val[1]],
           fmt='o', color='blue', markersize=0.5, capsize=2)
ax.set_xlabel(r'log (Age [yr$^{-1}$])')
ax.set_ylabel(r'$\xi_i$')
ax.set_yscale('log')

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")
plt.subplots_adjust(hspace=0)


In [ ]:
x = tab_test['ra']
y = tab_test['dec']

plt.scatter(x,y)

# **Star Formation History**

## **Run**

In [ ]:
import nest_asyncio
nest_asyncio.apply()
del nest_asyncio

In [ ]:
def sort_isos(isos):
    isos = sorted(isos, key = lambda x: float(x.split('_')[-3][6:-3]))
    for i in range(len(isos)):
        path = isos[i].split('/')
        if i < 9:
            path[-1] = filename = f'0{i+1}' + isos[i].split('/')[-1][2:]
        else:
            path[-1] = filename = f'{i+1}' + isos[i].split('/')[-1][2:]

        if isos[i] !='/'.join(path):
            os.system(f"mv {isos[i]} {'/'.join(path)}")
        isos[i] = '/'.join(path)


In [ ]:
tab = Table.read("../photometry/ngc628/f115w_f150w_f200w_photometry.fits")
#tab = tab_cut

for reggg in [0]:
    #tab = Table.read(f'../SFH/ngc628/sim_grid/comp_sim_reg_{reggg}.fits')
    #tab = Table.read(f'../SFH/ngc628/sim_bubble/comp_sim_{reggg}.fits')
    tab = tab[(tab['mag_err_F115W']<=0.2) & (tab['mag_err_F150W']<=0.2) & (tab['mag_err_F200W']<=0.2)]
    
    #for region in range(31,90):
    #    sig_i = 0.010
    for region, data in df_m74_bubbles_s1.reset_index().iterrows():

        Av_corr = 0#Av_map_trgb_bubble.ravel()[region]
        met_gas, met_trgb = met_map_muse_bubble.ravel()[region], met_map_trgb_bubble.ravel()[region]
        # Av_corr =  Av_map_trgb.ravel()[region]
        # met_gas, met_trgb = met_map_muse.ravel()[region], met_map_trgb.ravel()[region]
        
        print(met_gas, met_trgb)
        if np.isnan(met_gas):
            met_gas = 0.014
        else:
            mets = np.array([0.014,0.017,0.02])
            mets_diffs = abs(mets - met_gas)
            ind = mets_diffs == mets_diffs.min()
            met_gas = mets[ind][0]
    
        if np.isnan(met_trgb):
            met_trgb = 0.002  
        else:
            mets = np.array([0.001, 0.002, 0.003])
            mets_diffs = abs(mets - met_trgb)
            ind = mets_diffs == mets_diffs.min()
            met_trgb = mets[ind][0]
           
        Av     =  0.19 + Av_corr
        AF115  =  0.419*Av
        AF150  =  0.287*Av
        AF200  =  0.195*Av
        dismod =  29.81

        sig_i  = 0.010
    
        # JWST
    
        # Grid
        # fws = [f115w_m50.ravel()[region], f150w_m50.ravel()[region], f200w_m50.ravel()[region]]
        # region = f'reg_{region}'
        # data = regions_dict[region]
        # r_in       = 0
        # r_out      = 24 #data['radius']
        # ang        = 245.00492
        # ra_center  = data['ra']
        # dec_center = data['dec']
        # df_filt = jwst_data(tab, ra_center, dec_center, r_out, ang,  region_type='box')
        
        # Bubbles
        a          = data['a']
        b          = data['b']
        ang        = data['ang']
        ra_center  = data['ra']
        dec_center = data['dec']
        
        df_filt = jwst_data(tab, ra_center, dec_center, 0, ang,  region_type='ellipse',a=a,b=b)
        fws     = [f115w_m50_bubbles[region], f150w_m50_bubbles[region], f200w_m50_bubbles[region]]
    
        #fws = [30,30,30]
     
        print(met_gas, met_trgb)

        os.system(f'rm ../data/isochrones/JWST_temp2/*')
    
        os.system(f'cp ../data/isochrones/JWST_{met_gas}/* ../data/isochrones/JWST_temp2/')
        os.system(f'cp ../data/isochrones/JWST_{met_trgb}/* ../data/isochrones/JWST_temp2/')

        
        isos = glob.glob(f'../data/isochrones/JWST_temp2/*')
        sort_isos(isos)
        
        out_dir = f'../SFH/ngc628/JWST_bubble_obs/'
        os.makedirs(out_dir,exist_ok=True)
        os.makedirs(out_dir + 'output/',exist_ok=True)
        os.makedirs(out_dir + 'plots/',exist_ok=True)
        
        sfh = BaySFH(df_filt, parallel=True,N_wlk=20, N_smp=500, 
                  isodir='../data/isochrones/JWST_temp2/', 
                  fw1_lim=fws[0],fw2_lim=fws[1], fw3_lim=fws[2],
                  sig_fw1=sig_i, sig_fw2=sig_i, sig_fw3=sig_i,
                  A_fw1=AF115, A_fw2=AF150, A_fw3=AF200, 
                     dismod=dismod, out_dir=out_dir+'output/',
                     n_cores=1)
        
        Ncorrs_ = np.load(f"../SFH/ncorrs_{met_gas}_{met_trgb}.npy")
  
        fname = sfh()
        fname = out_dir +'output/'+ fname
        df_sfh = pd.read_csv(fname)
        P_ij = np.array(sfh.P_ij)
        
        ai_10 = df_sfh['p10'].values
        
        ai = df_sfh['p50'].values
        
        ai_90 = df_sfh['p90'].values
        
        ages = df_sfh['Log_age'].values
        df_sfh['Ni'] = ai/ai.sum()*sfh.dat.T.shape[0]
        
        df_sfh.to_csv(f'{out_dir}/{region}.csv', index = None)
        p_age = P_ij*ai
        P = []
        P_age = []
        
        for i in p_age:
            prob = i[i==i.max()]/i.sum()
            ind = np.where(i==i.max())[0][0]
            P.append(prob)
            P_age.append(ages[ind])
            
        df = pd.DataFrame(sfh.dat.T, columns=['RA','DEC','mag_vega_F115W',
                                        'mag_err_F115W', 'mag_vega_F150W',
                                        'mag_err_F150W','mag_vega_F200W',
                                        'mag_err_F200W'])
        
        df['Prob'] = P
        df['Log_Age'] = P_age
        df.to_csv(f'{out_dir}/{region}_spatial.csv', index = None)
        
        #F115W-F150W CMD
        x = df['mag_vega_F115W'] - df['mag_vega_F150W'] + (AF115-AF150)
        y = df['mag_vega_F150W'] + dismod + AF150
        c = df['Log_Age']
        fig, ax = plt.subplots(figsize=(12, 10))
    
        img = ax.scatter(x,y,c=c,s=1, cmap='jet', label='_nolegend_')
        cb = plt.colorbar(img, ax=ax)
        ind = -1
        ax.scatter(sfh.Iso[ind][2]-sfh.Iso[ind][3] + (AF115-AF150), sfh.Iso[ind][3] + dismod+ AF150, 
                                                   label = f'{ages[ind]}', color='black',s=5,alpha=0.8)
        ax.set_ylim(18, fws[1]+1)
        ax.set_xlim(-0.5,2)
        ax.set_xlabel('F115W-F150W')
        ax.set_ylabel('F150W')
        ax.invert_yaxis()
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
                       bottom = True, left = True)
        ax.tick_params(which='major', length=15,direction="in")
        ax.tick_params(which='minor', length=8, color='black',direction="in")
        
        fig.savefig(f'{out_dir}/plots/{region}_F115W-F150W_CMD.png', bbox_inches='tight')
        plt.close(fig)
        
        #F150W-F200W CMD
        x = df['mag_vega_F150W'] - df['mag_vega_F200W'] + (AF150-AF200)
        y = df['mag_vega_F200W'] + dismod + AF200
        c = df['Log_Age']
        fig, ax = plt.subplots(figsize=(12, 10))
    
        img = ax.scatter(x,y,c=c,s=1, cmap='jet', label='_nolegend_')
        cb = plt.colorbar(img, ax=ax)
        ind = -1
        ax.scatter(sfh.Iso[ind][3]-sfh.Iso[ind][4] + (AF150-AF200), sfh.Iso[ind][4] + dismod+ AF200, 
                                                   label = f'{ages[ind]}', color='black',s=5,alpha=0.8)
        ax.set_ylim(18, fws[2]+1)
        ax.set_xlim(-0.5,2)
        ax.set_xlabel('F150W-F200W')
        ax.set_ylabel('F200W')
        ax.invert_yaxis()
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
                       bottom = True, left = True)
        ax.tick_params(which='major', length=15,direction="in")
        ax.tick_params(which='minor', length=8, color='black',direction="in")
        
        fig.savefig(f'{out_dir}/plots/{region}_F150W-F200W_CMD.png', bbox_inches='tight')
        plt.close(fig)
        
        #F115W-F200W CMD
        x = df['mag_vega_F115W'] - df['mag_vega_F200W'] + (AF115-AF200)
        y = df['mag_vega_F200W'] + dismod + AF200
        c = df['Log_Age']
        fig, ax = plt.subplots(figsize=(12, 10))
    
        img = ax.scatter(x,y,c=c,s=1, cmap='jet', label='_nolegend_')
        cb = plt.colorbar(img, ax=ax)
        ind = -1
        ax.scatter(sfh.Iso[ind][2]-sfh.Iso[ind][4] + (AF115-AF200), sfh.Iso[ind][4] + dismod+ AF200, 
                                                   label = f'{ages[ind]}', color='black',s=5,alpha=0.8)
        ax.set_ylim(18, fws[2]+1)
        ax.set_xlabel('F115W-F200W')
        ax.set_ylabel('F200W')
        ax.set_xlim(-0.5,3)
        ax.invert_yaxis()
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
                       bottom = True, left = True)
        ax.tick_params(which='major', length=15,direction="in")
        ax.tick_params(which='minor', length=8, color='black',direction="in")
        
        fig.savefig(f'{out_dir}/plots/{region}_F115W-F200W_CMD.png', bbox_inches='tight')
        plt.close(fig)
        
        df_out = pd.read_csv(fname)
    
        fig, ax = plt.subplots(figsize=(12,8))
    
        N = len(df)
        Ni = N*ai/ai.sum()
        x = df_out['Log_age']
        y = Ni
    
        ax.step(x,y,where='mid', color='blue')
    
        ax.set_xlabel('log Age (yr)')
        ax.set_ylabel('No of stars')
    
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
                       bottom = True, left = True)
        ax.tick_params(which='major', length=15,direction="in")
        ax.tick_params(which='minor', length=8, color='black',direction="in")
        ax.set_yscale('log')
        fig.savefig(f'{out_dir}/plots/{region}_counts.png', bbox_inches='tight')
        plt.close(fig)
        
        # Probability to SFH
        
        alpha1 = 1.3
        alpha2 = 2.3
        Mc = 0.5
        Ml = 0.1
        
        FW1 = fws[0] - AF115 - dismod
        FW2 = fws[1] - AF150 - dismod
        FW3 = fws[2] - AF200 - dismod
        
        Mx = np.array([i[1].max() for i in sfh.Iso])
        Mlim = []
        for i in sfh.Iso:
            j = i[1][(i[2]<=FW1) & (i[3]<=FW2) & (i[4]<=FW3)]
            if len(j)>0:
                Mlim.append(j.min())
            else:
                Mlim.append(0)
        Mlim = np.array(Mlim)
        
        C2 = (Ni*(1-alpha2))/(Mx**(1-alpha2) - Mlim**(1-alpha2))
        C1 = C2*Mc**(alpha1 - alpha2)
        
        # Ncorrs
        Ncorrs = []
        for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni):
            if mlim!=0:
                M = np.linspace(Ml,Mc,1000)  
                Nlim1 = np.trapz(c1*M**(-alpha1), M)
    
                M = np.linspace(Mc,mlim,1000)  
                Nlim2 = np.trapz(c2*M**(-alpha2), M)
    
                Ncorr = ni + Nlim1 + Nlim2
            else:
                Ncorr = 0
            Ncorrs.append(Ncorr)
        Ncorrs = np.array(Ncorrs)
        
        # N errors
        Ni_up = (ai_90-ai)*Ni + Ni
        Ni_down = + Ni - (ai-ai_10)*Ni
        
        C2 = ((Ni_up)*(1-alpha2))/(Mx**(1-alpha2) - Mlim**(1-alpha2))
        C1 = C2*Mc**(alpha1 - alpha2)
        
        Ncorrs_up = []
        for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni_up):
            if mlim!=0:
                M = np.linspace(Ml,Mc,1000)  
                Nlim1 = np.trapz(c1*M**(-alpha1), M)
    
                M = np.linspace(Mc,mlim,1000)  
                Nlim2 = np.trapz(c2*M**(-alpha2), M)
    
                Ncorr = ni + Nlim1 + Nlim2
            else:
                Ncorr = 0
            Ncorrs_up.append(Ncorr)
        Ncorrs_up = np.array(Ncorrs_up)
        
        C2 = ((Ni_down)*(1-alpha2))/(Mx**(1-alpha2) - Mlim**(1-alpha2))
        C1 = C2*Mc**(alpha1 - alpha2)
        
        Ncorrs_down = []
        for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni_down):
            if mlim!=0:
                M = np.linspace(Ml,Mc,1000)  
                Nlim1 = np.trapz(c1*M**(-alpha1), M)
    
                M = np.linspace(Mc,mlim,1000)  
                Nlim2 = np.trapz(c2*M**(-alpha2), M)
    
                Ncorr = ni + Nlim1 + Nlim2
            else:
                Ncorr = 0
            Ncorrs_down.append(Ncorr)
        Ncorrs_down = np.array(Ncorrs_down)
        
        # Mean Mass
        h2 = (Mx**(1.-alpha2)-Mc**(1.-alpha2))/(1.-alpha2)
        h2 *= Mc**(alpha2-alpha1)
        h1 = (Mc**(1.-alpha1)-Ml**(1.-alpha1))/(1.-alpha1)
    
        C1 = 1/(h1+h2)
        C2 = C1*Mc**(alpha2-alpha1)
    
        Mean_M = []
        for c1, c2, mu in zip(C1,C2,Mx):
            M = np.linspace(Ml,Mc,1000)  
            Mean_M1 = np.trapz(M*c1*M**(-alpha1), M)
    
            M = np.linspace(Mc,mu,1000)  
            Mean_M2 = np.trapz(M*c2*M**(-alpha2), M)
            
            Mean_M.append(Mean_M1+Mean_M2)
          
        Mean_M = np.array(Mean_M)
        
        ages = df_sfh['Log_age'].values
        ages_r = ages+0.05
        ages_l = ages-0.05
        
        ages_l[0] = 6
        
        delta_t = 10**ages_r- 10**ages_l
    
        age_cen = 0.5*(ages_r + ages_l)
        
        SFRs = (Mean_M*Ncorrs)/delta_t
        SFRs_err_up = abs((Mean_M*Ncorrs_up)/delta_t - SFRs)
        SFRs_err_down = abs((Mean_M*Ncorrs_down)/delta_t - SFRs)
        
        df_out['Ncorr'] = Ncorrs
        df_out['SFR']   = SFRs
        
        x = age_cen
        y = SFRs*1e3
        
        fig, ax = plt.subplots(figsize=(12,8))
        ax.step(x,y,where='mid', color='blue')
        ax.errorbar(x,y,yerr=[SFRs_err_up*1e3, SFRs_err_down*1e3],
                    fmt='o', color = 'red', markersize=0.5, capsize=2)
        
        #ax.step(x_,10*y_*1e3,where='mid')
        
        ax.set_xlabel(r'$\log(Age)~[yr]$')
        ax.set_ylabel(r'$SFR~[[10^{-3}]M_{\odot}.yr^{-1}]$')
    
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
                       bottom = True, left = True)
        ax.tick_params(which='major', length=15,direction="in")
        ax.tick_params(which='minor', length=8, color='black',direction="in")
        ax.set_yscale('log')
        fig.savefig(f'{out_dir}/plots/{region}_SFH_log.png', bbox_inches='tight')
        plt.close(fig)
        
        fig, ax = plt.subplots(figsize=(12,8))
        ax.step(x,y,where='mid', color='blue')
        ax.errorbar(x,y,yerr=[SFRs_err_up*1e3, SFRs_err_down*1e3],
                    fmt='o', color = 'red', markersize=0.5, capsize=2)
        #ax.step(x_,y_*1e3,where='mid')
        
    
        ax.set_xlabel(r'$\log(Age)~[yr]$')
        ax.set_ylabel(r'$SFR~[[10^{-3}]M_{\odot}.yr^{-1}]$')
    
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
                       bottom = True, left = True)
        ax.tick_params(which='major', length=15,direction="in")
        ax.tick_params(which='minor', length=8, color='black',direction="in")
        fig.savefig(f'{out_dir}/plots/{region}_SFH.png', bbox_inches='tight')
        plt.close(fig)
        df_out.to_csv(f'{out_dir}/{region}_SFH.csv', index = None)
    
    print("Completed Bubbles!!!")

In [ ]:
sfh_obs = glob.glob("../SFH/ngc628/JWST_grid_obs/*SFH.csv")
sfh_spatial = glob.glob("../SFH/ngc628/JWST_grid_obs/*spatial.csv")
sfh_sim = glob.glob("../SFH/ngc628/JWST_grid_sim/?/*SFH.csv")


sfh_obs = sorted(sfh_obs, key=lambda x: int(x.split("_")[-2]))
sfh_spatial = sorted(sfh_spatial, key=lambda x: int(x.split("_")[-2]))
sfh_sim = sorted(sfh_sim, key=lambda x: int(x.split("_")[-2]))

## **Ncorr**

In [ ]:
!rm bubble_sim.zip

In [ ]:
!zip bubble_sim.zip ../SFH/ngc628/JWST_bubble_sim/figures/*

In [ ]:
reg_corrs   = []
reg_corr_hi = []
reg_corr_lo = []

Av     =  0.19
AF115  =  0.419*Av
AF150  =  0.287*Av
AF200  =  0.195*Av
dismod =  29.81

tab = Table.read("../photometry/ngc628/f115w_f150w_f200w_photometry.fits")
df_filt = jwst_data(tab, 0, 0, 0, 0,  region_type='ellipse',a=1,b=1)

for p in range(90):
    if p!=10:
        continue
    fig, axs = plt.subplots(2,1, figsize=(15,10), sharex = True, height_ratios=[1,0.4])

    fws = [f115w_m50.ravel()[p], f150w_m50.ravel()[p], f200w_m50.ravel()[p]]
    met_gas, met_trgb = met_map_muse.ravel()[p], met_map_trgb.ravel()[p]
    
    print(met_gas, met_trgb)
    if np.isnan(met_gas):
        met_gas = 0.014
    else:
        mets = np.array([0.014,0.017,0.02])
        mets_diffs = abs(mets - met_gas)
        ind = mets_diffs == mets_diffs.min()
        met_gas = mets[ind][0]
    
    if np.isnan(met_trgb):
        met_trgb = 0.002  
    else:
        mets = np.array([0.001, 0.002, 0.003])
        mets_diffs = abs(mets - met_trgb)
        ind = mets_diffs == mets_diffs.min()
        met_trgb = mets[ind][0]

    alpha1 = 1.3
    alpha2 = 2.3
    Mc = 0.5
    Ml = 0.1
    Mu = [350]
    
    FW1 = fws[0] - AF115 - dismod
    FW2 = fws[1] - AF150 - dismod
    FW3 = fws[2] - AF200 - dismod

    os.system(f'rm ../data/isochrones/JWST_temp/*')

    os.system(f'cp ../data/isochrones/JWST_{met_gas}/* ../data/isochrones/JWST_temp/')
    os.system(f'cp ../data/isochrones/JWST_{met_trgb}/* ../data/isochrones/JWST_temp/')

    isos = glob.glob(f'../data/isochrones/JWST_temp/*')
    sort_isos(isos)
    
    out_dir = '../SFH/ngc628/JWST_dummy'

    sfh = BaySFH(df_filt, parallel=True, isodir='../data/isochrones/JWST_temp/', 
              fw1_lim=fws[0],fw2_lim=fws[1], fw3_lim=fws[2],
              sig_fw1=sig_i, sig_fw2=sig_i, sig_fw3=sig_i,
              A_fw1=AF115, A_fw2=AF150, A_fw3=AF200, 
             dismod=dismod, out_dir=out_dir+'output/',
             n_cores=1)
    
    Mx = np.array([i[1].max() for i in sfh.Iso])
    Mlim = []
    for i in sfh.Iso:
        j = i[1][(i[2]<=FW1) & (i[3]<=FW2) & (i[4]<=FW3)]
        if len(j)>0:
            Mlim.append(j.min())
        else:
            Mlim.append(0)
    Mlim = np.array(Mlim)
    
    sfh_sim = glob.glob(f"../SFH/ngc628/JWST_grid_sim/?/reg_{p}_SFH.csv")
    spatial_sim = glob.glob(f"../SFH/ngc628/JWST_grid_sim/?/reg_{p}_spatial.csv")

    sim    = pd.read_csv(sfh_sim[0])
    ages   = sim['Log_age'].values
    ages_r = ages+0.05
    ages_l = ages-0.05
    #ages_l[0] = 6
    delta_t = 10**ages_r- 10**ages_l
    age_cen = 0.5*(ages_r + ages_l)
    
    ax             = axs[0]
    Ncorrs_reco    = []
    Ncorrs_reco_hi = []
    Ncorrs_reco_lo = []
    for f1, f2 in zip(sfh_sim, spatial_sim):
        sim     = pd.read_csv(f1)
        spatial = pd.read_csv(f2)

        N     = len(spatial)
        ai_10 = sim['p10'].values
        ai    = sim['p50'].values
        ai_90 = sim['p90'].values

        Ni = (ai/ai.sum())*N
                
        C2 = (Ni*(1-alpha2))/(Mx**(1-alpha2) - Mlim**(1-alpha2))
        C1 = C2*Mc**(alpha1 - alpha2)
        
        # Ncorrs
        Ncorrs = []
        for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni):
            if mlim!=0:
                M = np.linspace(Ml,Mc,10000)  
                Nlim1 = np.trapz(c1*M**(-alpha1), M)
    
                M = np.linspace(Mc,mlim,10000)  
                Nlim2 = np.trapz(c2*M**(-alpha2), M)
    
                Ncorr = ni + Nlim1 + Nlim2
            else:
                Ncorr = 0
            Ncorrs.append(Ncorr)
        Ncorrs = np.array(Ncorrs)
         
        # Ncorrs high
        Ncorrs_hi = []
        Ni_hi = Ni + (ai_90-ai)*Ni 

        Ni_hi = (Ni_hi/Ni_hi.sum())*N
        
        for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni_hi):
            if mlim!=0:
                M = np.linspace(Ml,Mc,10000)  
                Nlim1 = np.trapz(c1*M**(-alpha1), M)
    
                M = np.linspace(Mc,mlim,10000)  
                Nlim2 = np.trapz(c2*M**(-alpha2), M)
    
                Ncorr = (ni + np.sqrt(ni))*(1 + (Nlim1 + Nlim2)/ni)
            else:
                Ncorr = 0
            Ncorrs_hi.append(Ncorr)
            
        Ncorrs_hi = np.array(Ncorrs_hi)
    
        # Ncorrs low
        Ncorrs_lo = []
        Ni_lo = Ni - (ai-ai_10)*Ni
        Ni_lo = (Ni_lo/Ni_lo.sum())*N
        
        for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni_lo):
            if mlim!=0:
                M = np.linspace(Ml,Mc,10000)  
                Nlim1 = np.trapz(c1*M**(-alpha1), M)
    
                M = np.linspace(Mc,mlim,10000)  
                Nlim2 = np.trapz(c2*M**(-alpha2), M)
    
                Ncorr = (ni - np.sqrt(ni))*(1 + (Nlim1 + Nlim2)/ni)
            else:
                Ncorr = 0
            Ncorrs_lo.append(Ncorr)
            
        Ncorrs_lo = np.array(Ncorrs_lo)
        
        Ncorrs_reco.append(Ncorrs)
        Ncorrs_reco_hi.append(Ncorrs_hi)
        Ncorrs_reco_lo.append(Ncorrs_lo)
        
        ax.step(age_cen, Ncorrs, where='mid', color='red', lw=1.2)

    Ncorrs_reco    = np.array(Ncorrs_reco)
    Ncorrs_reco_hi = np.array(Ncorrs_reco_hi)
    Ncorrs_reco_lo = np.array(Ncorrs_reco_lo)
    
    Ncorrs_in = np.load(f'../SFH/ncorrs_{met_gas}_{met_trgb}.npy')
    
    corr_vals    = np.percentile(Ncorrs_in/Ncorrs_reco, [16,50,84], axis=0) 
    corr_vals_hi = np.percentile(Ncorrs_in/Ncorrs_reco_hi, [16,50,84], axis=0)
    corr_vals_lo = np.percentile(Ncorrs_in/Ncorrs_reco_lo, [16,50,84], axis=0) 

    # Median corr
    corr = corr_vals[1]

    # Error in corr
    corr_errs_high1 = corr_vals[2] -  corr_vals[1]
    corr_errs_high2 = corr_vals_hi[2] -  corr_vals_hi[1]
    corr_errs_high  = np.sqrt(corr_errs_high1**2 + corr_errs_high2**2)

    corr_errs_low1 = corr_vals[1] -  corr_vals[0]
    corr_errs_low2 = corr_vals_lo[1] -  corr_vals_lo[0]
    corr_errs_low  = np.sqrt(corr_errs_low1**2 + corr_errs_low2**2) 

    reg_corrs.append(corr)
    reg_corr_hi.append(corr_errs_high)
    reg_corr_lo.append(corr_errs_low)
    

    ax.step(age_cen, Ncorrs_in, where='mid', color='black',lw =3)

    ax.set_yscale('log')
    ax.set_ylabel(r'${\rm N_i + N_{lim,i}}$')
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
                   bottom = True, left = True)
    ax.tick_params(which='major', length=15,direction="in")
    ax.tick_params(which='minor', length=8, color='black',direction="in")
    ax.minorticks_on()
    
    ax = axs[1]
    
    ax.step(age_cen, corr, where='mid', label = 'Simulated', color='black', lw=2)
    ax.errorbar(age_cen, corr, yerr = [corr_errs_low, corr_errs_high],
               fmt='o', color='blue', markersize=1, capsize=3, lw=2)
    ax.set_xlabel(r'log (Age [yr$^{-1}$])')
    ax.set_ylabel(r'${\rm \xi_i}$')
    ax.axhline(1, lw=2, linestyle='--', color='black', alpha=0.8)
    ax.set_ylim(0.3, 1e3)
    ax.set_yscale('log')

    
    #ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(1.0, 10.0)*0.1, numticks=10))
    ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
                   bottom = True, left = True)
    ax.tick_params(which='major', length=15,direction="in")
    ax.tick_params(which='minor', length=8, color='black',direction="in")
    #ax.minorticks_on()
    
    plt.subplots_adjust(hspace=0)
    break
    fig.savefig(f'../SFH/ngc628/JWST_grid_sim/figures/region_{p}.png',bbox_inches='tight')
    plt.close(fig)

reg_corrs   = np.array(reg_corrs)
reg_corr_hi = np.array(reg_corr_hi)
reg_corr_lo = np.array(reg_corr_lo)

In [ ]:
np.save('../SFH/ngc628/reg_corrs.npy', reg_corrs) 
np.save('../SFH/ngc628/reg_corrs_hi.npy', reg_corr_hi) 
np.save('../SFH/ngc628/reg_corrs_lo.npy', reg_corr_lo) 

In [ ]:
reg_corrs = np.load('../SFH/ngc628/reg_corrs.npy') 

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))

ages = np.arange(6.7,10.01,0.1)
ages_r = ages+0.05
ages_l = ages-0.05

#ages_l[0] = 6

age_cen = 0.5*(ages_l + ages_r)
delta_t = 10**ages_r- 10**ages_l

for corr in reg_corrs:

    ax.step(age_cen, corr, where='mid', label = 'Simulated', color='black', alpha=0.2,
           lw=2)
    # ax.errorbar(age_cen, corr, yerr = [corr_errs_low, corr_errs_high],
    #            fmt='o', color='blue', markersize=0.5, capsize=2)

ax.step(age_cen, reg_corrs[9], where='mid', label = 'Simulated', color='red', alpha=1,
       lw=2)
    
ax.set_xlabel(r'log (Age [yr$^{-1}$])')
ax.set_ylabel(r'${\rm \xi_i}$')
ax.set_yscale('log')

#ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(1.0, 10.0)*0.1, numticks=10))
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")
ax.minorticks_on()

In [ ]:
tab = Table.read("../photometry/ngc628/f115w_f150w_f200w_photometry.fits")
df_filt = jwst_data(tab, 0, 0, 0, 0,  region_type='ellipse',a=1,b=1)

In [ ]:
bubble_corrs   = []
bubble_corr_hi = []
bubble_corr_lo = []

os.makedirs('../SFH/ngc628/JWST_bubble_sim/figures', exist_ok=True)

Av     =  0.19
AF115  =  0.419*Av
AF150  =  0.287*Av
AF200  =  0.195*Av
dismod =  29.81


for p in range(len(df_m74_bubbles_s1)):
    # if p!=27:
    #     continue
    fig, axs = plt.subplots(2,1, figsize=(15,10), sharex = True, height_ratios=[1,0.4])

    fws               = [f115w_m50_bubbles.ravel()[p], f150w_m50_bubbles.ravel()[p], f200w_m50_bubbles.ravel()[p]]
    met_gas, met_trgb = met_map_muse_bubble.ravel()[p], met_map_trgb_bubble.ravel()[p]
    
    print(met_gas, met_trgb)
    if np.isnan(met_gas):
        met_gas = 0.014
    else:
        mets = np.array([0.014,0.017,0.02])
        mets_diffs = abs(mets - met_gas)
        ind = mets_diffs == mets_diffs.min()
        met_gas = mets[ind][0]
    
    if np.isnan(met_trgb):
        met_trgb = 0.002  
    else:
        mets = np.array([0.001, 0.002, 0.003])
        mets_diffs = abs(mets - met_trgb)
        ind = mets_diffs == mets_diffs.min()
        met_trgb = mets[ind][0]

    alpha1 = 1.3
    alpha2 = 2.3
    Mc     = 0.5
    Ml     = 0.1
    Mu     = [350]
    
    FW1 = fws[0] - AF115 - dismod
    FW2 = fws[1] - AF150 - dismod
    FW3 = fws[2] - AF200 - dismod

    os.system(f'rm ../data/isochrones/JWST_temp/*')

    os.system(f'cp ../data/isochrones/JWST_{met_gas}/* ../data/isochrones/JWST_temp/')
    os.system(f'cp ../data/isochrones/JWST_{met_trgb}/* ../data/isochrones/JWST_temp/')

    isos = glob.glob(f'../data/isochrones/JWST_temp/*')
    sort_isos(isos)
    
    out_dir = '../SFH/ngc628/JWST_dummy'
    
    os.makedirs(out_dir,exist_ok=True)
    os.makedirs(out_dir + 'output/',exist_ok=True)
    os.makedirs(out_dir + 'plots/',exist_ok=True)

    sig_i = 0.01
    sfh = BaySFH(df_filt, parallel=True, isodir='../data/isochrones/JWST_temp/', 
              fw1_lim=fws[0],fw2_lim=fws[1], fw3_lim=fws[2],
              sig_fw1=sig_i, sig_fw2=sig_i, sig_fw3=sig_i,
              A_fw1=AF115, A_fw2=AF150, A_fw3=AF200, 
             dismod=dismod, out_dir=out_dir+'output/',
             n_cores=6)
    
    Mx = np.array([i[1].max() for i in sfh.Iso])
    Mlim = []
    for i in sfh.Iso:
        j = i[1][(i[2]<=FW1) & (i[3]<=FW2) & (i[4]<=FW3)]
        if len(j)>0:
            Mlim.append(j.min())
        else:
            Mlim.append(0)
    Mlim = np.array(Mlim)
    
    sfh_sim = glob.glob(f"../SFH/ngc628/JWST_bubble_sim/?/{p}_SFH.csv")
    spatial_sim = glob.glob(f"../SFH/ngc628/JWST_bubble_sim/?/{p}_spatial.csv")

    sim = pd.read_csv(sfh_sim[0])
    ages = sim['Log_age'].values
    ages_r = ages+0.05
    ages_l = ages-0.05
    #ages_l[0] = 6
    delta_t = 10**ages_r- 10**ages_l
    age_cen = 0.5*(ages_r + ages_l)
    
    ax = axs[0]
    Ncorrs_reco = []
    Ncorrs_reco_hi = []
    Ncorrs_reco_lo = []
    
    for f1, f2 in zip(sfh_sim, spatial_sim):
        sim     = pd.read_csv(f1)
        spatial = pd.read_csv(f2)

        N     = len(spatial)
        ai_10 = sim['p10'].values
        ai    = sim['p50'].values
        ai_90 = sim['p90'].values

        Ni = (ai/ai.sum())*N
                
        C2 = (Ni*(1-alpha2))/(Mx**(1-alpha2) - Mlim**(1-alpha2))
        C1 = C2*Mc**(alpha1 - alpha2)
        
        # Ncorrs
        Ncorrs = []
        for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni):
            if mlim!=0:
                M = np.linspace(Ml,Mc,10000)  
                Nlim1 = np.trapz(c1*M**(-alpha1), M)
    
                M = np.linspace(Mc,mlim,10000)  
                Nlim2 = np.trapz(c2*M**(-alpha2), M)
    
                Ncorr = ni + Nlim1 + Nlim2
            else:
                Ncorr = 0
            Ncorrs.append(Ncorr)
        Ncorrs = np.array(Ncorrs)
         
        # Ncorrs high
        Ncorrs_hi = []
        Ni_hi = Ni + (ai_90-ai)*Ni 
    
        for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni_hi):
            if mlim!=0:
                M = np.linspace(Ml,Mc,10000)  
                Nlim1 = np.trapz(c1*M**(-alpha1), M)
    
                M = np.linspace(Mc,mlim,10000)  
                Nlim2 = np.trapz(c2*M**(-alpha2), M)
    
                Ncorr = (ni + np.sqrt(ni))*(1 + (Nlim1 + Nlim2)/ni)
            else:
                Ncorr = 0
            Ncorrs_hi.append(Ncorr)
            
        Ncorrs_hi = np.array(Ncorrs_hi)
    
        # Ncorrs low
        Ncorrs_lo = []
        Ni_lo = Ni - (ai-ai_10)*Ni
        for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni_lo):
            if mlim!=0:
                M = np.linspace(Ml,Mc,10000)  
                Nlim1 = np.trapz(c1*M**(-alpha1), M)
    
                M = np.linspace(Mc,mlim,10000)  
                Nlim2 = np.trapz(c2*M**(-alpha2), M)
    
                Ncorr = (ni - np.sqrt(ni))*(1 + (Nlim1 + Nlim2)/ni)
            else:
                Ncorr = 0
            Ncorrs_lo.append(Ncorr)
            
        Ncorrs_lo = np.array(Ncorrs_lo)
        
        Ncorrs_reco.append(Ncorrs)
        Ncorrs_reco_hi.append(Ncorrs_hi)
        Ncorrs_reco_lo.append(Ncorrs_lo)

        ind_age = age_cen<=8.1
        ax.step(age_cen[ind_age], Ncorrs[ind_age]/1e6, where='mid', color='red', lw=1.2)

    Ncorrs_reco    = np.array(Ncorrs_reco)
    Ncorrs_reco_hi = np.array(Ncorrs_reco_hi)
    Ncorrs_reco_lo = np.array(Ncorrs_reco_lo)
    
    Ncorrs_in = np.load(f'../SFH/ncorrs_{met_gas}_{met_trgb}.npy')
        
    corr_vals    = np.percentile(Ncorrs_in/Ncorrs_reco,                          [16,50,84], axis=0) 
    corr_vals_hi = np.percentile(Ncorrs_in/Ncorrs_reco_hi - Ncorrs_in/Ncorrs_reco,    50, axis=0)
    corr_vals_lo = np.percentile(Ncorrs_in/Ncorrs_reco    - Ncorrs_in/Ncorrs_reco_lo, 50, axis=0) 

    # Median corr
    corr = corr_vals[1]

    # Error in corr
    corr_errs_high1 = corr_vals[2] -  corr_vals[1]
    corr_errs_high  = np.sqrt(corr_errs_high1**2 + corr_vals_hi**2)

    corr_errs_low1 = corr_vals[1] -  corr_vals[0]
    corr_errs_low  = np.sqrt(corr_errs_low1**2 + corr_vals_lo**2) 

    bubble_corrs.append(corr)
    bubble_corr_hi.append(corr_errs_high)
    bubble_corr_lo.append(corr_errs_low)

    ax.step(age_cen[ind_age], Ncorrs_in[ind_age]/1e6, where='mid', color='black',lw =3)

    #ax.set_yscale('log')
    ax.set_ylabel(r'${\rm N_i + N_{lim,i}}$ [$\times10^6$]')
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
                   bottom = True, left = True)
    ax.tick_params(which='major', length=15,direction="in")
    ax.tick_params(which='minor', length=8, color='black',direction="in")
    ax.minorticks_on()
    
    ax = axs[1]
    
    ax.step(age_cen[ind_age], corr[ind_age], where='mid', label = 'Simulated', color='black', lw=2)
    ax.errorbar(age_cen[ind_age], corr[ind_age], yerr = [corr_errs_low[ind_age], corr_errs_high[ind_age]],
               fmt='o', color='blue', markersize=1, capsize=3, lw=2)
    ax.set_xlabel(r'log (Age [yr])')
    ax.set_ylabel(r'${\rm \xi_i}$')
   # ax.set_yscale('log')

    ax.axhline(1, lw=2, linestyle='--', color='black', alpha=0.8)

    ax.axhline(1.5, lw=2, linestyle='--', color='red', alpha=0.8)
    
    ax.set_ylim(-1, 3)
    #ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(1.0, 10.0)*0.1, numticks=10))
    ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
                   bottom = True, left = True)
    ax.tick_params(which='major', length=15,direction="in")
    ax.tick_params(which='minor', length=8, color='black',direction="in")
    #ax.minorticks_on()
    
    plt.subplots_adjust(hspace=0)

    #fig.savefig(f'../SFH/ngc628/JWST_bubble_sim/figures/region_{p}.png',bbox_inches='tight')
    #plt.close(fig)

bubble_corrs   = np.array(bubble_corrs)
bubble_corr_hi = np.array(bubble_corr_hi)
bubble_corr_lo = np.array(bubble_corr_lo)

In [ ]:
np.save('../SFH/ngc628/bubble_corrs.npy', bubble_corrs) 
np.save('../SFH/ngc628/bubble_corrs_hi.npy', bubble_corr_hi) 
np.save('../SFH/ngc628/bubble_corrs_lo.npy', bubble_corr_lo) 

In [ ]:
bubble_corrs =  np.load('../SFH/ngc628/bubble_corrs.npy') 

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))

ages = np.arange(6.7,10.01,0.1)
ages_r = ages+0.05
ages_l = ages-0.05

#ages_l[0] = 6

age_cen = 0.5*(ages_l + ages_r)
delta_t = 10**ages_r- 10**ages_l

ind_age = age_cen<=8.0
for corr in bubble_corrs:

    ax.step(age_cen[ind_age], corr[ind_age], where='mid', label = 'Simulated', color='black', alpha=0.3,
           lw=2)
    # ax.errorbar(age_cen, corr, yerr = [corr_errs_low, corr_errs_high],
    #            fmt='o', color='blue', markersize=0.5, capsize=2)

ax.step(age_cen[ind_age], bubble_corrs[27][ind_age], where='mid', label = 'Simulated', color='blue', alpha=1,
       lw=3)
    
ax.set_xlabel(r'log (Age [yr])')
ax.set_ylabel(r'${\rm \xi_i}$')
ax.set_yscale('log')

ax.axhline(1, lw=2, linestyle='--', color='black', alpha=0.8)

ax.axhline(1.5, lw=2, linestyle='--', color='red', alpha=0.8)


#ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(1.0, 10.0)*0.1, numticks=10))
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")
ax.minorticks_on()

## **Obs SFR**

In [ ]:
p        = 0

cut_ages = [8]*90

SFRs_reg = []

SFRs_reg_hi = []
SFRs_reg_lo = []

corrs        = np.load('../SFH/ngc628/reg_corrs.npy')
corrs_err_hi = np.load('../SFH/ngc628/reg_corrs_hi.npy')
corrs_err_lo = np.load('../SFH/ngc628/reg_corrs_lo.npy')

Av     =  0.19
AF115  =  0.419*Av
AF150  =  0.287*Av
AF200  =  0.195*Av
dismod =  29.81

for p in range(90):
    # Precomputed corr for the given region from simulations
    corr = corrs[p]*0 + 1
    corr_hi = corrs_err_hi[p]*0
    corr_lo = corrs_err_lo[p]*0 


    # SFR Ncorr for the given region
    fws         = [f115w_m50.ravel()[p], f150w_m50.ravel()[p], f200w_m50.ravel()[p]]
    obs_sfh     = pd.read_csv(f'../SFH/ngc628/JWST_grid_obs/reg_{p}_SFH.csv')
    obs_spatial = pd.read_csv(f'../SFH/ngc628/JWST_grid_obs/reg_{p}_spatial.csv')

    N = len(obs_spatial)
    
    ai_10 = obs_sfh['p10'].values
    ai    = obs_sfh['p50'].values
    ai_90 = obs_sfh['p90'].values
    
    Ni = (ai/ai.sum())*N
    
    alpha1 = 1.3
    alpha2 = 2.3
    Mc = 0.5
    Ml = 0.1
    Mu = [350]
    
    FW1 = fws[0] - AF115 - dismod
    FW2 = fws[1] - AF150 - dismod
    FW3 = fws[2] - AF200 - dismod

    met_gas, met_trgb = met_map_muse.ravel()[p], met_map_trgb.ravel()[p]
    print(met_gas, met_trgb)
    if np.isnan(met_gas):
        met_gas = 0.014
    else:
        mets = np.array([0.014,0.017,0.02])
        mets_diffs = abs(mets - met_gas)
        ind = mets_diffs == mets_diffs.min()
        met_gas = mets[ind][0]

    if np.isnan(met_trgb):
        met_trgb = 0.002  
    else:
        mets = np.array([0.001, 0.002, 0.003])
        mets_diffs = abs(mets - met_trgb)
        ind = mets_diffs == mets_diffs.min()
        met_trgb = mets[ind][0]

    os.system(f'rm ../data/isochrones/JWST_temp/*')

    os.system(f'cp ../data/isochrones/JWST_{met_gas}/* ../data/isochrones/JWST_temp/')
    os.system(f'cp ../data/isochrones/JWST_{met_trgb}/* ../data/isochrones/JWST_temp/')

    isos = glob.glob(f'../data/isochrones/JWST_temp/*')
    sort_isos(isos)
    
    out_dir = '../SFH/ngc628/JWST_dummy'
  
    sfh = BaySFH(df_filt, parallel=True, isodir='../data/isochrones/JWST_temp/', 
              fw1_lim=fws[0],fw2_lim=fws[1], fw3_lim=fws[2],
              sig_fw1=sig_i, sig_fw2=sig_i, sig_fw3=sig_i,
              A_fw1=AF115, A_fw2=AF150, A_fw3=AF200, 
             dismod=dismod, out_dir=out_dir+'output/',
             n_cores=6)
    
    Mx = np.array([i[1].max() for i in sfh.Iso])
    Mlim = []
    for i in sfh.Iso:
        j = i[1][(i[2]<=FW1) & (i[3]<=FW2) & (i[4]<=FW3)]
        if len(j)>0:
            Mlim.append(j.min())
        else:
            Mlim.append(0)
    Mlim = np.array(Mlim)
    
    C2 = (Ni*(1-alpha2))/(Mx**(1-alpha2) - Mlim**(1-alpha2))
    C1 = C2*Mc**(alpha1 - alpha2)
    
    # Ncorrs
    Ncorrs = []
    for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni):
        if mlim!=0:
            M = np.linspace(Ml,Mc,10000)  
            Nlim1 = np.trapz(c1*M**(-alpha1), M)

            M = np.linspace(Mc,mlim,10000)  
            Nlim2 = np.trapz(c2*M**(-alpha2), M)

            Ncorr = ni + Nlim1 + Nlim2
        else:
            Ncorr = 0
        Ncorrs.append(Ncorr)
    Ncorrs = np.array(Ncorrs)*corr
    
    # Ncorrs high
    Ncorrs_hi = []
    Ni_hi = Ni + (ai_90-ai)*Ni 
    Ni_hi = Ni_hi/Ni_hi.sum()*N
    for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni_hi):
        if mlim!=0:
            M = np.linspace(Ml,Mc,10000)  
            Nlim1 = np.trapz(c1*M**(-alpha1), M)

            M = np.linspace(Mc,mlim,10000)  
            Nlim2 = np.trapz(c2*M**(-alpha2), M)

            Ncorr = (ni + np.sqrt(ni))*(1 + (Nlim1 + Nlim2)/(ni + np.sqrt(ni)))
        else:
            Ncorr = 0
        Ncorrs_hi.append(Ncorr)
        
    Ncorrs_hi = np.array(Ncorrs_hi) * (corr + corr_hi)

    # Ncorrs low
    Ncorrs_lo = []
    Ni_lo = Ni - (ai-ai_10)*Ni
    Ni_lo = N*Ni_lo/Ni_lo.sum()
    
    for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni_lo):
        if mlim!=0:
            M = np.linspace(Ml,Mc,10000)  
            Nlim1 = np.trapz(c1*M**(-alpha1), M)

            M = np.linspace(Mc,mlim,10000)  
            Nlim2 = np.trapz(c2*M**(-alpha2), M)

            Ncorr = (ni - np.sqrt(ni))*(1 + (Nlim1 + Nlim2)/(ni - np.sqrt(ni)))
        else:
            Ncorr = 0
        Ncorrs_lo.append(Ncorr)
        
    Ncorrs_lo = np.array(Ncorrs_lo) * (corr - corr_lo)
    
    # Mean Mass
    h2 = (Mx**(1.-alpha2)-Mc**(1.-alpha2))/(1.-alpha2)
    h2 *= Mc**(alpha2-alpha1)
    h1 = (Mc**(1.-alpha1)-Ml**(1.-alpha1))/(1.-alpha1)

    C1 = 1/(h1+h2)
    C2 = C1*Mc**(alpha2-alpha1)

    Mean_M = []
    for c1, c2, mu in zip(C1,C2,Mu):
        M = np.linspace(Ml,Mc,10000)  
        Mean_M1 = np.trapz(M*c1*M**(-alpha1), M)

        M = np.linspace(Mc,mu,10000)  
        Mean_M2 = np.trapz(M*c2*M**(-alpha2), M)
        
        Mean_M.append(Mean_M1+Mean_M2)
      
    Mean_M = np.array(Mean_M)

    ax         = axs[0]  
    ages       = obs_sfh['Log_age'].values
    ages_r     = ages+0.05
    ages_l     = ages-0.05 
    #ages_l[0] = 6
    
    delta_t    = 10**ages_r - 10**ages_l

    age_cen    = 0.5*(ages_r + ages_l)
    
    SFRs = (Mean_M*Ncorrs)/delta_t

    SFRs_hi = (Mean_M*(Ncorrs_hi-Ncorrs))/delta_t
    SFRs_lo = (Mean_M*(Ncorrs-Ncorrs_lo))/delta_t

    SFRs_lo = np.where(SFRs_lo<0, 1e3,SFRs_lo)
     
    SFRs_reg.append(SFRs)
    SFRs_reg_hi.append(SFRs_hi*corr)
    SFRs_reg_lo.append(SFRs_lo*corr)

    ages_cen = 0.5*(ages_l + ages_r)
    
    x   = ages_cen
    y   = SFRs
    ind = x<= cut_ages[p]
    
    fig, ax = plt.subplots(figsize=(15,7))
    ax.step(age_cen, SFRs*1e3, where='mid',  color='blue', lw=3)
    ax.errorbar(age_cen, SFRs*1e3, yerr = [SFRs_lo*1e3, SFRs_hi*1e3],
               fmt='o', color='black', markersize=0.5, capsize=3, lw=2)

    ax.set_yscale('log')
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(LogLocator(subs='all'))
    
    ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
                   bottom = True, left = True)
    ax.tick_params(which='major', length=15,direction="in")
    ax.tick_params(which='minor', length=8, color='black',direction="in")

    ax.set_xlabel(r'${\rm log(Age[yr^{-1}])}$')
    ax.set_ylabel(r'SFR $\times 10^{-3}{M_{\odot}.yr^{-1}}$')
    
    #fig.savefig(f"../SFH/ngc628/JWST_grid_obs/figures/reg_{p}.png", bbox_inches='tight')
    plt.close(fig)


In [ ]:
SFRs_reg    = np.array(SFRs_reg)
SFRs_reg_hi = np.array(SFRs_reg_hi)
SFRs_reg_lo = np.array(SFRs_reg_lo)

In [ ]:
np.save('../SFH/ngc628/SFR_reg.npy', SFRs_reg)
np.save('../SFH/ngc628/SFR_reg_hi.npy', SFRs_reg_hi)
np.save('../SFH/ngc628/SFR_reg_lo.npy', SFRs_reg_lo)

In [ ]:
grid = np.arange(0,30).reshape(6,5)

In [ ]:
SFRs_reg    = np.load("../SFH/ngc628/SFR_reg.npy")
SFRs_reg_hi = np.load("../SFH/ngc628/SFR_reg_hi.npy")
SFRs_reg_lo = np.load("../SFH/ngc628/SFR_reg_lo.npy")

In [ ]:
cut_ages = np.array([7.6 , 7.6, 7.4, 7.6, 7.6, 7.6, 8. , 7.6, 7.6, 7.6, 7.7, 8. , 7.9,
                   7.4, 7.6, 7.8, 7.6, 7.8, 7.7, 7.5, 7.5, 7.7, 7.4, 8. , 7.6, 7.5,
                   7.7, 8. , 7.6, 7.6])


cut_ages_n = array([7.6 , 7.6, 7.4, 7.4, 7.6, 7.4, 7.8, 7.5, 7.6, 7.6, 7.4, 7.7, 7.8,
       7.4, 7.5, 7.8, 7.6, 7.5, 7.7, 7.5, 7.5, 7.7, 7.4, 7.8, 7.6, 7.5,
       7.7, 8. , 7.5, 7.6])

In [ ]:
cut_ages = cut_ages_n 

In [ ]:
coords = []
for i in range(90):
    coords.append([regions_dict[f'reg_{i}']['ra'], regions_dict[f'reg_{i}']['dec']])
coords = np.array(coords)

r = angular_separation(regions_dict['ngc628']['ra']*u.deg, regions_dict['ngc628']['dec']*u.deg,
                       coords[:,0]*u.deg, coords[:,1]*u.deg).to(u.arcsec).value

t_25    = []
t_25_hi = []
t_25_lo = []

t_50 = []
t_50_hi = []
t_50_lo = []

t_75 = []
t_75_hi = []
t_75_lo = []

log_M = []

tab = Table.read("../photometry/ngc628/f115w_f150w_f200w_photometry.fits")
df_filt = jwst_data(tab, 0, 0, 0, 0,  region_type='ellipse',a=1,b=1)
sig_i = 0.01

corrs        = np.load('../SFH/ngc628/bubble_corrs.npy')
corrs_err_hi = np.load('../SFH/ngc628/bubble_corrs_hi.npy')
corrs_err_lo = np.load('../SFH/ngc628/bubble_corrs_lo.npy')


Bubble_SFHs    = []
Bubble_SFHs_uncorr    = []
Bubble_SFHs_hi = []
Bubble_SFHs_lo = []
total_mass_s   = []
total_mass_lo  = []
total_mass_hi  = []
cum_mass_s     = [] 

bubble_mets    = []

Av     =  0.19
AF115  =  0.419*Av
AF150  =  0.287*Av
AF200  =  0.195*Av
dismod =  29.81

for p in range(len(df_m74_bubbles_s1)):
    corr    = corrs[p]
    corr_hi = corrs_err_hi[p]
    corr_lo = corrs_err_lo[p]
    
    fws = [f115w_m50_bubbles[p], f150w_m50_bubbles[p], f200w_m50_bubbles[p]]
   
    met_gas, met_trgb = met_map_muse_bubble.ravel()[p], met_map_trgb_bubble.ravel()[p]

    print(met_gas, met_trgb)
    if np.isnan(met_gas):
        met_gas = 0.014
    else:
        mets       = np.array([0.014,0.017,0.02])
        mets_diffs = abs(mets - met_gas)
        ind        = mets_diffs == mets_diffs.min()
        met_gas    = mets[ind][0]

    if np.isnan(met_trgb):
        met_trgb = 0.002  
    else:
        mets       = np.array([0.001, 0.002, 0.003])
        mets_diffs = abs(mets - met_trgb)
        ind        = mets_diffs == mets_diffs.min()
        met_trgb   = mets[ind][0]


    obs_sfh     = pd.read_csv(f'../SFH/ngc628/JWST_bubble_obs/{p}_SFH.csv')
    obs_spatial = pd.read_csv(f'../SFH/ngc628/JWST_bubble_obs/{p}_spatial.csv')

    N = len(obs_spatial)
    
    ai_10 = obs_sfh['p10'].values
    ai    = obs_sfh['p50'].values
    ai_90 = obs_sfh['p90'].values
    
    Ni    = (ai/ai.sum())*N
    #Ni = resample_Ni(Ni)
    
    alpha1 = 1.3
    alpha2 = 2.3
    Mc = 0.5
    Ml = 0.1
    Mu = [350]
    
    FW1 = fws[0] - AF115 - dismod
    FW2 = fws[1] - AF150 - dismod
    FW3 = fws[2] - AF200 - dismod

    os.system(f'rm ../data/isochrones/JWST_temp/*')

    os.system(f'cp ../data/isochrones/JWST_{met_gas}/* ../data/isochrones/JWST_temp/')
    os.system(f'cp ../data/isochrones/JWST_{met_trgb}/* ../data/isochrones/JWST_temp/')

    isos = glob.glob(f'../data/isochrones/JWST_temp/*')
    sort_isos(isos)
    
    out_dir = '../SFH/ngc628/JWST_dummy'
    
    os.makedirs(out_dir,exist_ok=True)
    os.makedirs(out_dir + 'output/',exist_ok=True)
    os.makedirs(out_dir + 'plots/',exist_ok=True)
    
    sfh = BaySFH(df_filt, parallel=True, isodir='../data/isochrones/JWST_temp/', 
              fw1_lim=fws[0],fw2_lim=fws[1], fw3_lim=fws[2],
              sig_fw1=sig_i, sig_fw2=sig_i, sig_fw3=sig_i,
              A_fw1=AF115, A_fw2=AF150, A_fw3=AF200, 
             dismod=dismod, out_dir=out_dir+'output/',
             n_cores=6)
    
    Mx = np.array([i[1].max() for i in sfh.Iso])
    Mlim = []
    for i in sfh.Iso:
        j = i[1][(i[2]<=FW1) & (i[3]<=FW2) & (i[4]<=FW3)]
        if len(j)>0:
            Mlim.append(j.min())
        else:
            Mlim.append(0)
    Mlim = np.array(Mlim)
    
    C2 = (Ni*(1-alpha2))/(Mx**(1-alpha2) - Mlim**(1-alpha2))
    C1 = C2*Mc**(alpha1 - alpha2)
    
    # Ncorrs
    Ncorrs = []
    for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni):
        if mlim!=0:
            M = np.linspace(Ml,Mc,10000)  
            Nlim1 = np.trapz(c1*M**(-alpha1), M)

            M = np.linspace(Mc,mlim,10000)  
            Nlim2 = np.trapz(c2*M**(-alpha2), M)

            Ncorr = ni + Nlim1 + Nlim2
        else:
            Ncorr = 0
        Ncorrs.append(Ncorr)
    Ncorrs = np.array(Ncorrs)
    
    # Ncorrs high
    Ncorrs_hi = []
    Ni_hi = Ni + (ai_90-ai)*Ni
    Ni_hi = (Ni_hi/Ni_hi.sum())*N

    for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni_hi):
        if mlim!=0:
            M = np.linspace(Ml,Mc,10000)  
            Nlim1 = np.trapz(c1*M**(-alpha1), M)

            M = np.linspace(Mc,mlim,10000)  
            Nlim2 = np.trapz(c2*M**(-alpha2), M)

            Ncorr = (ni + np.sqrt(ni))*(1 + (Nlim1 + Nlim2)/(ni + np.sqrt(ni)))
        else:
            Ncorr = 0
        Ncorrs_hi.append(Ncorr)
        
    Ncorrs_hi = np.array(Ncorrs_hi)

    # Ncorrs low
    Ncorrs_lo = []
    Ni_lo = Ni - (ai-ai_10)*Ni
    Ni_lo = (Ni_lo/Ni_lo.sum())*N
    
    for c1, c2, mlim, ni in zip(C1,C2,Mlim, Ni_lo):
        if mlim!=0:
            M = np.linspace(Ml,Mc,10000)  
            Nlim1 = np.trapz(c1*M**(-alpha1), M)

            M = np.linspace(Mc,mlim,10000)  
            Nlim2 = np.trapz(c2*M**(-alpha2), M)

            Ncorr = (ni - np.sqrt(ni))*(1 + (Nlim1 + Nlim2)/(ni - np.sqrt(ni)))
        else:
            Ncorr = 0
        Ncorrs_lo.append(Ncorr)
        
    Ncorrs_lo = np.array(Ncorrs_lo)
    
    # Mean Mass
    h2 = (Mx**(1.-alpha2)-Mc**(1.-alpha2))/(1.-alpha2)
    h2 *= Mc**(alpha2-alpha1)
    h1 = (Mc**(1.-alpha1)-Ml**(1.-alpha1))/(1.-alpha1)

    C1 = 1/(h1+h2)
    C2 = C1*Mc**(alpha2-alpha1)

    Mean_M = []
    for c1, c2, mu in zip(C1,C2,Mu):
        M = np.linspace(Ml,Mc)  
        Mean_M1 = np.trapz(M*c1*M**(-alpha1), M)

        M = np.linspace(Mc,mu)  
        Mean_M2 = np.trapz(M*c2*M**(-alpha2), M)
        
        Mean_M.append(Mean_M1+Mean_M2)
      
    Mean_M = np.array(Mean_M)
    
    ages = obs_sfh['Log_age'].values
    ages_r = ages+0.05
    ages_l = ages-0.05
    
    #ages_l[0] = 6
    
    delta_t = 10**ages_r- 10**ages_l

    age_cen = 0.5*(ages_r + ages_l)
    
    SFRs = (Mean_M*Ncorrs)/delta_t

    SFRs_hi = (Mean_M*(Ncorrs_hi-Ncorrs))/delta_t
    SFRs_lo = (Mean_M*(Ncorrs-Ncorrs_lo))/delta_t
     
    x = age_cen
    y = SFRs

    # Subjacent disk
    ind = ages <= 8
    
    a = df_m74_bubbles_s1['a'].values[p]
    b = df_m74_bubbles_s1['b'].values[p]
    ra = df_m74_bubbles_s1['ra'].values[p]
    dec = df_m74_bubbles_s1['dec'].values[p]
    bub_r = np.sqrt(a*b)
    bub_area = np.pi*a*b
    reg_area = (24*24)
    
    bub_gal_r = df_m74_bubbles_s1['gal_r'].values[p]
    
    r_dist = angular_separation(coords[:,0]*u.deg, coords[:,1]*u.deg, ra*u.deg, dec*u.deg).to(u.arcsec).value
    
    ind = (r-bub_gal_r<12*np.sqrt(2)) & (r-bub_gal_r>-12*np.sqrt(2)) & (r_dist>bub_r+12*np.sqrt(2))

    if np.all(ind)==False:
        ind = (r-bub_gal_r<12*np.sqrt(2)) & (r-bub_gal_r>-12*np.sqrt(2)) 
    ind_grid = ind
    SFH_sub = np.array(SFRs_reg[ind])
    ages = np.arange(6.7,10.01,0.1)
    ages_r = ages+0.05
    ages_l = ages-0.05
    
    age_cen = 0.5*(ages_l + ages_r)
    delta_t = 10**ages_r- 10**ages_l
    
    SFH_sf_med  = np.median(SFH_sub, axis=0)
    SFH_sf_perc = np.percentile(SFH_sub, [16,84], axis=0)
    SFH_sf_low  = SFH_sf_med - SFH_sf_perc[0] 
    SFH_sf_high = SFH_sf_perc[1] - SFH_sf_med
    
    if not np.isnan(cut_ages[p]):

        ind_corr = (corr>1.5)
        ind_corr[0] = False
        corr_age = age_cen[ind_corr]
        
        if corr_age[0]<cut_ages[p]:
            cut_ages[p] = corr_age[0]

        ind = age_cen <= cut_ages[p]
        
        x_n = x[ind]  
        y_n = SFRs[ind]

        SFRs_diff    = (SFRs/bub_area - SFH_sf_med/reg_area)*corr*bub_area
        SFRs_diff[SFRs_diff<0] = 0
        SFRs_diff_hi = np.sqrt((SFRs_hi/bub_area)**2 + (SFH_sf_high/reg_area)**2)*(corr+corr_hi)*bub_area 
        SFRs_diff_lo = np.sqrt((SFRs_lo/bub_area)**2 + (SFH_sf_low/reg_area)**2)*(corr-corr_lo)*bub_area 

        Bubble_SFHs.append(SFRs_diff)  
        Bubble_SFHs_hi.append(SFRs_diff_hi)
        Bubble_SFHs_lo.append(SFRs_diff_lo)
        
        dm = SFRs_diff[ind][::-1]*delta_t[ind][::-1]

        # Cumulative mass formed
        cum_mass = np.cumsum(dm)
        cum_mass_s.append(cum_mass)
        # Total mass formed
        total_mass = cum_mass[-1]

        total_mass_s.append(total_mass)

        # Find age at which x% of mass was formed
        # t 25
        age_25 = np.interp(0.25*total_mass, cum_mass, age_cen[ind][::-1])

        dm = (SFRs_diff + SFRs_diff_hi)[ind][::-1]* delta_t[ind][::-1]
        cum_mass = np.cumsum(dm)
        total_mass = cum_mass[-1]
        age_25_hi = np.interp(0.25*total_mass, cum_mass, ages_r[ind][::-1])
        total_mass_hi.append(total_mass)

        dm = (SFRs_diff - SFRs_diff_lo)[ind][::-1]* delta_t[ind][::-1]
        cum_mass = np.cumsum(dm)
        total_mass = cum_mass[-1]
        age_25_lo = np.interp(0.25*total_mass, cum_mass, ages_l[ind][::-1])
        total_mass_lo.append(total_mass)

        # t50
        age_50 = np.interp(0.5*total_mass, cum_mass, age_cen[ind][::-1])

        dm = (SFRs_diff + SFRs_diff_hi)[ind][::-1]* delta_t[ind][::-1]
        cum_mass = np.cumsum(dm)
        total_mass = cum_mass[-1]
        age_50_hi = np.interp(0.5*total_mass, cum_mass, ages_r[ind][::-1])

        dm = (SFRs_diff - SFRs_diff_lo)[ind][::-1]* delta_t[ind][::-1]
        cum_mass = np.cumsum(dm)
        total_mass = cum_mass[-1]
        age_50_lo = np.interp(0.5*total_mass, cum_mass, ages_l[ind][::-1])

        # t75
        age_75 = np.interp(0.75*total_mass, cum_mass, age_cen[ind][::-1])

        dm = (SFRs_diff + SFRs_diff_hi)[ind][::-1]* delta_t[ind][::-1]
        cum_mass = np.cumsum(dm)
        total_mass = cum_mass[-1]
        age_75_hi = np.interp(0.75*total_mass, cum_mass, ages_r[ind][::-1])

        dm = (SFRs_diff - SFRs_diff_lo)[ind][::-1]* delta_t[ind][::-1]
        cum_mass = np.cumsum(dm)
        total_mass = cum_mass[-1]
        age_75_lo = np.interp(0.75*total_mass, cum_mass, ages_l[ind][::-1])

        t_25.append(age_25)
        t_25_hi.append(age_25_hi)
        t_25_lo.append(age_25_lo)
        
        t_50.append(age_50)
        t_50_hi.append(age_50_hi)
        t_50_lo.append(age_50_lo)
        
        t_75.append(age_75)
        t_75_hi.append(age_75_hi)
        t_75_lo.append(age_75_lo)

    else:
        t_25.append(np.nan)
        t_25_hi.append(np.nan)
        t_25_lo.append(np.nan)
        
        t_50.append(np.nan)
        t_50_hi.append(np.nan)
        t_50_lo.append(np.nan)
        
        t_75.append(np.nan)
        t_75_hi.append(np.nan)
        t_75_lo.append(np.nan)

        cum_mass_s.append([np.nan]*len(age_cen))
        total_mass_s.append(np.nan)

t_25 = np.array(t_25)
t_25_hi = np.array(t_25_hi)
t_25_lo = np.array(t_25_lo)

t_50 = np.array(t_50)
t_50_hi = np.array(t_50_hi)
t_50_lo = np.array(t_50_lo)

t_75 = np.array(t_75)
t_75_hi = np.array(t_75_hi)
t_75_lo = np.array(t_75_lo)

total_mass   = np.array(total_mass_s)
total_mass_hi = np.array(total_mass_hi)
total_mass_lo = np.array(total_mass_lo)

bubble_mets = np.array(bubble_mets)

In [ ]:
fig, ax = plt.subplots()
ax.imshow(ind_grid.reshape(15,6).T, extent=(-7.5*24,7.5*24, -3*24,3*24))
p = plt.Circle((0,0), radius = bub_gal_r, fill=False, lw=2)
ax.add_patch(p)

bub_grid = (r_dist==r_dist.min()).reshape(15,6).T
bub_grid = np.where(bub_grid,1,np.nan)
#ax.imshow(bub_grid,  cmap='grey', extent=(-7.5*24,7.5*24, -3*24,3*24))

In [ ]:
ages   = np.arange(6.7,10.01, 0.1)

ages_r = ages + 0.05
ages_l = ages - 0.05

#ages_l[0] = 6

delta_t = 10**ages_r- 10**ages_l

age_cen = 0.5*(ages_r + ages_l)

In [ ]:
corrs        = np.load('../SFH/ngc628/bubble_corrs.npy')

count = 0

xi_15_ages = []
for corr in corrs:
    corr_age = age_cen[(age_cen>=7.) &(age_cen<=9.0) & (corr>1.5)][0]
    xi_15_ages.append(np.round(corr_age,1))

xi_15_ages = np.array(xi_15_ages)

In [ ]:
for i in range(len(cum_mass_s)):
    np.save(f"../SFH/ngc628/cum_mass_bubble_{i}.npy", cum_mass_s[i])
    
np.save("../SFH/ngc628/Bubble_SFHs.npy", Bubble_SFHs)
np.save("../SFH/ngc628/Bubble_SFH_hi.npy", Bubble_SFHs_hi)
np.save("../SFH/ngc628/Bubble_SFH_lo.npy", Bubble_SFHs_lo)

np.save("../SFH/ngc628/t_25.npy", t_25)
np.save("../SFH/ngc628/t_50.npy", t_50)
np.save("../SFH/ngc628/t_75.npy", t_75)
np.save("../SFH/ngc628/total_mass.npy", total_mass_s)

np.save("../SFH/ngc628/t_25_hi.npy", t_25_hi)
np.save("../SFH/ngc628/t_50_hi.npy", t_50_hi)
np.save("../SFH/ngc628/t_75_hi.npy", t_75_hi)
np.save("../SFH/ngc628/total_mass_hi.npy", total_mass_hi)

np.save("../SFH/ngc628/t_25_lo.npy", t_25_lo)
np.save("../SFH/ngc628/t_50_lo.npy", t_50_lo)
np.save("../SFH/ngc628/t_75_lo.npy", t_75_lo)
np.save("../SFH/ngc628/total_mass_lo.npy", total_mass_lo)

In [ ]:
Bubble_SFHs    = np.load("../SFH/ngc628/Bubble_SFHs.npy")
Bubble_SFHs_uncorr    = np.load("../SFH/ngc628/Bubble_SFHs_uncorr.npy")
Bubble_SFHs_hi = np.load("../SFH/ngc628/Bubble_SFH_hi.npy")
Bubble_SFHs_lo = np.load("../SFH/ngc628/Bubble_SFH_lo.npy")

t_25 = np.load("../SFH/ngc628/t_25.npy")
t_50 = np.load("../SFH/ngc628/t_50.npy")
t_75 = np.load("../SFH/ngc628/t_75.npy")
total_mass = np.load("../SFH/ngc628/total_mass.npy")

t_25_hi = np.load("../SFH/ngc628/t_25_hi.npy")
t_50_hi = np.load("../SFH/ngc628/t_50_hi.npy")
t_75_hi = np.load("../SFH/ngc628/t_75_hi.npy")
total_mass_hi = np.load("../SFH/ngc628/total_mass_hi.npy")

t_25_lo = np.load("../SFH/ngc628/t_25_lo.npy")
t_50_lo = np.load("../SFH/ngc628/t_50_lo.npy")
t_75_lo = np.load("../SFH/ngc628/t_75_lo.npy")
total_mass_lo = np.load("../SFH/ngc628/total_mass_lo.npy")

cum_mass_s = []
for i in range(len(df_m74_bubbles_s1)):
    cum_mass = np.load(f"../SFH/ngc628/cum_mass_bubble_{i}.npy")
    cum_mass_s.append(cum_mass)

In [ ]:
reg_ids = np.arange(0,90)
coords= []
for i in range(90):
    coords.append([regions_dict[f'reg_{i}']['ra'], regions_dict[f'reg_{i}']['dec']])
coords = np.array(coords)

r = angular_separation(regions_dict['ngc628']['ra']*u.deg, regions_dict['ngc628']['dec']*u.deg,
                       coords[:,0]*u.deg, coords[:,1]*u.deg).to(u.arcsec).value
grid = np.arange(0,90)

In [ ]:
int_mass = total_mass

r = df_m74_bubbles_s1['r']
r_ind = np.argsort(r)[::-1].values

corrs        = np.load('../SFH/ngc628/bubble_corrs.npy')[r_ind]

m50_f200w = f200w_m50_bubbles.ravel()[r_ind]
df_m74_bubbles_s1['t0']    = cut_ages
df_m74_bubbles_s1['t25']   = t_25
df_m74_bubbles_s1['t50']   = t_50
df_m74_bubbles_s1['t75']   = t_75
df_m74_bubbles_s1['Mstar'] = np.log10(int_mass)
df_m74_bubbles_s1['Mstar_hi'] = np.log10(total_mass_hi - total_mass)
df_m74_bubbles_s1['Mstar_lo'] = np.log10(total_mass - total_mass_lo)

for i, row in df_m74_bubbles_s1.sort_values('Bubble ID').reset_index().iterrows():
    r = angular_separation(row['ra']*u.deg, row['dec']*u.deg,
                           coords[:,0]*u.deg, coords[:,1]*u.deg).to(u.arcsec)
    id_ = row['Bubble ID']
    t0   = np.round(10**row['t0']/1e6,1)
    t25  = np.round(10**row['t25']/1e6,1)
    t50  = np.round(10**row['t50']/1e6,1)
    t75  = np.round(10**row['t75']/1e6,1)
    mass = np.round(row['Mstar'],1)
    nstars = row['N_total']
    m50 = m50_f200w[i]
    print(f"""{id_} &  {grid[r==r.min()][0]} & {10**(xi_15_ages[r_ind][i]-6):0.1f} & {m50:0.1f} & {t0} & {t25} & {t50} & {t75} & {mass} & {nstars} \\\\""")

In [ ]:
df_m74_bubbles_s1.to_csv(
    "../data/DS9 regions/m74_bubbles_s1.dat",
    sep=" ",
    index=False
)

In [ ]:
SFR_25s = []
SFR_75s = []
for i in range(len(Bubble_SFHs)):

    SFR_25 = (10**cut_ages[i]- 10**t_25[i])
    SFR_75 = (10**t_25[i]- 10**t_75[i])/2

    SFR_25s.append(SFR_25)
    SFR_75s.append(SFR_75)
        
SFR_25s = np.array(SFR_25s)
SFR_75s = np.array(SFR_75s)
SFR_ratio = np.array(SFR_25s/SFR_75s)

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
x = SFR_25s/1e6
y = SFR_75s/1e6
ax.scatter(x, y, s=200, edgecolor='black', facecolor='grey', zorder=200)

ax.scatter(x[1],y[1],s=400, zorder=100,color='blue', edgecolor='black')
ax.scatter(x[27],y[27],s=400, zorder=100,color='red', edgecolor='black')

l = np.linspace(SFR_75s.min()/1e6, SFR_75s.max()/1e6)

ax.plot(l, l, ls='--', lw=2, color='black')
ax.set_xlabel(r't$_0$ - t$_{25}$ [Myr]')
ax.set_ylabel(r't$_{25}$ - t$_{75}$ [Myr]')

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")

In [ ]:
y[1]/x[1]

In [ ]:
fig, axs = plt.subplots(1,2, figsize=(14,7), sharey=True, width_ratios=[1,0.3])
#bins = np.linspace(0,1.4,9)

a = df_m74_bubbles_s1['a']
b = df_m74_bubbles_s1['b']
r = np.sqrt(a*b).values*44

ax = axs[0]
ax.scatter(r,SFR_ratio,s=200, zorder=200, color='grey', edgecolor='black')

ax.scatter(r[1],SFR_ratio[1],s=400, zorder=100,color='blue', edgecolor='blue')
ax.scatter(r[27],SFR_ratio[27],s=400, zorder=100,color='red', edgecolor='red')

ax.set_ylabel(r'SFR$_{25}$/SFR$_{75}$')
ax.set_xlabel(r'${\rm R_{Bubble}}$')

ax.set_xlim(40,1.5e3)
#ax.set_ylim(0,2)

ax.set_xscale('log')
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")

ax.tick_params(axis='x', pad=10)
ax.axhline(1, lw=2, linestyle='--', color='red', alpha=0.8)

ax = axs[1]
ax.hist(SFR_ratio, bins=16, color='grey', edgecolor='black', orientation='horizontal')

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")

ax.set_xlabel(r'{\rm No\,\,of\,\,Bubbles}', fontsize=25)
plt.subplots_adjust(wspace=0)

In [ ]:
r = df_m74_bubbles_s1['r'].values
r_ind = np.argsort(r)[::-1]

df    = df_m74_bubbles_s1.sort_values('r', ascending=False).reset_index()
corrs = np.load('../SFH/ngc628/bubble_corrs.npy')[r_ind]

grid_ids = []

for i in range(len(df)):
    bubble_id = df['Bubble ID'].values[i]
    
    a = df['a'].values[i]
    b = df['b'].values[i]
    bub_area = np.pi*a*b*(44e-3)**2
    reg_area = (24*24)*(44e-3)**2

    fig, ax = plt.subplots(1,1, figsize=(18,10))
    
    #i      = 30
    ages   = np.arange(6.7,10.01, 0.1)
    ages_r = ages+0.05
    ages_l = ages-0.05
    
    #ages_l[0] = 6.0
    
    delta_t = 10**ages_r- 10**ages_l
    
    age_cen = 0.5*(ages_r + ages_l)
    
    x       = age_cen
    y_bub   = 1e3*Bubble_SFHs[r_ind][i]/bub_area

    y_max   = y_bub[age_cen<=8].max()
    yerr_lo =  np.abs(1e3*Bubble_SFHs_lo[r_ind][i]/bub_area)
    yerr_hi =  np.abs(1e3*Bubble_SFHs_hi[r_ind][i]/bub_area)


    x = [6,6.65,6.7] 
    y = [0,1e3*SFR_bubble_muse[r_ind][i]/bub_area,
         1e3*Bubble_SFHs[r_ind][i][0]/bub_area]

    ax.step(x,y,where='pre', color='green', linestyle='--', lw=5)
    # ax.bar(x,y, width = 0.7, color = 'green', align='edge',lw=5, zorder=200, fill=False, 
    #       edgecolor='green', bottom =None)

    
    
    x = [6,6.65,6.7] 
    y = [0,1e3*SFR_bubble_muse_shell[r_ind][i]/(0.44*bub_area), 
         1e3*Bubble_SFHs[r_ind][i][0]/bub_area]
    ind = age_cen<=8

        
    ax.step(x,y,where='pre', color='cyan', linestyle='--', lw=5)
    
    ax.set_ylabel(r'$\Sigma_{SFR}\,\rm[10^{-3}.M_{\odot}.yr^{-1}.kpc^{-2}]$') 
    
    ax.axvline(t_25[r_ind][i], ymax =1, color='red', linestyle='--', lw=3, label=r'$t_{25}$')
    ax.axvline(t_50[r_ind][i], ymax =1, color='green', linestyle='--', lw=3, label=r'$t_{50}$')
    ax.axvline(t_75[r_ind][i], ymax =1, color='blue', linestyle='--', lw=3, label=r'$t_{75}$')
    ax.axvline(cut_ages[r_ind][i], ymax =1, color='black', linestyle='--', lw=3)
    
    ax.step(age_cen, y_bub, color='blue', lw=3, where='mid')

    ind_n = corrs[i]>=1.5

    ind_n[0]=False
    ax.fill_betweenx([-1,y_max*1.2], age_cen[ind_n][0],8.1, color='yellow', alpha=0.4)

    ax.errorbar(age_cen, y_bub,lw=2,
                yerr = [yerr_lo, yerr_hi],
                fmt='o', color = 'red', markersize=0.5, capsize=4, alpha=0.6)
    
    
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
                   bottom = True, left = True)
    ax.tick_params(which='major', length=15,direction="in")
    ax.tick_params(which='minor', length=8, color='black',direction="in")
    
    ax.set_xlabel('log(Age [yr])')
    #ax.set_ylim(0,)
    
    plt.subplots_adjust(hspace=0)
    #ax.set_yscale('log')
    
    ax1 = ax.twinx()
    ax1.set_ylabel(r'f$_M$') 
    ax1.yaxis.set_minor_locator(AutoMinorLocator())
    ax1.tick_params(which='both', width=3,direction="in", right = True)
    ax1.tick_params(which='major', length=15,direction="in")
    ax1.tick_params(which='minor', length=8, color='black',direction="in")
    
    ind = age_cen <= cut_ages[r_ind][i]
    x   = age_cen[ind][::-1]
    
    ages = np.arange(6.7,10.01,0.1)
    ages_r = ages + 0.05
    ages_l = ages - 0.05
    
    age_cen = 0.5*(ages_l + ages_r)
    delta_t = 10**ages_r - 10**ages_l
    
    dm = (Bubble_SFHs[r_ind][i]*delta_t)[ind]
    y = np.cumsum(dm[::-1])

    if ~np.isnan(cut_ages[r_ind][i]):
        
        ax1.annotate(r"t$_0=$ " + f"{10**cut_ages[r_ind][i]/1e6:.0f} Myr", (6.,0.9), fontsize=35)
        ax1.annotate(r"M$_*=$ " + f"{y[-1]:.1e}" +r" M$_{\odot}$", (6.,0.8),
                    fontsize=40)

        ax1.plot(x,y/y.max(), lw=6, color='magenta')

    ax1.set_ylim(0.0,1.05)
    ax.set_ylim(-1,y_max*1.2)
    ax.set_xlim(None, 8.05)

    fig.savefig(f"../SFH/ngc628/JWST_bubble_obs/figures/bubble_{i}.png")
    plt.close(fig)
    bubble_id = df['Bubble ID'].values[i]
    #print(bubble_id)
    #ax.legend(fontsize=2w5)

In [ ]:
ages_cen = np.round(np.arange(6.7,8.01,0.1),1)
df = df_m74_bubbles_s2[df_m74_bubbles_s2['n50']>0]
#df = df.drop(columns=['level_0'])
for i,row in df.sort_values('r', ascending=False).reset_index().iterrows():

    ra  = row['ra']
    dec = row['dec']
    a   = row['a']
    b   = row['b']
    ang = row['ang']
    df_filt = ellipse(tab_spatial,'RA','DEC',ra,dec,ang, 0,0, a/3600,b/3600)
    df_filt = df_filt[df_filt['Log_Age']<=8.5]

    x = np.round(df_filt.to_pandas().groupby(['Log_Age']).count().index.values,1)
    y = np.round(df_filt.to_pandas().groupby(['Log_Age']).count()['Z'].values,1)

    n_stars  = np.zeros_like(ages_cen).astype(float)
    for j,age in enumerate(x):
        n_stars[ages_cen==age] = y[j]
        
    fig, ax = plt.subplots(1,1, figsize=(18,10))
    ax.step(ages_cen,n_stars, lw =5,color='blue', where='mid', zorder=300)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
                   bottom = True, left = True)
    ax.tick_params(which='major', length=15,direction="in")
    ax.tick_params(which='minor', length=8, color='black',direction="in")

    ax.set_ylabel(r'No. of stars')
    ax.set_xlabel(r'log(Age [yr])')
    
    #ax.set_ylim(0,)
    
    fig.savefig(f"../SFH/ngc628/JWST_bubble_obs/figures2/bubble_{i+30}.png", bbox_inches='tight')
    
    plt.close(fig)
    #ax.legend(fontsize=2w5)

In [ ]:
annot_coords = {20:[0,2],
                13: [5,0],
                27: [0,1],
                 24: [-5,1]}

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(14,8), sharey=True, width_ratios=(1,0.3))

r = np.sqrt(df_m74_bubbles_s1['a'].values*df_m74_bubbles_s1['b'].values)
gal_r = df_m74_bubbles_s1['gal_r']*44e-3

age_stack = np.tile(r*44, (3,1))
t_stack = np.stack([t_25, t_50, t_75], axis=0)

#ax.plot(age_stack, t_stack, '--k')
r = np.sqrt(df_m74_bubbles_s1['a'].values*df_m74_bubbles_s1['b'].values)
area = np.pi*df_m74_bubbles_s1['a'].values*df_m74_bubbles_s1['b'].values*(44e-3)**2

x = r*44
y = 10**t_25/1e6

ax =axs[0]

logm = np.log10(int_mass)
logm_min = logm[~(np.isnan(logm) | np.isinf(logm))].min()
logm_max = logm[~(np.isnan(logm) | np.isinf(logm))].max()

for i,j,m in zip(x,y,int_mass):
    img = ax.scatter(i, j, c= np.log10(m), s=200, marker='o', cmap='jet', edgecolor='black', zorder=300,
                     vmin= logm_min, vmax= logm_max)
    
yerr = np.abs(np.array([10**t_25_hi-10**t_25, 10**t_25-10**t_25_lo])/1e6)

ax.errorbar(x,y,yerr=yerr,  fmt='o', color = 'red', markersize=0.5, capsize=4, alpha=0.7)

r = np.sqrt(df_m74_bubbles_s1['a'].values*df_m74_bubbles_s1['b'].values)
ids = df_m74_bubbles_s1['Bubble ID'].values
r_ind = np.argsort(r)[::-1]
x = r[r_ind]*44
y = 10**t_25[r_ind]/1e6
ids = ids[r_ind]
for t,x_,y_ in zip(ids, x,y):

    if t in annot_coords.keys():
        x_ -= annot_coords[t][0]
        y_ -= annot_coords[t][1]
    ax.annotate(r'$\textbf{'+str(t) + '}$', (x_*0.97 ,y_-5), fontsize=15,fontweight='bold')
    
#ax.scatter(x, y, color='red', label=r"_nolegend_")

y = 10**np.linspace(6.8,8.05)

x = (5/3)*10*y/1e6

ax.plot(x,10**np.log10(y)/1e6, color='blue', label='10km/s', zorder=100)

x = (5/3)*5*y/1e6

ax.plot(x,10**np.log10(y)/1e6, color='green', label='5km/s', zorder=100)

x = (5/3)*1*y/1e6

ax.plot(x,10**np.log10(y)/1e6, color='red', label='1km/s', zorder=100)


ax.set_ylabel(r't$_{25}~$[Myr]')
ax.set_xlabel(r'R$_{\rm B}~$[pc]', labelpad=-2)

ax.grid()
#ax.set_ylim(5,85)
ax.set_xlim(50,1.3e3)
ax.set_ylim(5,105)
ax.set_xscale('log')

#ax.xaxis.set_minor_locator(AutoMinorLocator())
#ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")

ax.tick_params(axis='x', pad=10)
ax = axs[1]

ind_class = df_m74_bubble_sub['class']==1
ax.hist(10**t_25[ind_class]/1e6, bins=9,range=(10**t_25.min()/1e6,100),orientation='horizontal', color='grey', edgecolor='black')
ax.axhline(np.nanmedian(10**t_25[ind_class]/1e6), xmax=10, linestyle='--', color='black', lw=3)
#ax.legend(fontsize=20)
ax.annotate(f"""{np.round(np.nanmedian(10**t_25[ind_class]/1e6),0)} Myr""",(0.4, 36), color='black')

cax = fig.add_axes([0.124, -0.07, 0.778, 0.03])  
# [left, bottom, width, height] in figure fraction

cb = plt.colorbar(img, cax=cax, orientation='horizontal')
cb.set_label(r'log${\rm (M_{*}/M_{\odot})}$', fontsize=25, labelpad=10)
cb.ax.tick_params(labelsize=20, pad=5)

ax.set_xlabel('No of Bubbles', fontsize=25)
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")

#ax.set_ylim(None,80)

plt.subplots_adjust(wspace=0)

#ax.legend(fontsize=25)

In [ ]:
tab_spatial = Table.from_pandas(pd.read_csv("../data/NGC628_spatial.csv"))

In [ ]:
age_median = []

for i, row in df_m74_bubbles_s2.iterrows():
    a = row['a']
    b = row['b']
    ra = row['ra']
    dec = row['dec']
    ang = row['ang']
    
    df_filt = ellipse(tab_spatial,'RA','DEC',ra,dec,ang, 0,0,a/3600,b/3600)
    df_filt = df_filt[df_filt['Log_Age']<=7.7]

    age_median.append(np.nanpercentile(df_filt['Log_Age'],50))
    
age_median = np.array(age_median)

In [ ]:
len(age_median[~np.isnan(age_median)])

In [ ]:
c1 = np.log10(5e40) # Starburst 99, Mayya 2023
c2 = np.log10(78.5e34) # Derived by Dr. Divakara

In [ ]:
c1-c2 + np.log10(0.05)

In [ ]:
fig, ax  = plt.subplots(1,1,figsize=(15,6), sharey=True)

df = df_m74_bubbles_s1

r = np.sqrt(df['a'].values*df['b'].values)

r = np.sqrt(df['a'].values*df['b'].values)
area = np.pi*df['a'].values*df['b'].values*(44e-3)**2

x = np.log10(r*44/100)
y = np.log10(int_mass/1e6) +  3*(t_25 - 7)
c = 10**(t_25 - 6)
img = ax.scatter(x,y, c=c, cmap='jet', s=60, zorder=200, edgecolor='black')

ax.scatter(x[1],y[1], zorder=100,  s=180, facecolor= (0,0,0,0), edgecolor='red', lw=2)
ax.scatter(x[27],y[27], zorder=100,  s=180, facecolor= (0,0,0,0), edgecolor='blue', lw=2)

y_up = np.log10(total_mass_hi/1e6) +  3*(t_25_hi - 7)
y_do = np.log10(total_mass_lo/1e6) +  3*(t_25_lo - 7)

y_hi = y_up-y
y_lo = y - y_do

#ax.errorbar(x,y,yerr=[y_lo,y_hi], fmt='o', color = 'red', markersize=0.5, capsize=4)
cb = plt.colorbar(img, ax=ax)
cb.set_label(r't$_{25}$ [Myr]')
bins  = np.linspace(-.5, 1,6)
med_mass = []
for i in range(len(bins)-1):
    ind = (x>bins[i]) & (x<=bins[1+i])
    med_mass.append(np.nanmedian(y[ind]))

bin_cen = 0.5*(bins[1:] + bins[:-1])

#ax.plot(bin_cen, med_mass, '--k', lw=3)

ind = ~np.isnan(y)

init   = models.Linear1D()
fit    = fitting.LevMarLSQFitter()
model = fit(init, x[ind], y[ind])

x = np.linspace(-0.9, x.max())

n_dense_0 = 1
eta_0   = 0.05

c_00 = -(c1-c2)+ np.log10(n_dense_0) - np.log10(eta_0)

ax.plot(x,  5*x + c_00 ,  linestyle='dashed', color='green', lw=3)
ax.annotate(r'$n=1~cm^{-3}$, $\eta = 0.05$', (0.25,-3.55), color='green', rotation = 18)

eta     = 0.01
n_dense = 50000

c_10 = c_00 + np.log10(n_dense/n_dense_0) - np.log10(eta/eta_0)

ax.plot(x,   5*x + c_10 ,  linestyle='dashed', color='blue', lw=3)

ax.annotate(r'$n=5\times10^{4}~cm^{-3}$, $\eta = 0.01$', (0.18,1.5), color='blue', rotation = 18)


#ax.plot(x,model(x), linestyle='dashed', color='red', lw=3, label = 'Fit')

#======================================
t = age_median
r = np.sqrt(df_m74_bubbles_s2['a'].values*df_m74_bubbles_s2['b'].values)
x = np.log10(r*44/100)
y = np.log10(800/1e6) +  3*(age_median - 7)
c = 10**(age_median - 6)

ax.scatter(x,y, c=c, marker= "v", cmap='jet', s=60, edgecolor='black')

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")
ax.set_xlabel(r'log (R$_{\rm B}/$100 pc)')
ax.set_ylabel(r'log (M$_*/{10^6M_\odot}$) + 3 log (t$_{25}$/10~Myr)', fontsize=20)
#ax.set_ylim(None,80)

ax.grid()

ax.legend(fontsize=15)

#ax.set_ylim(None, 5)
#ax.legend(fontsize=25)

In [ ]:
c_00

In [ ]:
import matplotlib.gridspec as gridspec

In [ ]:
fig, ax  = plt.subplots(figsize=(15, 8))  # adjust size as needed

# # Define the grid: 1 row, 2 columns
# gs = gridspec.GridSpec(1, 2, width_ratios=[1, 0.3], wspace=0.1, hspace=0)

# # Left column: 1 subplot
# ax = fig.add_subplot(gs[0, 0])

# # Right column: subdivide into 3 rows
# gs_right = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=gs[0, 1], hspace=0.2)


# ax_right_top = fig.add_subplot(gs_right[0])
# ax_right_mid = fig.add_subplot(gs_right[1])
# ax_right_bottom = fig.add_subplot(gs_right[2])


# for a in [ax_right_top, ax_right_mid, ax_right_bottom]:
#     a.yaxis.set_label_position('right')
#     a.set_ylabel('SFR', fontsize=15)
#     a.set_xticks([])
#     a.set_yticks([])

ages = np.arange(6.7,10.01,0.1)
ages_r = ages + 0.05
ages_l = ages - 0.05
t0 = cut_ages
age_cen = 0.5*(ages_r + ages_l)

ts = []
cmfs = []
for i, cum_mass in enumerate(cum_mass_s):
    if df_m74_bubble_sub['class'].values[i]==0:
        continue
    if np.all(np.isnan(cum_mass)):
        continue
    ind = age_cen <= cut_ages[i]
    
    t = 10**t0[i] - 10**age_cen[ind][::-1]

    x = (t - np.nanmin(t))/(np.nanmax(t) -  np.nanmin(t))
    y = cum_mass/cum_mass[-1]

    cmf_n = np.arange(0,1,0.1)
    xn = np.interp(cmf_n,y,x)

    ts.append(xn)
    cmfs.append(cmf_n)

    if i!=1:
        ax.plot(x,y, alpha=0.7, label="_nolegend_")
    else:
        ax.plot(x,y, lw=5, alpha=1, label="_nolegend_")


all_x = np.concatenate(ts)
all_y = np.concatenate(cmfs)

sb.kdeplot(x=all_x, y=all_y, fill=True, cmap='coolwarm', thresh=0.15, bw_adjust=1, levels=50, alpha=0.8, ax=ax, cbar=True,
          cbar_kws = {'format': '%.1f', 'ticks':[1,2,3]})

t0_ = 7.6
t = np.arange(7, t0_ + 0.05,0.1)

tau = 1.0e7
SFR = np.exp(-(10**t0_ - 10**t[:-1][::-1])/tau)
delta_t = 10**t[1:] - 10**t[:-1]
cum_mass = np.cumsum(SFR*delta_t[::-1])

x = 10**t0_ - 10**t[::-1][1:]
y = cum_mass/cum_mass[-1]
x = (x - x[0])/(x[-1] -  x[0])

ax.plot(x,y, '--', color='red', lw=3, label=r'exp($-(t_0-t)/\tau$)')

delta_t = 10**t[1:] - 10**t[:-1]
SFR = 10 
cum_mass = np.cumsum(SFR*delta_t[::-1])

x = 10**t0_ - 10**t[::-1][1:]
y = cum_mass/cum_mass[-1]
x = (x - x[0])/(x[-1] -  x[0])

ax.plot(x,y, '--', color='black', lw=3, label=f'Constant SFR')

tau = 1.e7

SFR = np.exp((10**t0_ - 10**t[:-1][::-1])/tau)

delta_t = 10**t[1:] - 10**t[:-1]

cum_mass = np.cumsum(SFR*delta_t[::-1])

x = 10**t0_ - 10**t[::-1][1:]
y = cum_mass/cum_mass[-1]
x = (x - x[0])/(x[-1] -  x[0])

ax.plot(x,y, '--', color='blue', lw=3, label=r'exp($(t_0-t)/\tau$)')

ax.set_xlabel(r'Time~$_{\rm norm}$') 
ax.set_ylabel(r'${\rm f}_{\rm M}$') 

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")
ax.legend(fontsize=30)

ax.set_xlim(0.,1)
ax.set_ylim(-0.05,1)
ax.grid(True)
fig.savefig("CMFs.png", bbox_inches='tight')

In [ ]:
yerr

In [ ]:
fig, ax = plt.subplots(figsize=(14,6))
r = np.sqrt(df_m74_bubble_sub['a'].values*df_m74_bubble_sub['b'].values)
x = (r*44)
y = total_mass
yerr = [(total_mass_hi-total_mass), (total_mass - total_mass_lo)]

ax.errorbar(x,y,yerr=yerr,  fmt='o', color = 'red', markersize=0.5, capsize=4, zorder=100, alpha=0.6)        
c = t_25
ax.scatter(x,y, color='gray', s=200, label='_nolegend_', edgecolor='black', zorder=200)

ax.scatter(x[1], y[1],  s =300, color='red', facecolor=(0,0,0,0), edgecolor='red',lw=3, zorder=200)

ax.scatter(x[27], y[27],  s =300, color='blue', facecolor=(0,0,0,0), edgecolor='blue',lw=3, zorder=200)


bins  = np.linspace(20,820,10)
med_mass = []
for i in range(len(bins)-1):
    ind = (x>bins[i]) & (x<=bins[1+i])
    med_mass.append(np.median(y[ind]))

bin_cen = 0.5*(bins[1:] + bins[:-1])

ax.plot(bin_cen, med_mass, '--k', lw=3)

t = age_median
x = np.sqrt(df_m74_bubbles_s2['a'].values*df_m74_bubbles_s2['b'].values)*44
y = np.log10(1e3)  + x*0
c = 10**(age_median - 6)

#ax.scatter(x,y, c=c, marker= "v", cmap='jet', s=100, edgecolor='black')
y  = 10**np.linspace(4,6)

x  = y/5e6

#ax.plot(x, np.log10(y), color='blue', lw=2, label=r'$5$~M$_{\odot}.pc^{-2}$')

x  = y/1e6

#ax.plot(x, np.log10(y), color='green', lw=2, label=r'$1$~M$_{\odot}.pc^{-2}$')

x  = y/(2e5)

#ax.plot(x, np.log10(y), color='red', lw=2, label=r'$0.2$~M$_{\odot}.pc^{-2}$')

# cb = plt.colorbar(img, ax=ax)

# cb.set_label(r't$_{25}~$[Myr]', fontsize=20)
# cb.ax.tick_params(labelsize=15)
#ax.set_ylim(4,6)
ax.set_xlabel(r'R$_{\rm B}$~[pc]')

ax.set_ylabel(r'log${\rm (M_*}/$M$_{\odot})$') 

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")

ax.grid(True, which='major')

ax.set_xlim(50,1e3)
ax.tick_params(axis='x', pad=10)
ax.set_xscale('log')
ax.legend(fontsize=20)

ax.set_yscale('log')

In [ ]:
fig, ax = plt.subplots(figsize=(14,6))

r = np.sqrt(df_m74_bubbles_s1['a'].values*df_m74_bubbles_s1['b'].values)
y = (3/5)*r*44/10**(t_25-6)
x = 10**t_25/1e6

xerr = np.abs(np.array([10**t_25_hi-10**t_25, 10**t_25-10**t_25_lo])/1e6)
yerr = np.abs(np.array([(r*44)/10**t_25_hi-(r*44)/10**t_25 ,(r*44)/10**t_25-(r*44)/10**t_25_lo]))*1e6

ax.errorbar(x,y, xerr=xerr, yerr=yerr,  fmt='o', color = 'red', markersize=0.5, capsize=4, zorder=100, alpha=0.6)   

img = ax.scatter(x,y, s=200,  color='grey', label='_nolegend_', edgecolor='black', zorder=200)
ax.scatter(x[1], y[1],  s =300, color='red', facecolor=(0,0,0,0), edgecolor='red',lw=3, zorder=200)

ax.scatter(x[-3], y[-3],  s =300, color='blue', facecolor=(0,0,0,0), edgecolor='blue',lw=3, zorder=200)

bins  = np.arange(0,95,22)
med_mass = []
for i in range(len(bins)-1):
    ind = (x>bins[i]) & (x<=bins[1+i])
    med_mass.append(np.median(y[ind]))

bin_cen = 0.5*(bins[1:] + bins[:-1])

ax.plot(bin_cen[:-1], med_mass[:-1], '--k', lw=3)


y = np.log10(1e3)  + age_median*0
x = 10**(age_median - 6)
#ax.scatter(x,y, s=100,  marker='v',color='red', label='_nolegend_', edgecolor='black')

ax.set_xlabel(r't$_{25}$ [Myr]')

ax.set_ylabel(r'v$_{\rm exp}$ [km/s]') 

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")
ax.grid(True)

#ax.set_yscale('log')
ax.legend(fontsize=20)

In [ ]:
(u.pc/u.Myr).to(u.km/u.s)

In [ ]:
fig, ax = plt.subplots(figsize=(14,6))

y = np.log10(int_mass)
x = 10**t_25/1e6

xerr = np.abs(np.array([10**t_25_hi-10**t_25, 10**t_25-10**t_25_lo])/1e6)
yerr = [np.log10(total_mass_hi)-np.log10(total_mass), np.log10(total_mass) - np.log10(total_mass_lo)]

ax.errorbar(x,y, xerr=xerr, yerr=yerr,  fmt='o', color = 'red', markersize=0.5, capsize=4, zorder=100, alpha=0.6)   
img = ax.scatter(x,y, s=200,  color='grey', label='_nolegend_', edgecolor='black', zorder=200)
ax.scatter(x[1], y[1],  s =300, color='red', facecolor=(0,0,0,0), edgecolor='red',lw=3, zorder=200)

ax.scatter(x[-3], y[-3],  s =300, color='blue', facecolor=(0,0,0,0), edgecolor='blue',lw=3, zorder=200)

bins  = np.arange(0,95,22)
med_mass = []
for i in range(len(bins)-1):
    ind = (x>bins[i]) & (x<=bins[1+i])
    med_mass.append(np.median(y[ind]))

bin_cen = 0.5*(bins[1:] + bins[:-1])

ax.plot(bin_cen, med_mass, '--k', lw=3)


y = np.log10(1e3)  + age_median*0
x = 10**(age_median - 6)
#ax.scatter(x,y, s=100,  marker='v',color='red', label='_nolegend_', edgecolor='black')

ax.set_xlabel(r't$_{25}$ [Myr]')

ax.set_ylabel(r'{log(M$_{*}/{\rm M_{\odot}}$)') 

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")
ax.grid(True)

#ax.set_yscale('log')
ax.legend(fontsize=20)

In [ ]:
bubble_clust_t50

In [ ]:
bubble_clust_t75

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))

y = np.log10(int_mass)
x = 10**t_25/1e6


for i, t in zip(clust_Bubble_IDs, bubble_clust_t50):
    if i in df_m74_bubbles_s1['Bubble ID'].values:
        x = df_m74_bubbles_s1[df_m74_bubbles_s1['Bubble ID']==i]['t25']
        y = t
        ax.scatter(x,y, s=100,  color='grey', label='_nolegend_', edgecolor='black')

x = np.arange(7,8,0.1)

ax.plot(x,x, '--r', lw=3)

#ax.set_xlim(7,8)
#ax.set_ylim(6,8)
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")
ax.grid(True)

ax.set_xlabel(r'Stellar t$_{25}$ [Myr]')
ax.set_ylabel(r'Cluster t$_{25}$ [Myr]')

In [ ]:
sfh_obs = glob.glob("../SFH/ngc628/JWST_grid_obs/*SFH.csv")
sfh_spatial = glob.glob("../SFH/ngc628/JWST_grid_obs/*spatial.csv")


In [ ]:
label = []
tab = Table.read("../photometry/ngc628/f115w_f150w_f200w_photometry.fits")
df_filt = jwst_data(tab, 0, 0, 0, 0,  region_type='ellipse',a=1,b=1)
sig_i = 0.01

dfs = []
for p, f in enumerate(sfh_spatial):
    df = pd.read_csv(f)
    df_spatial = pd.read_csv(f)
    met_gas, met_trgb = met_map_muse.ravel()[p], met_map_trgb.ravel()[p]
    print(met_gas, met_trgb)
    if np.isnan(met_gas):
        met_gas = 0.014
    else:
        mets = np.array([0.014,0.017,0.02])
        mets_diffs = abs(mets - met_gas)
        ind = mets_diffs == mets_diffs.min()
        met_gas = mets[ind][0]

    if np.isnan(met_trgb):
        met_trgb = 0.002  
    else:
        mets = np.array([0.001, 0.002, 0.003])
        mets_diffs = abs(mets - met_trgb)
        ind = mets_diffs == mets_diffs.min()
        met_trgb = mets[ind][0]

    print(met_gas, met_trgb)
    os.system(f'rm ../data/isochrones/JWST_temp/*')

    os.system(f'cp ../data/isochrones/JWST_{met_gas}/* ../data/isochrones/JWST_temp/')
    os.system(f'cp ../data/isochrones/JWST_{met_trgb}/* ../data/isochrones/JWST_temp/')

    isos = glob.glob(f'../data/isochrones/JWST_temp/*')
    sort_isos(isos)
    
    out_dir = '../SFH/ngc628/JWST_dummy'
    
    os.makedirs(out_dir,exist_ok=True)
    os.makedirs(out_dir + 'output/',exist_ok=True)
    os.makedirs(out_dir + 'plots/',exist_ok=True)
    
    sfh = BaySFH(df_filt, parallel=True, isodir='../data/isochrones/JWST_temp/', 
              fw1_lim=30,fw2_lim=30, fw3_lim=30,
              sig_fw1=sig_i, sig_fw2=sig_i, sig_fw3=sig_i,
              A_fw1=0., A_fw2=0, A_fw3=0, 
             dismod=29.81, out_dir=out_dir+'output/',
             n_cores=6)
    label = []
    Mini_s = []
    ages = sfh.ages
    for i, row in df_spatial.iterrows():
        age_ind = np.where(ages==np.round(row['Log_Age'],1))[0]

        ls = sfh.Iso[age_ind][0][0,:]
        Mini = sfh.Iso[age_ind][0][1,:]
        f115w_iso = sfh.Iso[age_ind][0][2,:]
        f150w_iso = sfh.Iso[age_ind][0][3,:]
        f200w_iso = sfh.Iso[age_ind][0][4,:]
        phase_prob = sfh.Iso[age_ind][0][7,:]
        
        f115w_obs = row['mag_vega_F115W']
        f150w_obs = row['mag_vega_F150W']
        f200w_obs = row['mag_vega_F200W']
        
        d_3d = np.sqrt( (f115w_iso - f115w_obs)**2 + (f150w_iso - f150w_obs)**2 + (f200w_iso - f200w_obs)**2)
        label.append(ls[np.where(d_3d==d_3d.min())[0]][0])
        Mini_s.append(Mini[np.where(d_3d==d_3d.min())[0]][0])
        
    df_spatial['label'] = label
    df_spatial['Mini'] = Mini_s
    df_spatial['Z'] = met_gas
    df_spatial.loc[df_spatial['Log_Age']>=9,'Z'] = met_trgb
    dfs.append(df_spatial)

In [ ]:
df_spatial_ = pd.concat(dfs)

In [ ]:
df_spatial_.to_csv("../data/NGC628_spatial.csv")

In [ ]:
df_spatial = pd.read_csv("../data/NGC628_spatial.csv")

In [ ]:
df_cmd_jwst_n = pd.read_csv('../data/isochrones_master/cmd_jwst_corr.csv')

In [ ]:
sfh_spatial = glob.glob("../SFH/ngc628/JWST_grid_obs/*spatial.csv")

In [ ]:
tab.keys()

In [ ]:
len(df_spatial)

In [ ]:
bubble_index

In [ ]:
df_spatial = pd.read_csv("../data/NGC628_spatial.csv")

In [ ]:
ra = df_spatial['RA']
dec = df_spatial['DEC']

plt.scatter(ra, dec, s=0.01)

In [ ]:
df_spatial = pd.read_csv("../data/NGC628_spatial.csv")

tab = Table.from_pandas(df_spatial)
tab['mag_vega_F115W'] += 29.81 + Av_dict['f115w']*0.19
tab['mag_vega_F150W'] += 29.81 + Av_dict['f150w']*0.19
tab['mag_vega_F200W'] += 29.81 + Av_dict['f200w']*0.19

r = df_m74_bubbles_s1['r']
r_ind = np.argsort(r)[::-1]

bubble_index = np.arange(0,len(r))[r_ind]

df = df_m74_bubbles_s2[df_m74_bubbles_s2['n50']>0].sort_values('r', ascending=False).reset_index()

#df = df_m74_bubbles_s1.sort_values('r', ascending=False).reset_index()
for i in range(len(df)):

    fig, ax = plt.subplots(figsize=(12,12))
    ra_cen  = df['ra'].values[i]
    dec_cen = df['dec'].values[i]
    
    filters = {'filt1':'f115w',
               'filt2':'f200w',
               'filt3':'f200w'}
    
    positions = {'ra_col' : 'RA',
                 'dec_col': 'DEC',
                 'ra_cen' : ra_cen,
                 'dec_cen': dec_cen}
    
    region = {'a1': 0,
              'b1': 0,
              'a2': df['a'].values[i],
              'b2': df['b'].values[i],
              'ang': df['ang'].values[i],
              'spatial_filter': 'ellipse'}
    # if r>=40:
    #     continue
    extinction = {'Av'  : 0.19,
                  'Av_x': 0.5,
                  'Av_y': 19,
                  'Av_' : 1}
    
    axis_limits= {'xlims': [-0.4, 2.], 
                  'ylims': [18, 26.5]}
    
    isochrone_params={'met': [0.02],
                      'label_min': 0,
                      'label_max': 10,
                      'ages': [ 7, 7.7,]}
    
    error_settings = {'ref_xpos': -0.25,
                      'mag_err_lim':0.2}
    
    plot_settings = {'s':30, 'legend.ncols':2, 'alpha':0.7, 'lw':3,
                    'Av.fontsize':25}
    
    kde_contours = {'gen_kde' :False}
    df_temp = df_cmd_jwst.copy()
    df_temp = df_temp[((df_temp['logAge']==6.7) & (df_temp['label']<6)) | (df_temp['logAge']>6.7)]
    try:
        fig,ax, tab1 = gen_CMD(tab, 
                              None,
                              filters, 
                              positions,
                              region,
                              extinction,
                              29.81,
                              axis_limits,
                              isochrone_params,
                              plot_settings=plot_settings,
                              error_settings=error_settings,
                              other_settings={'ab_dist': False, 'skip_data': False},
                              kde_contours=kde_contours,
                              fig=fig, ax=ax)
    except:
        continue
        
    tab1 = tab1[tab1['Log_Age']<=7.7]
    median_age = np.round(np.median(tab1['Log_Age']),1)
    isochrone_params['ages'] = [median_age]
    isochrone_params['met'] = [0.02]
    
    fig,ax, tab1 = gen_CMD(tab, 
                          df_cmd_jwst,
                          filters, 
                          positions,
                          region,
                          extinction,
                          29.81,
                          axis_limits,
                          isochrone_params,
                          plot_settings=plot_settings,
                          error_settings=error_settings,
                          other_settings={'ab_dist': True, 'skip_data': True},
                          fig=fig, ax=ax)
    
    x = tab1['mag_vega_F115W'] - tab1['mag_vega_F200W']
    y = tab1['mag_vega_F200W']
    c = tab1['Log_Age']
    lg = ax.legend(fontsize=25)
    #lg.remove()
    
    if len(tab1)<1000:
        s=100
    else:
        s=30
        
    img = ax.scatter(x,y,c=c, s=s, cmap='jet' , vmin=6.7, vmax=10, zorder=100, alpha=1, edgecolor='black')
    cb = plt.colorbar(img, ax=ax, orientation='horizontal',pad = 0.1,)
    cb.set_label(r'log(Age [yr])', fontsize=30)

    fig.savefig(f'../SFH/ngc628/JWST_bubble_obs/figures2/bubble_{i+30}_cmd.png', bbox_inches='tight')
    plt.close(fig)
print("Completed!")

In [ ]:
bubble_index[i]

In [ ]:
10**1.7

In [ ]:
xs = []
ys = []
rs = []
circ_radii = []
for region, data in df_m74_bubble_sub.iterrows():

    ra  = data['ra']
    dec = data['dec']
    a   = data['a']
    b   = data['b']
    q = b/a
    ang = data['ang']
    
    df_temp = Table.from_pandas(df_spatial.copy())
    
    df_temp  = ellipse(df_temp,'RA','DEC', ra, dec, ang, 0, 0, 1.2*a/3600, 1.2*b/3600)
    
    df_temp = df_temp[df_temp['Log_Age']<=t_25[region]]
    
    stars  = SkyCoord(df_temp['RA']*u.deg, df_temp['DEC']*u.deg)
    center = SkyCoord(ra*u.deg, dec*u.deg)
    dra, ddec = stars.spherical_offsets_to(center)  # returns Angles
    x_east  = dra.to(u.arcsec).value
    y_north = ddec.to(u.arcsec).value

    # rotate into ellipse frame
    pa_rad = np.radians(ang)
    xprime = x_east*np.sin(pa_rad) + y_north*np.cos(pa_rad)
    yprime = -x_east*np.cos(pa_rad) + y_north*np.sin(pa_rad)

    # equivalent semi-major/minor axes
    a_eq = np.sqrt(xprime**2 + (yprime/q)**2)
    b_eq = q * a_eq
    rs += [np.sqrt(a*b)]*len(xprime)
    xs += list(np.sqrt(a_eq*b_eq))
    ys += list(df_temp['Log_Age'])

xs = np.array(xs)
ys = np.array(ys)
rs = np.array(rs)

circ_radii = np.array(circ_radii)

In [ ]:
from astropy.coordinates import SkyCoord, AltAz, SkyOffsetFrame

In [ ]:
x = df_spatial['RA']
y = df_spatial['DEC']
c = df_spatial['Log_Age']
plt.scatter(x,y,c=c, cmap='jet',s=0.1)

In [ ]:
df_m74_bubble_sub2['reff'] = np.sqrt(df_m74_bubble_sub2['a']*df_m74_bubble_sub2['b'])*44

In [ ]:
len(df_m74_bubble_sub2[df_m74_bubble_sub2['reff']<40])

In [ ]:
df_m74_bubbles_s1[1:]

In [ ]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.table import Table
import seaborn as sb
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator

xs = []  # Will store semi-major axis distances
ys = []  # Will store log ages
rs = []  # Will store bubble radii for normalization

df_sel = df_m74_bubbles_s0
for region, data in df_sel.iterrows():
    ra = data['ra']
    dec = data['dec']
    a = data['a']  # semi-major axis (assuming arcsec)
    b = data['b']  # semi-minor axis (assuming arcsec)
    q = b / a  # axis ratio
    ang = data['ang']  # position angle in degrees
    
    # Create temporary table and apply elliptical selection
    df_temp = Table.from_pandas(df_spatial.copy())
    df_temp = ellipse(df_temp, 'RA', 'DEC', ra, dec, ang, 0, 0, 1.0 * a / 3600, 1.0 * b / 3600)
    
    # Apply age cut
    if region>=30:
        t_cut = 7.6
    else:
        t_cut = cut_ages[region]
    if t_cut>=7.7:
        t_cut = 7.6
    df_temp = df_temp[df_temp['Log_Age'] <= t_cut]
    
    # Calculate offsets from bubble center
    stars = SkyCoord(df_temp['RA'] * u.deg, df_temp['DEC'] * u.deg)
    center = SkyCoord(ra * u.deg, dec * u.deg)
    
    tan_plane = SkyOffsetFrame(origin=center)
    proj_coords = stars.transform_to(tan_plane)
    
    offset_ra = proj_coords.lon.to(u.arcsec).value  # Offset RA in degrees
    offset_dec = proj_coords.lat.to(u.arcsec).value # Offset Dec in degrees

    # Calculate rotation matrix
    theta = np.radians(ang)
    cos_theta = np.cos(theta)
    sin_theta = np.sin(theta)
    rotation_matrix = np.array([[cos_theta, -sin_theta],
                                [sin_theta, cos_theta]])

    # Apply rotation to offsets
    rotated_offsets = np.dot(rotation_matrix, np.vstack((offset_ra, offset_dec)))
    x_rotated = rotated_offsets[0]
    y_rotated = rotated_offsets[1]
    
    # Calculate equivalent semi-major axis distance
    # This gives the distance where the star would be on an ellipse with same axis ratio
    semi_major_distances = np.sqrt((x_rotated)**2 + (y_rotated / q)**2)
    semi_minor_distances = q*semi_major_distances
    
    xs.extend(np.sqrt(semi_major_distances*semi_minor_distances))
    ys.extend(df_temp['Log_Age'])
    rs.extend([np.sqrt(a * b)] * len(df_temp))

# Convert to arrays
xs = np.array(xs)  # Semi-major axis distances in arcsec
ys = np.array(ys)   # Log ages
rs = np.array(rs)   # Bubble radii in arcsec

# Calculate normalized radii
normalized_radii = xs / rs

# Convert log age to linear age in Myr
ages_myr = 10**(ys - 6)  # Assuming Log_Age = log10(age_in_years)

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
sb.kdeplot(x=normalized_radii, y=ages_myr, fill=True, cmap='grey_r',
           thresh=0.00, bw_adjust=0.8, levels=np.linspace(0.1,1,100), alpha=1,ax=ax,
          zorder=1)

age_bins = [5]
radii_med = []
age = age_bins[-1]

# while age_bins[-1] <80:

#     ind  = (ages_myr >=age_bins[-1]) & (ages_myr < age)
#     if len(ages_myr[ind])<200 and age<80:
#         age += 1
#     else:
#         radii_med.append(np.quantile(normalized_radii[ind], 0.84))
#         age_bins.append(age)

        # vp = ax.violinplot(normalized_radii[ind],[0.5*(age+ age_bins[-2])], widths=10,
        #              orientation='horizontal', quantiles=[0.16,0.84],
        #                    showmedians = True, showextrema=True,)

        # for pc in vp['bodies']:
        #     pc.set_facecolor('red')   # or any color name / RGB / hex
        #     pc.set_edgecolor('black')
        #     pc.set_alpha(0.3)
        # for partname in ('cmedians', 'cbars', 'cmins', 'cmaxes','cquantiles'):
        #     if partname in vp:
        #         vp[partname].set_color('black')     # line color
        #         vp[partname].set_linewidth(1.)

        
# age_bins = np.array(age_bins)
# radii_med = np.array(radii_med)
# age_bin_cen = 0.5*(age_bins[1:] + age_bins[:-1])

# ax.plot(radii_med, age_bin_cen, color='black', lw=2, linestyle='--')
#ax.scatter(xs/rs, y=10**ys/1e6, zorder=20, color='black')
ax.set_xlabel(r'${\rm Normalized\, Radius}$')
ax.set_ylabel(r'${\rm Age}$ [Myr]')

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', width=3,direction="in", top = True,right = True,
               bottom = True, left = True)
ax.tick_params(which='major', length=15,direction="in")
ax.tick_params(which='minor', length=8, color='black',direction="in")
ax.set_ylim(5,45)
ax.set_axisbelow(False)  # this brings grid/ticks/spines ABOVE images
    
ax.set_xlim(-0.05,1.05)

In [ ]:
stats.median(normalized_radii[ind])

In [ ]:
radii_med

In [ ]:
vp

In [ ]:
r_young = []
r_inter = []
r_old = []
r = []
bub_id = []

#plt.figure(figsize=(15,5))
for region, data in df_m74_bubble_sub.iterrows():

    ra  = data['ra']
    dec = data['dec']
    a   = data['a']*1.2
    b   = data['b']*1.2
    ang = data['ang']
    
    df_temp = Table.from_pandas(df_spatial.copy())
    
    df_temp  = ellipse(df_temp,'RA','DEC', ra, dec, ang, 0, 0, a/3600, b/3600)
    i = region#int(region.split('_')[-1])
        
    bub_id.append(i)
    r.append(a/1.2)
    
    #--------------Young---------------
    df_age = df_temp[(df_temp['Log_Age']<=7.0)]

    cum_y = []
    x = np.arange(0, a, 0.1)
    for a1 in np.arange(0, a, 0.1):
        
        df_t = ellipse(df_age,'RA','DEC', ra, dec, ang, 0, 0, a1/3600,a1*(b/a)/3600)
        cum_y.append(len(df_t))

    if np.sum(cum_y)>=2:
        
        yn = np.arange(0,1,0.05)
        cum_y = np.array(cum_y)/cum_y[-1]
        xn = np.interp(yn,cum_y,x)

        r_young.append(xn[yn>=0.25][0])
    else:
        r_young.append(np.nan)
    #plt.plot(x,cum_y,'-b')
    #-------------Inter-----------
    df_age = df_temp[(df_temp['Log_Age']>7.0) & (df_temp['Log_Age']<=7.7)]

    cum_y = []
    x = np.arange(0, a, 0.1)
    for a1 in np.arange(0, a, 0.1):
        
        df_t = ellipse(df_age,'RA','DEC', ra, dec, ang, 0, 0, a1/3600,a1*(b/a)/3600)
        cum_y.append(len(df_t))
    
    if np.sum(cum_y)>=3:
        
        yn = np.arange(0,1,0.05)
        cum_y = np.array(cum_y)/cum_y[-1]
        xn = np.interp(yn,cum_y,x)

        r_inter.append(xn[yn>=0.25][0])
    else:
        r_inter.append(np.nan)
    
r = np.array(r)
ind = np.argsort(r)

r = r[ind]

bub_id = np.array(bub_id)[ind]
r_young = np.array(r_young)[ind]
r_inter = np.array(r_inter)[ind]


In [ ]:
r_stack = np.vstack([r_young, r_inter])

In [ ]:
x_ = r.copy()
#x_[bub_id==1] = x_[bub_id==1] + 5/44
#x_[bub_id==7] = x_[bub_id==7] - 2/44
#x_[bub_id==14] = x_[bub_id==14] - 2/44
#x_[bub_id==10] = x_[bub_id==10] + 5/44

r_tile_x = np.tile(x_,(2,1))
r_tile_r = np.tile(r,(2,1))

In [ ]:
fig, axs = plt.subplots(2,1,figsize=(10,8), sharex=True)
#ax.plot(r_tile_x*44, r_stack/r_tile_r,'-', color='black')

ax = axs[0]
ax.scatter(x_*44, r_young/r, s=40, color='blue', zorder=100,label=r'$log(Age)\leq7.2$', edgecolor='black')
ax.set_ylabel(r'R$_{Young}$/R$_{Bubble}$')
ax = axs[1]

ax.scatter(x_*44, r_young/r_inter, color='blue', s=40, zorder=100,label=r'$log(Age)\leq7.2$', edgecolor='black')
ax.axhline(1, color='black', linestyle='--')
ax.set_ylabel(r'R$_{Young}$/R$_{Old}$')
ax.set_xlabel(r'R$_{Bubble}~$[pc]')
#ax.scatter(x_*44, r_inter/r, color='red', zorder=100,label=r'$7.3<log(Age)\leq7.5$')
#ax.scatter(x_*44, r_old/r, color='red', zorder=100, label=r'$7.3<log(Age)\leq7.6$')

#for x,y, i in  zip(x_*44,   r_inter/r, bub_id):
 #   ax.annotate(str(i), (x-2,y+0.01))
    
#ax.axhline(1, color='black', linestyle='--')


#ax.set_ylim(0,1.1)
for ax in axs:
    ax.grid()
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.tick_params(which='both', width=2,direction="in", top = True,right = True,
                   bottom = True, left = True)
    ax.tick_params(which='major', length=7,direction="in")
    ax.tick_params(which='minor', length=4, color='black',direction="in")

plt.subplots_adjust(hspace=0)
#ax.legend()

# **RGB Images**

In [ ]:
from composite import colorize_image, combine_multicolor, greyRGBize_image
from astropy.visualization.wcsaxes import add_beam, add_scalebar

In [ ]:
muse_wcs

In [ ]:
hdul = fits.open('../data/ngc628/jw01783-o908_t016_miri_f770w_i2d.fits')
data_f770w = hdul[1].data
f770w_wcs = WCS(hdul[1].header)

hdul = fits.open('../data/ngc628/F200W_i2d.fits')
data_f200w = hdul[1].data
f200w_wcs = WCS(hdul[1].header)

hdul = fits.open('../data/ngc628/jw01783-o004_t008_nircam_clear-f150w_i2d.fits')
data_f150w = hdul[1].data
f150w_wcs = WCS(hdul[1].header)

hdul = fits.open('../data/ngc628/jw01783-o004_t008_nircam_clear-f115w_i2d.fits')
data_f115w = hdul[1].data
f115w_wcs = WCS(hdul[1].header)


data_f150w = ha
f150w_wcs  = muse_wcs

In [ ]:
tab_spatial = Table.from_pandas(pd.read_csv("../data/NGC628_spatial.csv"))

In [ ]:
s1 = df_m74_bubbles_s1
s2 = df_m74_bubbles_s2[df_m74_bubbles_s2['n50']>0]
s3 = df_m74_bubbles_s2[df_m74_bubbles_s2['n50']==0]
len(s1), len(s2), len(s3)

In [ ]:
df = df_m74_bubbles_s2[df_m74_bubbles_s2['n50']==0].sort_values('r',ascending=False).reset_index()

In [ ]:
df = df_m74_bubbles_s2[df_m74_bubbles_s2['n50']==0].sort_values('r',ascending=False).reset_index()

for region, data in df.iterrows():
    ra   = data['ra']
    dec  = data['dec']
    a    = data['a']
    b    = data['b']
    ang  = data['ang']
    name = data['Bubble ID']
    if name not in [92]:
        continue
    reff = np.sqrt(a*b)*44
    # if reff>=40:
    #     continue
    radius = a*2
    
    pos = SkyCoord(ra=ra, dec=dec, unit='deg', frame='icrs')

    cutout_770 = Cutout2D(data_f770w,pos,size = 4*2*radius*u.arcsec, wcs = f770w_wcs)
    
    cutout_200 = Cutout2D(data_f200w,pos,size = 4*2*radius*u.arcsec, wcs = f200w_wcs)
    try:
        ha_flag = True
        cutout_150 = Cutout2D(data_f150w,pos,size = 4*2*radius*u.arcsec, wcs = f150w_wcs)
    except:
        ha_flag = False
    cutout_115 = Cutout2D(data_f115w,pos,size = 4*2*radius*u.arcsec, wcs = f115w_wcs)
    
    wcs_new =  cutout_200.wcs.deepcopy()
    
    cen_coord_pixel = [[cutout_200.data.shape[0]/2, cutout_200.data.shape[1]/2]]
    cen_coord_world = cutout_200.wcs.pixel_to_world_values(cen_coord_pixel)
    
    wcs_new.wcs.crpix = cen_coord_pixel[0]
    wcs_new.wcs.crval = cen_coord_world[0]
    wcs_new.wcs.pc = np.array([[1,0],
                               [0,1]])
    
    f770w,_ = reproject_interp((cutout_770.data, cutout_770.wcs), wcs_new, shape_out=cutout_200.shape,
                              parallel=True)
    
    f200w,_ = reproject_interp((cutout_200.data, cutout_200.wcs), wcs_new, shape_out=cutout_200.shape,
                              parallel=True)

    if ha_flag:
        f150w,_ = reproject_interp((cutout_150.data, cutout_150.wcs), wcs_new, shape_out=cutout_200.shape,
                                  parallel=True)
    
    f115w,_ = reproject_interp((cutout_115.data, cutout_115.wcs), wcs_new, shape_out=cutout_200.shape,
                              parallel=True)
    
    
    cutout_770 = Cutout2D(f770w,pos,size = 2*2*radius*u.arcsec, wcs = wcs_new)
    
    cutout_200_ = Cutout2D(f200w,pos,size = 2*2*radius*u.arcsec, wcs = wcs_new)
    if ha_flag:
        cutout_150 = Cutout2D(f150w,pos,size = 2*2*radius*u.arcsec, wcs = wcs_new)
    cutout_115 = Cutout2D(f115w,pos,size = 2*2*radius*u.arcsec, wcs = wcs_new)

    red_llim = np.nanpercentile(cutout_200_.data.ravel(),10)
    red_ulim = np.nanpercentile(cutout_200_.data.ravel(),99)
    #red = colorize_image(greyRGBize_image(cutout_200_.data,'log', min_max=[red_llim,red_ulim], gamma=2),'#FF0000','hex')
    red = colorize_image(greyRGBize_image(cutout_200_.data,'log', min_max=[red_llim,red_ulim], gamma=2),'#008000','hex')

    if ha_flag:
        green_llim = np.nanpercentile(cutout_150.data.ravel(),10)
        green_ulim = np.nanpercentile(cutout_150.data.ravel(),99)
        green = colorize_image(greyRGBize_image(cutout_150.data,'log',min_max=[green_llim,green_ulim], gamma=2),'#FF0000','hex')
        #green = colorize_image(greyRGBize_image(cutout_150.data,'log',min_max=[green_llim,green_ulim], gamma=2),'#008000','hex')
    blue_llim = np.nanpercentile(cutout_115.data.ravel(),10)
    blue_ulim = np.nanpercentile(cutout_115.data.ravel(),99)
    blue = colorize_image(greyRGBize_image(cutout_115.data,'log',min_max=[blue_llim,blue_ulim], gamma=2),'#0000FF','hex')
    
    white_ulim = np.nanpercentile(cutout_770.data.ravel(),99)
    white = colorize_image(greyRGBize_image(cutout_770.data,'log',min_max=[0.8,white_ulim], gamma=2),'#90EE90','hex')

    if ha_flag:
        rgb = combine_multicolor([red,2*green, blue, 1*white],gamma=0.9)
    else:
        rgb = combine_multicolor([red, blue, 1*white],gamma=0.9)
    fig = plt.figure(figsize=(12,10))
    ax = fig.add_subplot( projection=cutout_200_.wcs)
    ax.imshow(rgb)

    coord = SkyCoord(ra = data['ra']* u.deg, dec = data['dec']* u.deg)
    ellip = EllipseSkyRegion(coord, 2*data['a']*u.arcsec, 2*data['b']*u.arcsec,
                             180*u.deg-data['ang']*u.deg).to_pixel(cutout_200_.wcs)
    
    r = ellip.as_artist(color='black', lw=3)
    
    #ax.annotate(str(key), (data['ra'], data['dec']- 1/3600), fontsize=15, color='cyan', zorder=301,
     #           xycoords =ax.get_transform('icrs'))
    ax.add_patch(r)
    ax.invert_xaxis()

    coords = cutout_200_.wcs.calc_footprint()

    height = coords[:,1].max() - coords[:,1].min()
    width  = coords[:,0].max() - coords[:,0].min()

    df = box(tab_spatial, 'RA','DEC', data['ra'], data['dec'],0,0,width, height,0)
    df = df[df['Log_Age']<=7.7]
    x = df['RA']
    y = df['DEC']
    c = 10**(df['Log_Age']-6)

    h_min = 0.011149627875919776

    img  = ax.scatter(x,y,c=c, s=(180/height)*h_min,cmap='jet', zorder=300,marker='*',
               transform = ax.get_transform('icrs'), vmin=5, vmax=50,edgecolor='white')
    cb = plt.colorbar(img, ax= ax)
    cb.set_label('Age [Myr]')
    #ax.set_xlim(coords[:,0][-1], coords[:,0][0])
    #ax.set_ylim(coords[:,1][0], coords[:,1][-1])

    df = box(tab_cluster, 'PHANGS_RA', 'PHANGS_DEC', data['ra'], data['dec'],0,0,width, height,0)

    x = np.array(df['PHANGS_RA'])
    y = np.array(df['PHANGS_DEC'])
    if len(x)>0:
        ax.scatter(x,y, s=(120/height)*h_min, edgecolor='white', zorder=200,
                   color='magenta',marker='o', transform = ax.get_transform('icrs'))

    ax.set_ylabel(r'Dec ${\rm \,(J2000)}$', fontsize=30)
    ax.set_xlabel(r'RA ${\rm\,(J2000)}$', fontsize=30)
    ax.set_xlim(cutout_115.data.shape[0],0)
    ax.set_ylim(0,cutout_115.data.shape[1])
    radius_bubble = np.round(reff,0).astype(int)
    text = f"Bubble {name} | " + r"R$_{\rm B}$ " + f": {radius_bubble} pc"
    if not ha_flag:
        text = r'$^*$ ' + text

    fig.suptitle(text, y=0.92, x=0.45,fontsize=40)
    fig.savefig(f'../SFH/ngc628/JWST_bubble_obs/figures3/bubble_{region + 81}_rgb.png', 
                bbox_inches='tight')

    plt.close(fig)

In [ ]:
region + 81

In [ ]:
10**7.4/1e6

In [ ]:
rgb_files = glob.glob("figures/RGB_images/*.png") # change to .jpg if needed
cmd_files = glob.glob("figures/stars/*.png")

In [ ]:
!rm bubbles_output1.pdf

In [ ]:
for i in range(len(df_m74_bubbles_s1)):
    
    rgb = plt.imread(f"../SFH/ngc628/JWST_bubble_obs/figures/bubble_{i}_rgb.png")
    cmd = plt.imread(f"../SFH/ngc628/JWST_bubble_obs/figures/bubble_{i}_cmd.png")
    sfh = plt.imread(f"../SFH/ngc628/JWST_bubble_obs/figures/bubble_{i}.png")

    fig, ax = plt.subplots(1, 3, figsize=(15*5, 5*5),width_ratios=[0.42,
                                                                   0.4,
                                                                   0.615])
    
    ax[0].imshow(rgb, origin='upper')
    ax[0].axis("off")

    ax[2].imshow(sfh, origin='upper')
    ax[2].axis("off")

    ax[1].imshow(cmd, origin='upper')
    ax[1].axis("off")

    plt.subplots_adjust(hspace=0,wspace=-0.01)
    pos1 = ax[1].get_position()
    pad1 = 0.043 

    pos2 = ax[2].get_position()
    pad2 = -0.021

    ax[1].set_position([pos1.x0+0.01, pos1.y0 - pad1, pos1.width, pos1.height])
    ax[2].set_position([pos2.x0, pos2.y0 - pad2, pos2.width, pos2.height])

    ax[1].set_zorder(10)
    ax[0].set_zorder(5)
    ax[2].set_zorder(1)

    fig.savefig(f"../SFH/ngc628/JWST_bubble_obs/panels/bubble_{i}.png", bbox_inches='tight')
    plt.close(fig)


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
panels = glob.glob("../SFH/ngc628/JWST_bubble_obs/panels/*")

with PdfPages("bubbles_sample1.pdf") as pdf:
    images_per_page = 4
    for i in range(0, len(panels), images_per_page):
        fig, axes = plt.subplots(images_per_page, 1, figsize=(8, 10))
        if images_per_page == 1:
            axes = [axes]
        for j, ax in enumerate(axes):
            idx = i + j
            if idx < len(panels):
                img = plt.imread(panels[idx])
                ax.imshow(img, origin='upper')
                ax.axis("off")
            else:
                ax.axis("off")
        plt.tight_layout(pad=0.2)
        pdf.savefig(fig)
        plt.close(fig)

In [ ]:
panels = glob.glob("../SFH/ngc628/JWST_bubble_obs/panels2/*")

with PdfPages("bubbles_sample2.pdf") as pdf:
    images_per_page = 4
    for i in range(0, len(panels), images_per_page):
        fig, axes = plt.subplots(images_per_page, 1, figsize=(8, 10))
        if images_per_page == 1:
            axes = [axes]
        for j, ax in enumerate(axes):
            idx = i + j
            if idx < len(panels):
                img = plt.imread(panels[idx])
                ax.imshow(img, origin='upper')
                ax.axis("off")
            else:
                ax.axis("off")
        plt.tight_layout(pad=0.2)
        pdf.savefig(fig)
        plt.close(fig)

In [ ]:
panels = glob.glob("../SFH/ngc628/JWST_bubble_obs/figures3/*")

with PdfPages("bubbles_sample3.pdf") as pdf:
    images_per_page = 4
    for i in range(0, len(panels), images_per_page):
        fig, axes = plt.subplots(images_per_page, 1, figsize=(8, 10))
        if images_per_page == 1:
            axes = [axes]
        for j, ax in enumerate(axes):
            idx = i + j
            if idx < len(panels):
                img = plt.imread(panels[idx])
                ax.imshow(img, origin='upper')
                ax.axis("off")
            else:
                ax.axis("off")
        plt.tight_layout(pad=0.2)
        pdf.savefig(fig)
        plt.close(fig)

In [ ]:
df = df_m74_bubbles_s2[df_m74_bubbles_s2['n50']>0]

In [ ]:
df = df_m74_bubbles_s2[df_m74_bubbles_s2['n50']>0]

for i in range(30,len(df)+30):
    
    rgb = plt.imread(f"../SFH/ngc628/JWST_bubble_obs/figures2/bubble_{i}_rgb.png")
    cmd = plt.imread(f"../SFH/ngc628/JWST_bubble_obs/figures2/bubble_{i}_cmd.png")
    sfh = plt.imread(f"../SFH/ngc628/JWST_bubble_obs/figures2/bubble_{i}.png")

    fig, ax = plt.subplots(1, 3, figsize=(15*5, 5*5),width_ratios=[0.42,0.4,0.525])
    ax[0].imshow(rgb, origin='upper')
    ax[0].axis("off")

    ax[2].imshow(sfh, origin='upper')
    ax[2].axis("off")

    ax[1].imshow(cmd, origin='upper')
    ax[1].axis("off")

    plt.subplots_adjust(hspace=0,wspace=-0.01)
    pos1 = ax[1].get_position()
    pad1 = 0.043 

    pos2 = ax[2].get_position()
    pad2 = 0.005

    ax[1].set_position([pos1.x0+0.01, pos1.y0 - pad1, pos1.width, pos1.height])
    ax[2].set_position([pos2.x0+0.01, pos2.y0 - pad2, pos2.width, pos2.height])

    ax[1].set_zorder(10)
    ax[0].set_zorder(5)
    ax[2].set_zorder(1)
    
    fig.savefig(f"../SFH/ngc628/JWST_bubble_obs/panels2/bubble_{i}.png", bbox_inches='tight')
    plt.close(fig)

In [ ]:
len(df_m74_bubbles)

In [ ]:
10*10**((25.50 - -5.816)/5)/1e6

In [ ]:
df_m74_bubbles[df_m74_bubbles.duplicated('area', keep=False)]

In [ ]:
for i in range(len(df_m74_bubble_sub2)):
    i+=36
    try:
        rgb = plt.imread(f"../SFH/ngc628/JWST_bubble_obs/figures3/bubble_{i}_rgb.png")
        cmd = plt.imread(f"../SFH/ngc628/JWST_bubble_obs/figures3/bubble_{i}_cmd.png")
    except:
        continue

    fig, ax = plt.subplots(1, 2, figsize=(10, 5),width_ratios=[0.42,0.4])
    ax[0].imshow(rgb, origin='upper')
    ax[0].axis("off")

    ax[1].imshow(cmd, origin='upper')
    ax[1].axis("off")

    plt.subplots_adjust(hspace=0,wspace=-0.01)
    pos1 = ax[1].get_position()
    pad1 = 0.043 

    ax[1].set_position([pos1.x0+0.01, pos1.y0 - pad1, pos1.width, pos1.height])

    ax[1].set_zorder(10)
    ax[0].set_zorder(5)

    fig.savefig(f"../SFH/ngc628/JWST_bubble_obs/panels3/bubble_{i}.png", bbox_inches='tight')
    plt.close(fig)

In [ ]:
!rm bubbles_panels.zip

In [ ]:
!zip bubbles_panels.zip ../SFH/ngc628/JWST_bubble_obs/panels/*

In [ ]:
!rm bubbles_panels.zip

In [ ]:
!zip bubbles_panels.zip ../SFH/ngc628/JWST_bubble_obs/panels/*